# 통신데이터 작업기록

역할: 기존 노트북에서 수행했던 전처리/검증/EDA 작업 기록을 보존한다.  
튜터님께 보여줄 핵심 파일은 `통신데이터_통합.ipynb`, `통신데이터_검증.ipynb`, `통신데이터_EDA.ipynb`, `통신데이터_최종정리.md`이다.

## 기존 작업 기록: 기존 통신데이터_통합.ipynb

## 노트북 정리

- 통신 원본 데이터를 월별/연도별로 합치고, 분석에 쓰기 좋게 parquet 형태로 바꾼 파일.
- t25, t26, t27처럼 용량이 큰 통신 데이터를 먼저 통합하고 저장하는 흐름을 정리함.
- 날짜 컬럼 타입이 달라서 생긴 오류를 확인했고, 최종적으로 날짜 타입을 맞추는 작업까지 진행함.
- 분석보다는 데이터 통합과 저장용 노트북이라, 뒤쪽 EDA에서 쓸 후보 파일을 만드는 역할로 보면 됨.


# 통신 데이터 통합 및 전처리

## 1. 파일 로드 및 환경 설정

In [ ]:
import os
print(os.getcwd())
print(os.listdir())

In [ ]:
import os
import glob
import duckdb
import pandas as pd

DATA_DIR = "../data"
TEMP_DIR = "../duck_temp"

con = duckdb.connect()
con.execute("SET threads=2;")
con.execute(f"SET temp_directory='{TEMP_DIR}';")

## 2. parquet 통합 및 저장

In [ ]:
# t25: CSV → Parquet 변환 (분석 후보용 아님, 저장 효율 개선 목적)

input_file = r"../data/t25_2023_2025_all.csv"
output_file = r"../data/t25_2023_2025_all.parquet"

con.execute(f"""
COPY (
    SELECT *
    FROM read_csv_auto('{input_file}', header=True)
)
TO '{output_file}'
(FORMAT PARQUET, COMPRESSION ZSTD);
""")

print("✅ t25 parquet 변환 완료 (CSV → parquet)")

이코드를 이름만바꿔 변환 완료(후보 파일은 parquet) 이미 만든 원본 데이터를 합쳐서 만든 csv t25 all은 parquet로 변환

In [ ]:
# 잘 변환됬는지 확인 코드
import pyarrow.parquet as pq

file_path = "../data/t25_2023_2025_all.parquet"
parquet_file = pq.ParquetFile(file_path)

print(parquet_file.schema.names)

# 3. 통신 원본 데이터 통합 및 저장
- t4 ~ t27 (t25 제외) 데이터를 월별·연도별 원본 파일에서 통합하여 parquet로 저장
- 본 섹션에서는 파일 생성(통합 및 저장)만 수행하며, 검증은 이후 섹션에서 수행.

In [ ]:
# 1. 파일 찾기
files = sorted(glob.glob("../data/T26_2025*.csv"))
print("찾은 파일 수:", len(files))
print(files[:3])  # 일부만 확인

# 파일 없으면 바로 중단
if len(files) == 0:
    raise ValueError("파일을 못 찾았습니다. 경로 또는 파일명 확인하세요.")

# 2. 하나씩 읽어서 리스트에 저장
df_list = []

for f in files:
    print("읽는 중:", f)
    
    try:
        temp = pd.read_csv(f, encoding="cp949")
    except UnicodeDecodeError:
        temp = pd.read_csv(f, encoding="utf-8-sig")
    
    df_list.append(temp)

# 3. 합치기
df_2025 = pd.concat(df_list, ignore_index=True)

print("합친 후 shape:", df_2025.shape)

# 4. parquet로 저장
df_2025.to_parquet("../data/t26_2025_all.parquet", index=False)

print("저장 완료: t26_2025_all.parquet")

In [ ]:
con.execute("""
SELECT ETL_YMD, COUNT(*) AS cnt
FROM read_parquet('../data/t26_2023_all.parquet')
GROUP BY ETL_YMD
ORDER BY ETL_YMD
""").fetchdf()

In [ ]:
print(con.execute("""
SELECT COUNT(*) 
FROM read_parquet('../data/t27_2025_all.parquet')
""").fetchall())

In [ ]:
print(con.execute("""
SELECT SUBSTR(CAST(ETL_YMD AS VARCHAR), 1, 6) AS ym, COUNT(*) AS cnt
FROM read_parquet('../data/t27_2025_all.parquet')
GROUP BY ym
ORDER BY ym
""").fetchdf())

### 3-1) 월별로 합친 데이터 통합을 년도로 묶어서 통합

In [ ]:
# 데이터 이미 합쳤는데 돌려서 생긴 오류
con.execute("""
COPY (
    SELECT * FROM read_parquet('../data/t26_2023_all.parquet')
    UNION ALL
    SELECT * FROM read_parquet('../data/t26_2024_all.parquet')
    UNION ALL
    SELECT * FROM read_parquet('../data/t26_2025_all_clean.parquet')
)
TO '../data/t26_2023_2025_all.parquet'
(FORMAT PARQUET, COMPRESSION ZSTD);
""")

print("✅ 최종 통합 완료")

### 3-2) 원본 파일중 공백표시된 데이터로 인해 공백제거 후 통합

In [ ]:
files = sorted(glob.glob(r"../data/T26_2025*.csv"))
os.makedirs("../data/t26_2025_clean", exist_ok=True)

for f in files:
    print("처리 중:", f)
    
    out_path = os.path.join("../data/t26_2025_clean", os.path.basename(f))
    
    with open(f, "r", encoding="utf-8") as fr, open(out_path, "w", encoding="utf-8-sig") as fw:
        # 첫 줄(헤더)만 읽어서 공백 제거
        header = fr.readline().strip("\n")
        clean_header = ",".join([col.strip() for col in header.split(",")])
        fw.write(clean_header + "\n")
        
        # 나머지 데이터는 그대로 복사
        for line in fr:
            fw.write(line)
    
    print("저장 완료:", out_path)

print("✅ 헤더 공백 제거 파일 생성 완료")

In [ ]:
# t26의 2025파일 중 컬럼 불일치가 있어 코드 재확인
files = sorted(glob.glob(r"../data/t26_2025_clean/T26_2025*.csv"))
file_list_sql = ", ".join([f"'{f}'" for f in files])

con.execute(f"""
COPY (
    SELECT *
    FROM read_csv_auto(
        [{file_list_sql}],
        header=true,
        union_by_name=true,
        ignore_errors=true
    )
)
TO '../data/t26_2025_all_clean.parquet'
(FORMAT PARQUET, COMPRESSION ZSTD);
""")

print("✅ 2025 clean parquet 저장 완료")

In [ ]:
# 컬럼 확인
df_cols = con.execute("""
DESCRIBE SELECT * FROM read_parquet('../data/t26_2025_all_clean.parquet')
""").df()

print(df_cols['column_name'].tolist())
print("컬럼 수:", len(df_cols))

In [ ]:
# 최종 코드 컬럼 불일치로 인한 2023~2025 컬럼 통일 후 union all
con.execute("""
COPY (
    SELECT 
        ETL_YMD, DOW, D_TIME_CD, D_ADMI_CD, D_MEGA_NM, D_CTY_NM, D_ADMI_NM,
        D_CENTER_X, D_CENTER_Y, PURPOSE, TRANS_GB, DURATION, SEX_CD, AGE_GRP, CNT
    FROM read_parquet('../data/t26_2023_all.parquet')

    UNION ALL

    SELECT 
        ETL_YMD, DOW, D_TIME_CD, D_ADMI_CD, D_MEGA_NM, D_CTY_NM, D_ADMI_NM,
        D_CENTER_X, D_CENTER_Y, PURPOSE, TRANS_GB, DURATION, SEX_CD, AGE_GRP, CNT
    FROM read_parquet('../data/t26_2024_all.parquet')

    UNION ALL

    SELECT 
        ETL_YMD, DOW, D_TIME_CD, D_ADMI_CD, D_MEGA_NM, D_CTY_NM, D_ADMI_NM,
        D_CENTER_X, D_CENTER_Y, PURPOSE, TRANS_GB, DURATION, SEX_CD, AGE_GRP, CNT
    FROM read_parquet('../data/t26_2025_all_clean.parquet')
)
TO '../data/t26_2023_2025_all_fixed.parquet'
(FORMAT PARQUET, COMPRESSION ZSTD);
""")

print("✅ 최종 통합 완료")

In [ ]:
# 나머지 파일들 이코드로 파일명만 바꿔 년도 통합
con.execute("SET threads=2;")
con.execute("SET temp_directory='../duck_temp';")

con.execute("""
COPY (
    SELECT *
    FROM read_csv_auto(
        '../data/T24_GG_PURPOSE_ADMI_POP_*.csv',
        header=true,
        union_by_name=true,
        ignore_errors=true
    )
)
TO '../data/t24_2023_2025_all.parquet'
(FORMAT PARQUET, COMPRESSION ZSTD);
""")

print("✅ T24 2023~2025 통합 완료")

### 같은 데이터가 있어 합칠려는 중 날짜의 타입이 다르다고 오류가 떠 코드 지운 후 날짜 컬럼 데이터 타입 확인

In [ ]:
# 날짜 컬럼 데이터 타입 확인
print(con.execute("""
DESCRIBE SELECT * 
FROM read_parquet('../data/t24_2023_2025_all.parquet')
""").df())

In [ ]:
# 컬럼 수 확인
print(con.execute("""
SELECT COUNT(*) AS cnt
FROM read_parquet('../data/t22_2023_2025_all.parquet')
""").df())

In [ ]:
# 월별 이상치 있는지 확인
print(con.execute("""
SELECT MIN(ETL_YMD) AS min_ymd, MAX(ETL_YMD) AS max_ymd
FROM read_parquet('../data/t22_2023_2025_all.parquet')
""").df())

In [ ]:
# 년도별로 데이터가 얼마나 있는지 확인
print(con.execute("""
SELECT SUBSTR(CAST(ETL_YMD AS VARCHAR), 1, 6) AS ym, COUNT(*) AS cnt
FROM read_parquet('../data/t23_2023_2025_all.parquet')
GROUP BY 1
ORDER BY 1
""").df())

최종 날짜 변환 완료

In [ ]:
final_files = pd.DataFrame([
    ["t20_2023_2025_all.parquet", "통합", "사용 예정", "t20 통합 데이터"],
    ["t21_2023_2025_all.parquet", "통합", "사용 예정", "t21 통합 데이터"],
    ["t22_2023_2025_all.parquet", "통합", "사용 예정", "DATE 변환 후 통합"],
    ["t23_2023_2025_all.parquet", "통합", "사용 예정", "DATE 변환 후 통합"],
    ["t24_2023_2025_all.parquet", "통합", "사용 예정", "DATE 변환 후 통합"],
    ["t25_2023_2025_clean.parquet", "통합", "사용 예정", "컬럼 공백 제거"],
    ["t26_2023_2025_all_fixed.parquet", "통합", "사용 후보", "컬럼 불일치 수정 후 최종"],
    ["t27_2023_2025_all.parquet", "통합", "검토", "t27 통합 데이터"]
], columns=["파일명", "통합 여부", "사용 여부", "설명"])

final_files

## 최종 결과
- t26 파일은 컬럼명에 공백이 포함되어 있어, 공백 제거 후 parquet 형식으로 변환하였다.
- t22 ~ t24 데이터는 연도 컬럼이 BIGINT 타입으로 저장되어 있어, DATE 형식으로 변환한 후 데이터 검증 및 통합을 수행하였다.
- t25, t27은 동일한 구조를 가지므로, 하나의 통합 코드에서 파일명만 변경하여 parquet 파일로 변환 및 저장하였다.(t20,t21은 파일이 작아 all.csv로 최종 생성)
- t20 ~ t24, t27 데이터는 각각 연도 통합을 수행하여 `tXX_2023_2025_all.parquet` 형태의 후보 파일로 생성하였다.
- t25 데이터는 컬럼 공백 문제로 인해 공백 제거 후 `clean.parquet` 형태로 후보 파일을 생성하였다.
- t26 데이터는 2023년 1월부터 2025년 12월까지의 원본 파일 중 일부에서 컬럼 불일치가 발생하여, 공통 컬럼 기준으로 재정의한 후 `fixed.parquet` 형태의 최종 통합 파일을 생성하였다.

## 기존 작업 기록: 기존 1차_전처리.ipynb

## 노트북 정리

- t13, t25, t26, t27 중심으로 날짜 변환, 결측 확인, 컬럼 타입 확인을 진행한 전처리 노트북.
- O/D 지역명, 행정동명, 좌표 결측을 확인하고 같은 행정동 코드 안에서 채울 수 있는 값은 보정함.
- CNT, DURATION, PURPOSE, TRANS_GB, SEX_CD, AGE_GRP 분포를 확인해서 이상치처럼 보이는 값이 실제로 제거 대상인지 확인함.
- PURPOSE, TRANS_GB는 코드 정의가 확실하지 않아서 확정 해석은 보류하고, 분석에서는 조심해서 쓰는 쪽으로 정리함.
- 큰 통신 데이터의 최종 전처리 흐름을 남겨둔 파일로 보면 됨.


In [ ]:
import pandas as pd
import duckdb
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import numpy as np
import sys

con = duckdb.connect()

# t13 데이터 하....결측 처리 완료

In [ ]:
# duckdb 컬럼 확인(일단 큰데이터순으로 t13, t25, t26, t27)
path = "../data/t13_2023_2025_all.parquet"
schema = con.execute(f"""
DESCRIBE SELECT * FROM read_parquet('{path}')
""").fetchdf()
schema

- t13 데이터는 출발/도착/목적/성별/연령대별 등 이동 집계 데이터로 확인
- ETL_YMD = 특정날짜
  출발 행정동/도착 행정동(예: O_ADMI_CD, O_ADMI_NM, D_ADMI_CD, D_ADMI_NM)
- PURPOSE = 목적
- SEX_CD = 성별
- AGE_GRP	= 연령대
- CNT	= 이동량 및 집계값들

일단 날짜데이터부터 확인.

In [ ]:
con.execute("""
select
    min(ETL_YMD) AS min_date,
    max(ETL_YMD) AS max_date,
    count(*) AS row_cnt,
    count(ETL_YMD) AS non_null_date
from read_parquet('../data/t13_2023_2025_all.parquet')
""").fetchdf()

날짜는 정삭정으로 확인 date형태로만 바꿔주면 됨, 전체행은 약2억8천행 모든행에 다 값이 들어있음.

In [ ]:
# 날짜부터 변환
con.execute("""
COPY (
    SELECT
        CAST(STRPTIME(ETL_YMD, '%Y%m%d') AS DATE) AS ETL_YMD,
        *
    EXCLUDE (ETL_YMD)
    FROM read_parquet('../data/t13_2023_2025_all.parquet')
)
TO '../data/t13_2023_2025_all_clean.parquet' (FORMAT PARQUET);
""")

In [ ]:
# date변환 후 행 확인
print(con.execute("""
SELECT COUNT(*) FROM read_parquet('../data/t13_2023_2025_all.parquet')
""").fetchdf())

print(con.execute("""
SELECT COUNT(*) FROM read_parquet('../data/t13_2023_2025_all_clean.parquet')
""").fetchdf())

In [ ]:
# 타입 확인
con.execute("""
describe select *
from read_parquet('../data/t13_2023_2025_all_clean.parquet')""").fetchdf()

In [ ]:
con.execute("""
select etl_ymd
from read_parquet('../data/t13_2023_2025_all_clean.parquet')
limit 10""").fetchdf()

날짜변환 완료!

이제 모든컬럼 결측여부 확인.

## t13 결측 처리

In [ ]:
con.execute("""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN ETL_YMD IS NULL THEN 1 ELSE 0 END) AS etl_ymd_null,
    SUM(CASE WHEN O_ADMI_CD IS NULL THEN 1 ELSE 0 END) AS o_admi_cd_null,
    SUM(CASE WHEN O_MEGA_NM IS NULL THEN 1 ELSE 0 END) AS o_mega_nm_null,
    SUM(CASE WHEN O_CTY_NM IS NULL THEN 1 ELSE 0 END) AS o_cty_nm_null,
    SUM(CASE WHEN O_ADMI_NM IS NULL THEN 1 ELSE 0 END) AS o_admi_nm_null,
    SUM(CASE WHEN O_CENTER_X IS NULL THEN 1 ELSE 0 END) AS o_x_null,
    SUM(CASE WHEN O_CENTER_Y IS NULL THEN 1 ELSE 0 END) AS o_y_null,
    SUM(CASE WHEN D_ADMI_CD IS NULL THEN 1 ELSE 0 END) AS d_admi_cd_null,
    SUM(CASE WHEN D_MEGA_NM IS NULL THEN 1 ELSE 0 END) AS d_mega_nm_null,
    SUM(CASE WHEN D_CTY_NM IS NULL THEN 1 ELSE 0 END) AS d_cty_nm_null,
    SUM(CASE WHEN D_ADMI_NM IS NULL THEN 1 ELSE 0 END) AS d_admi_nm_null,
    SUM(CASE WHEN D_CENTER_X IS NULL THEN 1 ELSE 0 END) AS d_x_null,
    SUM(CASE WHEN D_CENTER_Y IS NULL THEN 1 ELSE 0 END) AS d_y_null,
    SUM(CASE WHEN PURPOSE IS NULL THEN 1 ELSE 0 END) AS purpose_null,
    SUM(CASE WHEN SEX_CD IS NULL THEN 1 ELSE 0 END) AS sex_null,
    SUM(CASE WHEN AGE_GRP IS NULL THEN 1 ELSE 0 END) AS age_null,
    SUM(CASE WHEN CNT IS NULL THEN 1 ELSE 0 END) AS cnt_null
FROM read_parquet('../data/t13_2023_2025_all_clean.parquet')
""").fetchdf()

In [ ]:
con.execute("""
SELECT *
FROM read_parquet('../data/t13_2023_2025_all_clean.parquet')
WHERE O_CTY_NM IS NULL 
""").fetchdf()

In [ ]:
con.execute("""
SELECT *
FROM read_parquet('../data/t13_2023_2025_all_clean.parquet')
WHERE O_MEGA_NM IS NULL 
    AND O_CTY_NM IS NULL 
    AND O_ADMI_NM IS NULL
""").fetchdf()

결측 확인
- O_MEGA_NM: 9,112(출발 광역시도명)
- O_CTY_NM: 116,454(출발 시군구명)
- O_ADMI_NM: 9,112(출발 행정동명)
- O_CENTER_X/Y: 9,112(출발 좌표)
- D_MEGA_NM: 9,055(도착 광역시도명)
- D_CTY_NM: 114,893(도착 시군구명)
- D_ADMI_NM: 9,055(도착 행정동명)
- D_CENTER_X/Y: 9,055(도착 좌표)

## t13 데이터 결론
- 전체 행: 279,996,851
- 날짜, 코드, CNT, 성별, 연령, 목적 -> 결측 0
- 행정명/좌표 컬럼에만 결측 존재

# t13-1 결측 처리 방법

In [ ]:
con.execute("""
COPY (
    SELECT
        ETL_YMD,
        O_ADMI_CD,

        MAX(O_MEGA_NM) OVER (PARTITION BY O_ADMI_CD) AS O_MEGA_NM,
        MAX(O_CTY_NM)  OVER (PARTITION BY O_ADMI_CD) AS O_CTY_NM,
        MAX(O_ADMI_NM) OVER (PARTITION BY O_ADMI_CD) AS O_ADMI_NM,

        MAX(O_CENTER_X) OVER (PARTITION BY O_ADMI_CD) AS O_CENTER_X,
        MAX(O_CENTER_Y) OVER (PARTITION BY O_ADMI_CD) AS O_CENTER_Y,

        D_ADMI_CD,
        D_MEGA_NM,
        D_CTY_NM,
        D_ADMI_NM,
        D_CENTER_X,
        D_CENTER_Y,
        PURPOSE,
        SEX_CD,
        AGE_GRP,
        CNT
    FROM read_parquet('../data/t13_2023_2025_all_clean.parquet')
)
TO '../data/t13_2023_2025_all_clean_fixed.parquet'
(FORMAT PARQUET);
""")

In [ ]:
con.execute("""
SELECT *
FROM read_parquet('../data/t13_2023_2025_all_clean_fixed.parquet')
WHERE O_ADMI_CD = 11230533
""").fetchdf()

In [ ]:
con.execute("""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN ETL_YMD IS NULL THEN 1 ELSE 0 END) AS etl_ymd_null,
    SUM(CASE WHEN O_ADMI_CD IS NULL THEN 1 ELSE 0 END) AS o_admi_cd_null,
    SUM(CASE WHEN O_MEGA_NM IS NULL THEN 1 ELSE 0 END) AS o_mega_nm_null,
    SUM(CASE WHEN O_CTY_NM IS NULL THEN 1 ELSE 0 END) AS o_cty_nm_null,
    SUM(CASE WHEN O_ADMI_NM IS NULL THEN 1 ELSE 0 END) AS o_admi_nm_null,
    SUM(CASE WHEN O_CENTER_X IS NULL THEN 1 ELSE 0 END) AS o_x_null,
    SUM(CASE WHEN O_CENTER_Y IS NULL THEN 1 ELSE 0 END) AS o_y_null,
    SUM(CASE WHEN D_ADMI_CD IS NULL THEN 1 ELSE 0 END) AS d_admi_cd_null,
    SUM(CASE WHEN D_MEGA_NM IS NULL THEN 1 ELSE 0 END) AS d_mega_nm_null,
    SUM(CASE WHEN D_CTY_NM IS NULL THEN 1 ELSE 0 END) AS d_cty_nm_null,
    SUM(CASE WHEN D_ADMI_NM IS NULL THEN 1 ELSE 0 END) AS d_admi_nm_null,
    SUM(CASE WHEN D_CENTER_X IS NULL THEN 1 ELSE 0 END) AS d_x_null,
    SUM(CASE WHEN D_CENTER_Y IS NULL THEN 1 ELSE 0 END) AS d_y_null,
    SUM(CASE WHEN PURPOSE IS NULL THEN 1 ELSE 0 END) AS purpose_null,
    SUM(CASE WHEN SEX_CD IS NULL THEN 1 ELSE 0 END) AS sex_null,
    SUM(CASE WHEN AGE_GRP IS NULL THEN 1 ELSE 0 END) AS age_null,
    SUM(CASE WHEN CNT IS NULL THEN 1 ELSE 0 END) AS cnt_null
FROM read_parquet('../data/t13_2023_2025_all_clean_fixed.parquet')
""").fetchdf()

11230533은 코드 확인후 코드랑 같은 지역으로 결측값 정리 완료 나머지 99코드는 데이터 제공측 자체에서 미확인을 표시하기 위해 99라고 둔걸로 설명해줘 그대로 두기로 결정 이제 세종특별시의 시군구만 none값을 unknown으로 대체하면 됨

In [ ]:
con.execute("""
SELECT *
FROM read_parquet('../data/t13_2023_2025_all_clean_fixed.parquet')
WHERE O_MEGA_NM IS NULL 
    AND O_CTY_NM IS NULL 
    AND O_ADMI_NM IS NULL
""").fetchdf()

In [ ]:
con.execute("""
COPY (
    SELECT
        * EXCLUDE (O_CTY_NM),

        CASE
            WHEN O_MEGA_NM = '세종특별자치시' AND O_CTY_NM IS NULL
                THEN 'UNKNOWN'
            ELSE O_CTY_NM
        END AS O_CTY_NM

    FROM read_parquet('../data/t13_2023_2025_all_clean_fixed.parquet')
)
TO '../data/t13_2023_2025_all_final.parquet'
(FORMAT PARQUET);
""")

In [ ]:
con.execute("""
SELECT *
FROM read_parquet('../data/t13_2023_2025_all_final.parquet')
WHERE O_CTY_NM IS NULL 
""").fetchdf()

In [ ]:
con.execute("""
SELECT
    O_MEGA_NM,
    O_CTY_NM,
    COUNT(*) AS cnt
FROM read_parquet('../data/t13_2023_2025_all_final.parquet')
WHERE O_MEGA_NM = '세종특별자치시'
GROUP BY 1,2;
""").fetchdf()

In [ ]:
con.execute("""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN ETL_YMD IS NULL THEN 1 ELSE 0 END) AS etl_ymd_null,
    SUM(CASE WHEN O_ADMI_CD IS NULL THEN 1 ELSE 0 END) AS o_admi_cd_null,
    SUM(CASE WHEN O_MEGA_NM IS NULL THEN 1 ELSE 0 END) AS o_mega_nm_null,
    SUM(CASE WHEN O_CTY_NM IS NULL THEN 1 ELSE 0 END) AS o_cty_nm_null,
    SUM(CASE WHEN O_ADMI_NM IS NULL THEN 1 ELSE 0 END) AS o_admi_nm_null,
    SUM(CASE WHEN O_CENTER_X IS NULL THEN 1 ELSE 0 END) AS o_x_null,
    SUM(CASE WHEN O_CENTER_Y IS NULL THEN 1 ELSE 0 END) AS o_y_null,
    SUM(CASE WHEN D_ADMI_CD IS NULL THEN 1 ELSE 0 END) AS d_admi_cd_null,
    SUM(CASE WHEN D_MEGA_NM IS NULL THEN 1 ELSE 0 END) AS d_mega_nm_null,
    SUM(CASE WHEN D_CTY_NM IS NULL THEN 1 ELSE 0 END) AS d_cty_nm_null,
    SUM(CASE WHEN D_ADMI_NM IS NULL THEN 1 ELSE 0 END) AS d_admi_nm_null,
    SUM(CASE WHEN D_CENTER_X IS NULL THEN 1 ELSE 0 END) AS d_x_null,
    SUM(CASE WHEN D_CENTER_Y IS NULL THEN 1 ELSE 0 END) AS d_y_null,
    SUM(CASE WHEN PURPOSE IS NULL THEN 1 ELSE 0 END) AS purpose_null,
    SUM(CASE WHEN SEX_CD IS NULL THEN 1 ELSE 0 END) AS sex_null,
    SUM(CASE WHEN AGE_GRP IS NULL THEN 1 ELSE 0 END) AS age_null,
    SUM(CASE WHEN CNT IS NULL THEN 1 ELSE 0 END) AS cnt_null
FROM read_parquet('../data/t13_2023_2025_all_final.parquet')
""").fetchdf()

In [ ]:
con.execute("""
SELECT
    O_ADMI_CD,
    O_MEGA_NM,
    O_CTY_NM,
    O_ADMI_NM,
    O_CENTER_X,
    O_CENTER_Y,
    COUNT(*) AS cnt
FROM read_parquet('../data/t13_2023_2025_all_final.parquet')
WHERE O_MEGA_NM IS NULL
   OR O_CTY_NM IS NULL
   OR O_ADMI_NM IS NULL
   OR O_CENTER_X IS NULL
   OR O_CENTER_Y IS NULL
GROUP BY 1,2,3,4,5,6
ORDER BY cnt DESC;
""").fetchdf()

In [ ]:
con.execute("""
SELECT
    O_ADMI_CD,
    COUNT(*) AS cnt
FROM read_parquet('../data/t13_2023_2025_all_final.parquet')
WHERE O_MEGA_NM IS NULL
   OR O_CTY_NM IS NULL
   OR O_ADMI_NM IS NULL
   OR O_CENTER_X IS NULL
   OR O_CENTER_Y IS NULL
GROUP BY 1
ORDER BY cnt DESC;
""").fetchdf()

1. 세종특별자치시 시군구 결측 처리
세종특별자치시는 시군구 컬럼이 None으로 존재
행정구조상 시군구가 없는 케이스로 판단
O_CTY_NM의 None 값을 UNKNOWN으로 대체
2. 1123으로 시작하는 출발지 코드 결측 보정
일부 O_ADMI_CD가 1123으로 시작하는 데이터에서 지역명/좌표가 None으로 확인됨
동일 코드의 정상 매핑값을 확인한 뒤,
코드와 일치하는 지역명 및 좌표값으로 대체 완료
3. 99 코드 유지
O_ADMI_CD = 99는 총 8건 확인
데이터 공급처에서 미확인 지역을 표시하기 위해 99로 제공한 코드로 판단
임의 보정하지 않고 원본 의미를 유지하기로 결정

출발지 기준 결측은 구조적 결측과 매핑 누락을 보정했으며,
미확인 코드인 99는 원본 의미 보존을 위해 유지했다.
분석 후보용 파일은 t13_2023_2025_all_final.parquet로 후보로 정리한다.(도착도 똑같이 할 예정)

In [ ]:
con.execute("""
SELECT *
FROM read_parquet('../data/t13_2023_2025_all_clean.parquet')
WHERE D_MEGA_NM IS NULL 
    AND D_CTY_NM IS NULL 
    AND D_ADMI_NM IS NULL
""").fetchdf()

도착 전체가 비는 행 수: 9055

도착 부분은 코드가 있는데 none처리된게 여러개인거 같아 확인부터하고 결측처리

In [ ]:
con.execute("""
SELECT
    D_ADMI_CD,
    COUNT(*) AS cnt
FROM read_parquet('../data/t13_2023_2025_all_final.parquet')
WHERE D_MEGA_NM IS NULL
   OR D_CTY_NM IS NULL
   OR D_ADMI_NM IS NULL
   OR D_CENTER_X IS NULL
   OR D_CENTER_Y IS NULL
GROUP BY 1
ORDER BY cnt DESC;
""").fetchdf()

시군구 코드별 여러코드에서 매핑이 빠진걸로 보인 99는 13개로 그대로 두고 나머지를 통일된 코드중 정보가 나와있는거 확인 후 결측 값을 대체 해야 할거 같음.

In [ ]:
con.execute("""
SELECT
    D_ADMI_CD,
    COUNT(*) AS total_cnt,
    COUNT(D_MEGA_NM) AS mega_cnt,
    COUNT(D_CTY_NM) AS cty_cnt,
    COUNT(D_ADMI_NM) AS admi_cnt,
    COUNT(D_CENTER_X) AS x_cnt,
    COUNT(D_CENTER_Y) AS y_cnt,

    MAX(D_MEGA_NM) AS sample_mega_nm,
    MAX(D_CTY_NM) AS sample_cty_nm,
    MAX(D_ADMI_NM) AS sample_admi_nm,
    MAX(D_CENTER_X) AS sample_x,
    MAX(D_CENTER_Y) AS sample_y

FROM read_parquet('../data/t13_2023_2025_all_final.parquet')
WHERE D_ADMI_CD IN (
    36110523, 36110250, 36110520, 36110518, 36110550,
    36110556, 36110530, 11230515, 36110370, 36110515,
    36110525, 36110580, 11230533, 36110555, 36110560,
    36110570, 36110510, 36110540, 36110360, 36110340,
    36110350, 36110330, 36110390, 36110380, 36110320,
    36110310, 52113575, 26440590, 99
)
GROUP BY D_ADMI_CD
ORDER BY total_cnt DESC;
""").fetchdf()

| 유형              | 처리               |
| --------------- | ---------------- |
| 세종 (3611...)    | CTY_NM → UNKNOWN |
| 일반 코드 (서울/부산 등) | 코드 기준으로 채움       |
| 99 코드           | 그대로 유지           |


In [ ]:
con.execute("""
COPY (
    SELECT
        * EXCLUDE (D_MEGA_NM, D_CTY_NM, D_ADMI_NM, D_CENTER_X, D_CENTER_Y),

        -- 시도
        MAX(D_MEGA_NM) OVER (PARTITION BY D_ADMI_CD) AS D_MEGA_NM,

        -- 시군구
        CASE
            WHEN MAX(D_MEGA_NM) OVER (PARTITION BY D_ADMI_CD) = '세종특별자치시'
                THEN 'UNKNOWN'
            ELSE MAX(D_CTY_NM) OVER (PARTITION BY D_ADMI_CD)
        END AS D_CTY_NM,

        -- 행정동
        MAX(D_ADMI_NM) OVER (PARTITION BY D_ADMI_CD) AS D_ADMI_NM,

        -- 좌표
        MAX(D_CENTER_X) OVER (PARTITION BY D_ADMI_CD) AS D_CENTER_X,
        MAX(D_CENTER_Y) OVER (PARTITION BY D_ADMI_CD) AS D_CENTER_Y

    FROM read_parquet('../data/t13_2023_2025_all_final.parquet')
)
TO '../data/t13_2023_2025_all_final_v2.parquet'
(FORMAT PARQUET);
""")

In [ ]:
con.execute("""
SELECT
    COUNT(*) AS total,
    COUNT(D_ADMI_NM) AS filled_admi,
    COUNT(D_CTY_NM) AS filled_cty
FROM read_parquet('../data/t13_2023_2025_all_final_v2.parquet');
""").fetchdf()

도착지는 현재 13건빼고는 전부 매핑완료뜸 잘되었는지 확인

In [ ]:
con.execute("""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN ETL_YMD IS NULL THEN 1 ELSE 0 END) AS etl_ymd_null,
    SUM(CASE WHEN O_ADMI_CD IS NULL THEN 1 ELSE 0 END) AS o_admi_cd_null,
    SUM(CASE WHEN O_MEGA_NM IS NULL THEN 1 ELSE 0 END) AS o_mega_nm_null,
    SUM(CASE WHEN O_CTY_NM IS NULL THEN 1 ELSE 0 END) AS o_cty_nm_null,
    SUM(CASE WHEN O_ADMI_NM IS NULL THEN 1 ELSE 0 END) AS o_admi_nm_null,
    SUM(CASE WHEN O_CENTER_X IS NULL THEN 1 ELSE 0 END) AS o_x_null,
    SUM(CASE WHEN O_CENTER_Y IS NULL THEN 1 ELSE 0 END) AS o_y_null,
    SUM(CASE WHEN D_ADMI_CD IS NULL THEN 1 ELSE 0 END) AS d_admi_cd_null,
    SUM(CASE WHEN D_MEGA_NM IS NULL THEN 1 ELSE 0 END) AS d_mega_nm_null,
    SUM(CASE WHEN D_CTY_NM IS NULL THEN 1 ELSE 0 END) AS d_cty_nm_null,
    SUM(CASE WHEN D_ADMI_NM IS NULL THEN 1 ELSE 0 END) AS d_admi_nm_null,
    SUM(CASE WHEN D_CENTER_X IS NULL THEN 1 ELSE 0 END) AS d_x_null,
    SUM(CASE WHEN D_CENTER_Y IS NULL THEN 1 ELSE 0 END) AS d_y_null,
    SUM(CASE WHEN PURPOSE IS NULL THEN 1 ELSE 0 END) AS purpose_null,
    SUM(CASE WHEN SEX_CD IS NULL THEN 1 ELSE 0 END) AS sex_null,
    SUM(CASE WHEN AGE_GRP IS NULL THEN 1 ELSE 0 END) AS age_null,
    SUM(CASE WHEN CNT IS NULL THEN 1 ELSE 0 END) AS cnt_null
FROM read_parquet('../data/t13_2023_2025_all_final_v2.parquet')
""").fetchdf()

In [ ]:
con.execute("""
SELECT *
FROM read_parquet('../data/t13_2023_2025_all_final_v2.parquet')
WHERE D_MEGA_NM IS NULL
   OR D_CTY_NM IS NULL
   OR D_ADMI_NM IS NULL
   OR D_CENTER_X IS NULL
   OR D_CENTER_Y IS NULL
LIMIT 20;
""").fetchdf()

In [ ]:
con.execute("""
SELECT
    ETL_YMD,
    D_ADMI_CD,
    D_MEGA_NM,
    D_CTY_NM,
    D_ADMI_NM,
    D_CENTER_X,
    D_CENTER_Y,
    CNT
FROM read_parquet('../data/t13_2023_2025_all_final_v2.parquet')
WHERE D_ADMI_CD = 11230533
ORDER BY ETL_YMD
LIMIT 50;
""").fetchdf()

t13 데이터에 대해 출발지(O) 및 도착지(D) 기준 행정코드 매핑을 수행하여 결측을 보정하였다.
세종특별자치시의 경우 행정구조 특성상 시군구 값이 존재하지 않아 UNKNOWN으로 처리하였다.
코드별 매핑 검증 결과, 모든 None 값은 정상적으로 보정되었으며,
최종적으로 남은 결측은 데이터 공급처에서 정의한 미확인 코드(99)로 확인되어 원본 의미를 유지하였다.
분석 후보용 데이터는 t13_2023_2025_all_final_v2.parquet로 저장하였다.

출발 도착 전체가 비는행은 없음

| 구분 | 결측 수 | 처리 방식 |
|------|--------|-----------|
| 출발지(O) | 8건 | 99 코드 (미확인) 유지 |
| 도착지(D) | 13건 | 99 코드 (미확인) 유지 |
| 세종 시군구 | 구조적 결측 | UNKNOWN 처리 |
| 기타 결측 | 0건 | 전부 매핑 완료 |

# t13 1차 전처리(결측) 정리 초안 완료

## t13 이상치 확인

In [ ]:
con.execute("""
summarize
            select cnt
from read_parquet('../data/t13_2023_2025_all_clean.parquet')""").fetchdf()

확인 결과
- min: 0.93
- max: 4256
- 평균: 7.34
- 중앙값: 3.58
- Q3: 6.40
- 표준편차: 20.3

특별히 문제될만한 이상치는 안보임.

In [ ]:
con.execute("""
SELECT
    QUANTILE_CONT(CNT, 0.25) AS q1,
    QUANTILE_CONT(CNT, 0.50) AS median,
    QUANTILE_CONT(CNT, 0.75) AS q3,
    QUANTILE_CONT(CNT, 0.95) AS p95,
    QUANTILE_CONT(CNT, 0.99) AS p99
FROM read_parquet('../data/t13_2023_2025_all_clean.parquet')
""").fetchdf()

In [ ]:
# 데이터가 너무커 100000행으로 샘플데이터로 확인해봄
df13 = con.execute("""
SELECT CNT
FROM read_parquet('../data/t13_2023_2025_all_clean.parquet')
USING SAMPLE 100000
""").fetchdf()

In [ ]:
# 일반 박스플롯으로 이상치 확인
plt.boxplot(df13['CNT'])
plt.title('CNT Boxplot')
plt.show()

In [ ]:
plt.boxplot(np.log1p(df13['CNT']))
plt.title('CNT Boxplot (log scale)')
plt.show()

# cnt 이상치 결과 결론
CNT의 평균(7.34)과 중앙값(3.58)을 비교한 결과, 분포가 오른쪽으로 긴 형태(right-skewed)를 보았고, 특히 최대값(4256)이 상위 분위수(Q3=6.4) 대비 매우 크게 나타나면서, 일반 스케일의 박스플롯에서는 다수의 이상치로 보여지고있음.
하지만 로그 변환 후 분포를 확인한 결과, 극단값의 영향이 완화되었으며 데이터는 자연스러운 분포 형태를 보였고,
따라서 해당 값들은 제거해야 할 이상치라고 보기는 힘들어 보이고, 이동량 집계 데이터의 특성에서 발생한 값으로 판단됨.

- 이상치 제거 필요 없음
- 로그 변환 기반 분석 권장
- 데이터 품질 나쁘지 않음
- 결측치 처리만 해결하면 됨

# t13 범주 데이터 purpose(목적), sex_cd(성별), age_grp(나이) 확인

In [ ]:
# 목적 데이터부터 확인
con.execute("""
select purpose, count(*) as cnt
from read_parquet('../data/t13_2023_2025_all_clean.parquet')
            group by purpose
            order by cnt desc""").fetchdf()

- 0(귀가) : 98456437
- 1(노선버스) : 54581952
- 2(지하철) : 5053207
- 3(도보) : 3039365
- 4(고속버스) : 906684
- 5(기차) : 1352525
- 6(항공) : 116606681
- 7(기타) : 데이터가 없음.

이거는 전화해서 물어보는걸로 그리고 공유드릴게요

In [ ]:
con.execute("""
SELECT
    PURPOSE,
    AVG(CNT) AS avg_cnt,
    COUNT(*) AS cnt
FROM read_parquet('../data/t13_2023_2025_all_clean.parquet')
GROUP BY PURPOSE
ORDER BY PURPOSE
""").fetchdf()

이 데이터는 통신사 기반 유동인구 데이터로 보이고
PURPOSE는 내부 코드라 공식 정의가 없는 상태입니다

PURPOSE 변수는 이동 유형을 나타내는 코드값(0~6)으로 구성되어 있으나,
원본 데이터 설명 부재로 인해 각 코드의 정확한 의미는 확인되지 않았다.
따라서 본 분석에서는 해당 변수를 범주형 변수로 활용하고, 코드 간 상대적 비교 중심으로 해석하였다.

- 0(귀가) : 98456437 
- 1(노선버스) : 54581952 
- 2(지하철) : 5053207 
- 3(도보) : 3039365 
- 4(고속버스) : 906684 
- 5(기차) : 1352525 
- 6(항공) : 116606681 
- 7(기타) : 0
정확한 정보는 아님... 검색해서 알아본 정보

In [ ]:
con.execute("""
SELECT SEX_CD, COUNT(*) AS cnt
FROM read_parquet('../data/t13_2023_2025_all_clean.parquet')
GROUP BY SEX_CD
ORDER BY cnt DESC
""").fetchdf()

In [ ]:
con.execute("""
SELECT AGE_GRP, COUNT(*) AS cnt
FROM read_parquet('../data/t13_2023_2025_all_clean.parquet')
GROUP BY AGE_GRP
ORDER BY AGE_GRP
""").fetchdf()

In [ ]:
con.execute("""
SELECT
    PURPOSE,
    COUNT(*) AS cnt,
    COUNT(*) * 1.0 / SUM(COUNT(*)) OVER() AS ratio
FROM read_parquet('../data/t13_2023_2025_all_clean.parquet')
GROUP BY PURPOSE
ORDER BY cnt DESC
""").fetchdf()

In [ ]:
con.execute("""
SELECT
    SEX_CD,
    COUNT(*) AS cnt,
    COUNT(*) * 1.0 / SUM(COUNT(*)) OVER() AS sex_cnt
FROM read_parquet('../data/t13_2023_2025_all_clean.parquet')
GROUP BY SEX_CD
ORDER BY cnt DESC
""").fetchdf()

In [ ]:
con.execute("""
SELECT
    AGE_GRP,
    COUNT(*) AS cnt,
    COUNT(*) * 1.0 / SUM(COUNT(*)) OVER() AS age_cnt
FROM read_parquet('../data/t13_2023_2025_all_clean.parquet')
GROUP BY AGE_GRP
ORDER BY cnt DESC
""").fetchdf()

In [ ]:
df13 = con.execute("""
SELECT PURPOSE
FROM read_parquet('../data/t13_2023_2025_all_clean.parquet')
USING SAMPLE 100000
""").fetchdf()

df13['PURPOSE'].value_counts().sort_index().plot(kind='bar')

PURPOSE 변수의 분포를 확인한 결과, 특정 코드(6)에 데이터가 가장 많이 집중되어 있으며, 이후 0, 1 순으로 감소하는 형태를 보였다.
이는 전체 이동 데이터가 일부 특정 유형에 편중되어 있음을 의미하며, PURPOSE 변수 간 불균형이 존재하는 것으로 해석된다.

In [ ]:
df13 = con.execute("""
SELECT SEX_CD
FROM read_parquet('../data/t13_2023_2025_all_clean.parquet')
USING SAMPLE 100000
""").fetchdf()

df13['SEX_CD'].value_counts().plot(kind='bar')

남자의 비중이 가장 높았고 그다음 여자인데 차이가 심하다. 그리고 W는 찾아보니 통신사쪽 Unknown / 미확인 / 기타로 나타나 이게 20%나 되서 분석할때 따로 해봐야 할거 같습니다.

In [ ]:
df13 = con.execute("""
SELECT AGE_GRP
FROM read_parquet('../data/t13_2023_2025_all_clean.parquet')
USING SAMPLE 100000
""").fetchdf()

df13['AGE_GRP'].value_counts().sort_index().plot(kind='bar')

나이는 정규분포 형태를 띄고 있어 데이터에 이상이 없는것으로 보입니다. 4,5,6,7에 많이 몰려있는것으로 나타납니다. 유동인구 데이터에 적합한 느낌이 든다 나이가 많거나 적을수록 수가 적은거라서...

# T25 데이터

In [ ]:
path = "../data/t25_2023_2025_all.parquet"
con = duckdb.connect()
schema = con.execute(f"""
DESCRIBE SELECT * FROM read_parquet('{path}')
""").fetchdf()
schema

동일 컬럼 데이터인줄 알았는데 같은 동일컬럼의 확장된 파일이었다....

# 공통(같은 유동인구 데이터는 맞음)
- ETL_YMD = 특정날짜
  출발 행정동/도착 행정동(예: O_ADMI_CD, O_ADMI_NM, D_ADMI_CD, D_ADMI_NM)
- PURPOSE = 목적
- SEX_CD = 성별
- AGE_GRP	= 연령대
- CNT	= 이동량 및 집계값들
# 추가 된 것
- O_TIME_CD (출발 시간)
- D_TIME_CD (도착 시간)
- DOW (요일)

즉 이전 13데이터보다 더 정확해진 데이터(시간 기반으로 분석 가능, 언제 얼마나 이동했는지 13은 얼마나 이동했는지만)

In [ ]:
# 날짜 확인
con.execute("""
select
    min(ETL_YMD) AS min_date,
    max(ETL_YMD) AS max_date,
    count(*) AS row_cnt,
    count(ETL_YMD) AS non_null_date
from read_parquet('../data/t25_2023_2025_all.parquet')
""").fetchdf()

In [ ]:
# 날짜 컬럼이 BIGINT 형태이므로 분석을 위해 DATE 타입으로 변환
con.execute("""
COPY (
    SELECT
        CAST(STRPTIME(CAST(ETL_YMD AS VARCHAR), '%Y%m%d') AS DATE) AS ETL_YMD,
        *
    EXCLUDE (ETL_YMD)
    FROM read_parquet('../data/t25_2023_2025_all.parquet')
)
TO '../data/t25_2023_2025_all_clean.parquet' (FORMAT PARQUET);
""")

In [ ]:
con.execute("""
SELECT MIN(ETL_YMD), MAX(ETL_YMD)
FROM read_parquet('../data/t25_2023_2025_all_clean.parquet')
""").fetchdf()

In [ ]:
# 타입 확인
con.execute("""
describe select *
from read_parquet('../data/t25_2023_2025_all_clean.parquet')""").fetchdf()

In [ ]:
con.execute("""
SELECT ETL_YMD
FROM read_parquet('../data/t25_2023_2025_all_clean.parquet')
LIMIT 5
""").fetchdf()

# 컬럼 결측 확인

In [ ]:
con.execute("""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN ETL_YMD IS NULL THEN 1 ELSE 0 END) AS etl_ymd_null,
    SUM(CASE WHEN DOW IS NULL THEN 1 ELSE 0 END) AS dow_null,
    SUM(CASE WHEN O_TIME_CD IS NULL THEN 1 ELSE 0 END) AS o_time_null,
    SUM(CASE WHEN O_CTY_CD IS NULL THEN 1 ELSE 0 END) AS o_cty_cd_null,
    SUM(CASE WHEN O_MEGA_NM IS NULL THEN 1 ELSE 0 END) AS o_mega_null,
    SUM(CASE WHEN O_CTY_NM IS NULL THEN 1 ELSE 0 END) AS o_cty_null,
    SUM(CASE WHEN O_CENTER_X IS NULL THEN 1 ELSE 0 END) AS o_x_null,
    SUM(CASE WHEN O_CENTER_Y IS NULL THEN 1 ELSE 0 END) AS o_y_null,
    SUM(CASE WHEN D_TIME_CD IS NULL THEN 1 ELSE 0 END) AS d_time_null,
    SUM(CASE WHEN D_CTY_CD IS NULL THEN 1 ELSE 0 END) AS d_cty_cd_null,
    SUM(CASE WHEN D_MEGA_NM IS NULL THEN 1 ELSE 0 END) AS d_mega_null,
    SUM(CASE WHEN D_CTY_NM IS NULL THEN 1 ELSE 0 END) AS d_cty_null,
    SUM(CASE WHEN D_CENTER_X IS NULL THEN 1 ELSE 0 END) AS d_x_null,
    SUM(CASE WHEN D_CENTER_Y IS NULL THEN 1 ELSE 0 END) AS d_y_null,
    SUM(CASE WHEN PURPOSE IS NULL THEN 1 ELSE 0 END) AS purpose_null,
    SUM(CASE WHEN TRANS_GB IS NULL THEN 1 ELSE 0 END) AS trans_null,
    SUM(CASE WHEN SEX_CD IS NULL THEN 1 ELSE 0 END) AS sex_null,
    SUM(CASE WHEN AGE_GRP IS NULL THEN 1 ELSE 0 END) AS age_null,
    SUM(CASE WHEN CNT IS NULL THEN 1 ELSE 0 END) AS cnt_null
FROM read_parquet('../data/t25_2023_2025_all_clean.parquet')
""").fetchdf()

결측 확인 결과
- O_MEGA_NM: 4건 (출발 광역시도명)
- O_CTY_NM: 120,422건 (출발 시군구명)
- O_CENTER_X/Y: 각 4건 (출발 좌표)
- D_MEGA_NM: 11건 (도착 광역시도명)
- D_CTY_NM: 119,701건 (도착 시군구명)
- D_CENTER_X/Y: 각 11건 (도착 좌표)

In [ ]:
con.execute("""
SELECT *
FROM read_parquet('../data/t25_2023_2025_all_clean.parquet')
WHERE O_CTY_NM IS NULL 
""").fetchdf()

In [ ]:
con.execute("""
SELECT *
FROM read_parquet('../data/t25_2023_2025_all_clean.parquet')
WHERE D_CTY_NM IS NULL 
""").fetchdf()

In [ ]:
con.execute("""
SELECT *
FROM read_parquet('../data/t25_2023_2025_all_clean.parquet')
WHERE O_MEGA_NM IS NULL 
""").fetchdf()

In [ ]:
con.execute("""
SELECT *
FROM read_parquet('../data/t25_2023_2025_all_clean.parquet')
WHERE D_MEGA_NM IS NULL 
""").fetchdf()

생긴 결측들 느낌이
위치 매핑 실패 일부
좌표/행정코드 연결 오류

컬럼 확인 후 다시 공유

## t25 데이터 결론
- 전체 행: 261,148,533	
- 날짜, 코드, CNT, 성별, 연령, 목적 -> 결측 0
- 행정명/좌표 컬럼에만 결측 존재(t13에서 데이터구조만 다르고 패턴은 동일)

t25는 세종시부분말고는 4건 11건이 99코드로 되어있어 세종시만 unknown으로 매핑 후 확인 및 최종 전처리 완료로 끝

In [ ]:
con.execute("""
COPY (
    SELECT
        * EXCLUDE (O_CTY_NM),

        CASE
            WHEN O_MEGA_NM = '세종특별자치시'
                THEN 'UNKNOWN'
            ELSE O_CTY_NM
        END AS O_CTY_NM

    FROM read_parquet('../data/t25_2023_2025_all_clean.parquet')
)
TO '../data/t25_2023_2025_all_final.parquet'
(FORMAT PARQUET);
""")

In [ ]:
con.execute("""
SELECT
    O_MEGA_NM,
    O_CTY_NM,
    COUNT(*) AS cnt
FROM read_parquet('../data/t25_2023_2025_all_final.parquet')
WHERE O_MEGA_NM = '세종특별자치시'
GROUP BY 1, 2
ORDER BY cnt DESC;
""").fetchdf()

출발지 unknown으로 매핑 잘된걸로 확인 나머지4개는 99코드

도착지의 세종특별자치시 none값을 unknown으로 매핑

In [ ]:
con.execute("""
COPY (
    SELECT
        * EXCLUDE (D_CTY_NM),

        CASE
            WHEN D_MEGA_NM = '세종특별자치시'
                THEN 'UNKNOWN'
            ELSE D_CTY_NM
        END AS D_CTY_NM

    FROM read_parquet('../data/t25_2023_2025_all_final.parquet')
)
TO '../data/t25_2023_2025_all_final_v2.parquet'
(FORMAT PARQUET);
""")

In [ ]:
con.execute("""
SELECT
    D_MEGA_NM,
    D_CTY_NM,
    COUNT(*) AS cnt
FROM read_parquet('../data/t25_2023_2025_all_final_v2.parquet')
WHERE D_MEGA_NM = '세종특별자치시'
GROUP BY 1, 2
ORDER BY cnt DESC;
""").fetchdf()

매핑 잘되었고 11개는 99코드

In [ ]:
con.execute("""
SELECT *
FROM read_parquet('../data/t25_2023_2025_all_final_v2.parquet')
WHERE D_MEGA_NM IS NULL 
""").fetchdf()

# t25 이상치 확인

In [ ]:
con.execute("""
summarize
            select cnt
from read_parquet('../data/t25_2023_2025_all_clean.parquet')""").fetchdf()

- min: 0.93
- max: 2949.68
- 평균: 7.81
- 중앙값: 3.68
- Q25: 2.86
- q50: 3.68
- q75: 6.82
- 표준편차: 18.36
t13이랑 똑같이 오른쪽으로 긴 분포로 뛴다 평균 중앙값 결과 여진히 max값이 2949로 큰값이 존재해 일반 박스플롯으로 하면 이상치로 나타날 수 있어 로그변환 후 박스플롯해서 확인해봐야됨

In [ ]:
# 데이터가 너무커 100000행으로 샘플데이터로 확인해봄
df25 = con.execute("""
SELECT CNT
FROM read_parquet('../data/t25_2023_2025_all_clean.parquet')
USING SAMPLE 100000
""").fetchdf()

In [ ]:
# 일반 박스플롯으로 이상치 확인(비교)
plt.boxplot(df25['CNT'])
plt.title('CNT Boxplot')
plt.show()

In [ ]:
plt.boxplot(np.log1p(df25['CNT']))
plt.title('CNT Boxplot (log scale)')
plt.show()

t25 cnt 이상치 결과
t25 데이터의 CNT 분포는 t13과 유사하게 평균이 중앙값보다 크게 나타나며, 오른쪽으로 긴 분포(right-skewed)를 보였다.
또한 최대값(2949.68)이 상위 분위수(Q3=6.82) 대비 크게 나타나 일부 큰 값이 존재하며, 일반 스케일의 박스플롯에서는 이상치로 표현될 수 있다.
그러나 이는 유동인구 이동량 데이터의 특성에서 기인한 자연스러운 현상으로 판단되며, 분포를 보다 정확히 파악하기 위해 로그 변환 후 시각화를 수행하는 것이 적절하다. 따라서 해당 값들은 제거 대상이 아닌, 유동인구 이동량 데이터의 특성을 반영한 자연스러운 값으로 판단된다.

# t25 범주형 데이터(PURPOSE,TRANS_GB(이동수단 핵심), SEX_CD, AGE_GRP) 이상치 확인

In [ ]:
con.execute("""
SELECT
    PURPOSE,
    COUNT(*) AS cnt,
    COUNT(*) * 1.0 / SUM(COUNT(*)) OVER() AS ratio
FROM read_parquet('../data/t25_2023_2025_all_clean.parquet')
GROUP BY PURPOSE
ORDER BY cnt DESC
""").fetchdf()

6 = 0.45
0 = 0.35
1 = 0.15로 3개가 95%정도로 차지함
해석결과 소수 목적이 대부분의 이동을 차지

In [ ]:
con.execute("""
SELECT
    TRANS_GB,
    COUNT(*) AS cnt,
    COUNT(*) * 1.0 / SUM(COUNT(*)) OVER() AS ratio
FROM read_parquet('../data/t25_2023_2025_all_clean.parquet')
GROUP BY TRANS_GB
ORDER BY cnt DESC
""").fetchdf()

이동수단 이동목적 같이 공유

0 = 0.51
2 = 0.19
1 = 0.15로 이동유형에서도 약 80%이상의 비중을 차지 특히 0번 0.51로 50%이상 차지 시간분석 및 연령분석때 사용하면 될듯?

In [ ]:
con.execute("""
SELECT
    SEX_CD,
    COUNT(*) AS cnt,
    COUNT(*) * 1.0 / SUM(COUNT(*)) OVER() AS ratio
FROM read_parquet('../data/t25_2023_2025_all_clean.parquet')
GROUP BY SEX_CD
ORDER BY cnt DESC
""").fetchdf()

M(남자) → 54%
F(기타) → 24%
W(여자) → 21% t13에서도 보았듯이 F가 약24%로 나와 유심히 살펴봐야할듯 합니다.

In [ ]:
con.execute("""
SELECT
    AGE_GRP,
    COUNT(*) AS cnt,
    COUNT(*) * 1.0 / SUM(COUNT(*)) OVER() AS ratio
FROM read_parquet('../data/t25_2023_2025_all_clean.parquet')
GROUP BY AGE_GRP
ORDER BY AGE_GRP
""").fetchdf()

t13이랑 동일하게 중간연령대에서 약 70%정도 활동량이 높다는 결과로 해석됨

In [ ]:
# 수랑 비율본걸 시각화(샘플데이터 100000으로 추출 후)
df25 = con.execute("""
SELECT PURPOSE
FROM read_parquet('../data/t25_2023_2025_all_clean.parquet')
USING SAMPLE 100000
""").fetchdf()

df25['PURPOSE'].value_counts().sort_index().plot(kind='bar')

In [ ]:
df25 = con.execute("""
SELECT TRANS_GB
FROM read_parquet('../data/t25_2023_2025_all_clean.parquet')
USING SAMPLE 100000
""").fetchdf()

df25['TRANS_GB'].value_counts().sort_index().plot(kind='bar')

알아보고

In [ ]:
df25 = con.execute("""
SELECT SEX_CD
FROM read_parquet('../data/t25_2023_2025_all_clean.parquet')
USING SAMPLE 100000
""").fetchdf()

df25['SEX_CD'].value_counts().sort_index().plot(kind='bar')

In [ ]:
df25 = con.execute("""
SELECT AGE_GRP
FROM read_parquet('../data/t25_2023_2025_all_clean.parquet')
USING SAMPLE 100000
""").fetchdf()

df25['AGE_GRP'].value_counts().sort_index().plot(kind='bar')

확실히 나이쪽은 중간쪽이 높아서 정규분포 형태를 뜀 주요그룹중심으로 접근하는 방법도 좋을 듯?? ??
지금까지 t13 t25확인결과 분석 시 컬럼들 사용할때 이상 없음
t13과 t25는 합칠 수있긴한데 컬럼이 추가된걸 맞춘후 합치거나 해야됨

# t26 데이터 확인

In [ ]:
path = "../data/t26_2023_2025_all_fixed.parquet"
con = duckdb.connect()
schema = con.execute(f"""
DESCRIBE SELECT * FROM read_parquet('{path}')
""").fetchdf()
schema

하... 다다른 데이터였음.... t26은 도착기준 + 체류시간 포함 데이터
- t13/ t25는 이동데이터(어디에서 어디로 이동 그리고 얼마나 이동했는지)
- t26은 체류 데이터(얼마나 머물렀는지 어디에 얼마나 있었는지)

In [ ]:
# 날짜 확인
con.execute("""
select
    min(ETL_YMD) AS min_date,
    max(ETL_YMD) AS max_date,
    count(*) AS row_cnt,
    count(ETL_YMD) AS non_null_date
from read_parquet('../data/t26_2023_2025_all_fixed.parquet')
""").fetchdf()

전체 행: 258637442
날짜는 max까지 이상 무

In [ ]:
# 날짜 컬럼이 BIGINT 형태이므로 분석을 위해 DATE 타입으로 변환
con.execute("""
COPY (
    SELECT
        CAST(STRPTIME(CAST(ETL_YMD AS VARCHAR), '%Y%m%d') AS DATE) AS ETL_YMD,
        *
    EXCLUDE (ETL_YMD)
    FROM read_parquet('../data/t26_2023_2025_all_fixed.parquet')
)
TO '../data/t26_2023_2025_all_clean.parquet' (FORMAT PARQUET);
""")

In [ ]:
con.execute("""
SELECT MIN(ETL_YMD), MAX(ETL_YMD)
FROM read_parquet('../data/t26_2023_2025_all_clean.parquet')
""").fetchdf()

In [ ]:
# 타입 확인
con.execute("""
describe select *
from read_parquet('../data/t26_2023_2025_all_clean.parquet')""").fetchdf()

In [ ]:
con.execute("""
SELECT ETL_YMD
FROM read_parquet('../data/t25_2023_2025_all_clean.parquet')
LIMIT 5
""").fetchdf()

In [ ]:
con.execute("""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN ETL_YMD IS NULL THEN 1 ELSE 0 END) AS etl_ymd_null,
    SUM(CASE WHEN DOW IS NULL THEN 1 ELSE 0 END) AS dow_null,
    SUM(CASE WHEN D_TIME_CD IS NULL THEN 1 ELSE 0 END) AS d_time_null,
    SUM(CASE WHEN D_ADMI_CD IS NULL THEN 1 ELSE 0 END) AS d_admi_cd_null,
    SUM(CASE WHEN D_MEGA_NM IS NULL THEN 1 ELSE 0 END) AS d_mega_null,
    SUM(CASE WHEN D_CTY_NM IS NULL THEN 1 ELSE 0 END) AS d_cty_null,
    SUM(CASE WHEN D_ADMI_NM IS NULL THEN 1 ELSE 0 END) AS d_admi_null,
    SUM(CASE WHEN D_CENTER_X IS NULL THEN 1 ELSE 0 END) AS d_x_null,
    SUM(CASE WHEN D_CENTER_Y IS NULL THEN 1 ELSE 0 END) AS d_y_null,
    SUM(CASE WHEN PURPOSE IS NULL THEN 1 ELSE 0 END) AS purpose_null,
    SUM(CASE WHEN TRANS_GB IS NULL THEN 1 ELSE 0 END) AS trans_null,
    SUM(CASE WHEN DURATION IS NULL THEN 1 ELSE 0 END) AS duration_null,
    SUM(CASE WHEN SEX_CD IS NULL THEN 1 ELSE 0 END) AS sex_null,
    SUM(CASE WHEN AGE_GRP IS NULL THEN 1 ELSE 0 END) AS age_null,
    SUM(CASE WHEN CNT IS NULL THEN 1 ELSE 0 END) AS cnt_null
FROM read_parquet('../data/t26_2023_2025_all_clean.parquet')
""").fetchdf()

- 와 t26은 결측이 없어서 바로 이상치 확인만 하면 됨
- 전체행 258637442중 날짜, 시간, 지역, 목적, 이동유형, 체류시간, 성별, 연령, CNT까지 전부 값이 있음

# t26은 CNT 분포
DURATION 분포
범주형(PURPOSE, TRANS_GB, SEX_CD, AGE_GRP) 이상치 확인

In [ ]:
# cnt 확인
con.execute("""
summarize
            select cnt
from read_parquet('../data/t26_2023_2025_all_clean.parquet')""").fetchdf()

- min: 0.93
- max: 1893.67
- 평균: 5.57
- 중앙값: 3.51
- Q25: 2.78
- q50: 3.50
- q75: 5.91
- 표준편차: 6.81

역시 중앙값보다 평균값이 커서 오른쪽으로 긴 분포 형태를 나타낼거 같음 rmflrh duwjsgl 분위수보다 max값이 1893으로 커서 일반 박스플롯으론 이상치 존재로 판별 할수도 있다고 판단해서 로그변환 후 박스플롯 찍어보기

체류 데이터는 이동 데이터보다 더 안정적인 분포를 가질거 같음 max값이 그래도 다른데이터에 비해 낮음

In [ ]:
# 데이터가 너무커 100000행으로 샘플데이터로 확인해봄
df26 = con.execute("""
SELECT CNT, duration
FROM read_parquet('../data/t26_2023_2025_all_clean.parquet')
USING SAMPLE 100000
""").fetchdf()

In [ ]:
# 일반 박스플롯으로 이상치 확인(비교)
plt.boxplot(df26['CNT'])
plt.title('CNT Boxplot')
plt.show()

In [ ]:
plt.boxplot(np.log1p(df26['CNT']))
plt.title('CNT Boxplot (log scale)')
plt.show()

로그 변환 후 박스플롯 해보니 이상치는 보이지 않음
극단값의 영향이 완화

이제 duration(체류시간) 컬럼 확인

In [ ]:
# duration 확인
con.execute("""
summarize
            select duration
from read_parquet('../data/t26_2023_2025_all_clean.parquet')""").fetchdf()

- min: 30
- max: 1440
- 중앙값: 180(약3시간)
- Q25: 90
- q50: 180
- q75: 359

음... 최소 30분이상은 체류한다 중앙값이 180인걸 보야 대부분 3시간은 있는다로 보여짐 그리고 max:1440인걸보아 24시간이 최대로 보여짐 이미 정제된?? ??
결론 짦게는 30분 길게는 24시간 이걸로 지역이랑 비교하면 좋을듯?? ??

이것도 오른쪽으로 긴 분포형이라 로그변환후 확인

In [ ]:
# 일반 박스플롯으로 이상치 확인(비교)
plt.boxplot(df26['DURATION'])
plt.title('duration Boxplot')
plt.show()

In [ ]:
plt.boxplot(np.log1p(df26['DURATION']))
plt.title('CNT Boxplot (log scale)')
plt.show()

??? ??? ?? ??? ??? ??? ?? ??? ??? ??? ??? ??? ??? ?? ??? ??? ??? ??? ??? ?? ??? ??? ??? ??? ??? ?? ??? ??? ??? ??? ??? ?? ??? ??? ??? ??? ??? ?? ??? ??? ??? ??? ??? ?? ??? ??? ??? ??? ??? ?? ??? ??? ??? ??? ??? ?? ??? ??? ??? ??? ??? ?? ??? ??? ??? ??? ??? ?? ??? ??? ??? ??? ??? ?? ??? ??? ??? ??? ??? ?? ??? ??? ??? ??? ??? ?? ??? ??? ??? ??? ??? ?? ??? ??? ??? ??? ??? ?? ??? ??? ??? ??? ??? ?? ??? ??? ??? ??? ??? ?? ??? ??? ??? ??? ??? ?? ??? ??? ??? ??? ??? ?? ??? ??? ??? ??? ??? ?? ??? ??? ??? ??? ??? ?? ??? ??? ??? ??? ??? ?? ??? ??? ??? ??? ??? ?? ??? ??? ??? ??? ??? ?? ??? ??? ??? ??? ??? ?? ??? ??? ??? ??? ??

In [ ]:
plt.hist(np.log1p(df26['DURATION']), bins=50)
plt.title('Log DURATION Distribution')
plt.show()

음... 체류시간 보는건 박스플롯으로 하면 안되는구나....

로그 변환된 DURATION의 히스토그램을 확인한 결과, 약 log 값 4.0 부근에 데이터가 집중된 것으로 나타났다. 이를 실제 체류시간으로 환산하면 약 50~60분 수준에 해당

# 이제 범주형 데이터 4개 확인

In [ ]:
con.execute("""
SELECT
    PURPOSE,
    COUNT(*) AS cnt,
    COUNT(*) * 1.0 / SUM(COUNT(*)) OVER() AS ratio
FROM read_parquet('../data/t26_2023_2025_all_clean.parquet')
GROUP BY PURPOSE
ORDER BY cnt DESC
""").fetchdf()

6 = 0.43
0 = 0.35
1 = 0.17 T26도 약 95%이상 차지 T25랑 비슷함

In [ ]:
con.execute("""
SELECT
    TRANS_GB,
    COUNT(*) AS cnt,
    COUNT(*) * 1.0 / SUM(COUNT(*)) OVER() AS ratio
FROM read_parquet('../data/t26_2023_2025_all_clean.parquet')
GROUP BY TRANS_GB
ORDER BY cnt DESC
""").fetchdf()

0 = 0.34
1 = 0.22
3 = 0.19
2 = 0.13
7 = 0.09로 5개의 이동수단이 과반수 차지 극 소수만 다른걸 이용

In [ ]:
con.execute("""
SELECT
    SEX_CD,
    COUNT(*) AS cnt,
    COUNT(*) * 1.0 / SUM(COUNT(*)) OVER() AS ratio
FROM read_parquet('../data/t26_2023_2025_all_clean.parquet')
GROUP BY SEX_CD
ORDER BY cnt DESC
""").fetchdf()

똑같이 w가 0.20으로 나와 뺄 수는 없을거 같음 다포함해서 연령대별 확인하는게 좋을거 같음

In [ ]:
con.execute("""
SELECT
    AGE_GRP,
    COUNT(*) AS cnt,
    COUNT(*) * 1.0 / SUM(COUNT(*)) OVER() AS ratio
FROM read_parquet('../data/t26_2023_2025_all_clean.parquet')
GROUP BY AGE_GRP
ORDER BY cnt DESC
""").fetchdf()

AGE_GRP에서 06,04등 으로 인해 문자열로 되어있어 값이 서로다른값으로 찍힘 그래서 형변환을해 정리 해줘야 할듯

In [ ]:
con.execute("""
SELECT
    CAST(AGE_GRP AS INTEGER) AS AGE_GRP,
    COUNT(*) AS cnt,
    COUNT(*) * 1.0 / SUM(COUNT(*)) OVER() AS ratio
FROM read_parquet('../data/t26_2023_2025_all_clean.parquet')
GROUP BY AGE_GRP
ORDER BY AGE_GRP
""").fetchdf()

같은 숫자 2번씩 나오네 하...

In [ ]:
con.execute("""
SELECT
    CAST(TRIM(AGE_GRP) AS INTEGER) AS AGE_GRP,
    COUNT(*) AS cnt,
    COUNT(*) * 1.0 / SUM(COUNT(*)) OVER() AS ratio
FROM read_parquet('../data/t26_2023_2025_all_clean.parquet')
GROUP BY CAST(TRIM(AGE_GRP) AS INTEGER)
ORDER BY AGE_GRP
""").fetchdf()

AGE_GRP 변수는 문자열 형식으로 저장되면서 동일한 연령 구간이 중복 표기된 문제가 있었으나, 정수형으로 변환하여 통합한 결과 정상적인 분포를 확인할 수 있었다.
분석 결과, 중간 연령대(4~7 구간)에 데이터가 집중되어 있으며 특히 5~6 구간의 비중이 가장 높게 나타났다. 이는 체류 데이터 역시 활동량이 높은 주요 연령대 중심으로 구성되어 있음을 보여준다. 중간 연령대 집중 구조 똑같음 위에랑

In [ ]:
# 수랑 비율본걸 시각화(샘플데이터 100000으로 추출 후)
df26 = con.execute("""
SELECT PURPOSE
FROM read_parquet('../data/t26_2023_2025_all_clean.parquet')
USING SAMPLE 100000
""").fetchdf()

df26['PURPOSE'].value_counts().sort_index().plot(kind='bar')

In [ ]:
df26 = con.execute("""
SELECT TRANS_GB
FROM read_parquet('../data/t26_2023_2025_all_clean.parquet')
USING SAMPLE 100000
""").fetchdf()

df26['TRANS_GB'].value_counts().sort_index().plot(kind='bar')

In [ ]:
df26 = con.execute("""
SELECT SEX_CD
FROM read_parquet('../data/t26_2023_2025_all_clean.parquet')
USING SAMPLE 100000
""").fetchdf()

df26['SEX_CD'].value_counts().sort_index().plot(kind='bar')

In [ ]:
df26['AGE_GRP'] = df26['AGE_GRP'].astype(str).str.strip().str.lstrip('0')
df26['AGE_GRP'] = df26['AGE_GRP'].replace('', '0').astype(int)

df26['AGE_GRP'].value_counts().sort_index().plot(kind='bar')
plt.title('AGE_GRP Distribution')
plt.show()

In [ ]:
# agb_grp 문자열 정리 후 integer로 통합 원본은 남기고 1차 전처리 final(T26)
con.execute("""
COPY (
    SELECT
        CAST(TRIM(AGE_GRP) AS INTEGER) AS AGE_GRP,
        *
    EXCLUDE (AGE_GRP)
    FROM read_parquet('../data/t26_2023_2025_all_clean.parquet')
)
TO '../data/t26_2023_2025_all_final.parquet' (FORMAT PARQUET);
""")

In [ ]:
con.execute("""
SELECT
    AGE_GRP,
    COUNT(*) AS cnt,
    COUNT(*) * 1.0 / SUM(COUNT(*)) OVER() AS ratio
FROM read_parquet('../data/t26_2023_2025_all_final.parquet')
GROUP BY AGE_GRP
ORDER BY cnt DESC
""").fetchdf()

t26 형변환까지 완료 후 final 저장

결론: t26은 체류시간 즉 이지역에 얼마나 머물러있나의 데이터이고 t13 t25랑은 직접적으로 파일연결 불가 하지만 분석적으로는 이동 데이터랑 체류데이터로 스토리 이어서 가능 어디로 많이 이동했고 이동한 곳에서 얼마나 머물렀는지로 그리고 t26데이터는 결측이 없어 분석하는데 이상 없음 이상치도 따로 발견 된건 없음

# T27 데이터

In [ ]:
path = "../data/t27_2023_2025_all.parquet"

con = duckdb.connect()

schema = con.execute(f"""
DESCRIBE SELECT * FROM read_parquet('{path}')
""").fetchdf()

schema

T26이랑 같은 동일컬럼의 데이터이고 이상치 및 결측치 확인후 데이터 유니온해 T26~27파일 생성해도 될거 같음

In [ ]:
# 일단 날짜 확인
con.execute("""
select
    min(ETL_YMD) AS min_date,
    max(ETL_YMD) AS max_date,
    count(*) AS row_cnt,
    count(ETL_YMD) AS non_null_date
from read_parquet('../data/t27_2023_2025_all.parquet')
""").fetchdf()

전체 행: 278322883

In [ ]:
# 날짜 컬럼이 BIGINT 형태이므로 분석을 위해 DATE 타입으로 변환
con.execute("""
COPY (
    SELECT
        CAST(STRPTIME(CAST(ETL_YMD AS VARCHAR), '%Y%m%d') AS DATE) AS ETL_YMD,
        *
    EXCLUDE (ETL_YMD)
    FROM read_parquet('../data/t27_2023_2025_all.parquet')
)
TO '../data/t27_2023_2025_all_clean.parquet' (FORMAT PARQUET);
""")

In [ ]:
con.execute("""
SELECT MIN(ETL_YMD), MAX(ETL_YMD)
FROM read_parquet('../data/t27_2023_2025_all_clean.parquet')
""").fetchdf()

In [ ]:
# 타입 확인
con.execute("""
describe select *
from read_parquet('../data/t27_2023_2025_all_clean.parquet')""").fetchdf()

In [ ]:
con.execute("""
SELECT ETL_YMD
FROM read_parquet('../data/t27_2023_2025_all_clean.parquet')
LIMIT 5
""").fetchdf()

In [ ]:
# 똑같이 결측확인
con.execute("""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN ETL_YMD IS NULL THEN 1 ELSE 0 END) AS etl_ymd_null,
    SUM(CASE WHEN DOW IS NULL THEN 1 ELSE 0 END) AS dow_null,
    SUM(CASE WHEN O_TIME_CD IS NULL THEN 1 ELSE 0 END) AS o_time_null,
    SUM(CASE WHEN O_ADMI_CD IS NULL THEN 1 ELSE 0 END) AS o_admi_cd_null,
    SUM(CASE WHEN O_MEGA_NM IS NULL THEN 1 ELSE 0 END) AS o_mega_null,
    SUM(CASE WHEN O_CTY_NM IS NULL THEN 1 ELSE 0 END) AS o_cty_null,
    SUM(CASE WHEN O_ADMI_NM IS NULL THEN 1 ELSE 0 END) AS o_admi_null,
    SUM(CASE WHEN O_CENTER_X IS NULL THEN 1 ELSE 0 END) AS o_x_null,
    SUM(CASE WHEN O_CENTER_Y IS NULL THEN 1 ELSE 0 END) AS o_y_null,
    SUM(CASE WHEN PURPOSE IS NULL THEN 1 ELSE 0 END) AS purpose_null,
    SUM(CASE WHEN TRANS_GB IS NULL THEN 1 ELSE 0 END) AS trans_null,
    SUM(CASE WHEN DURATION IS NULL THEN 1 ELSE 0 END) AS duration_null,
    SUM(CASE WHEN SEX_CD IS NULL THEN 1 ELSE 0 END) AS sex_null,
    SUM(CASE WHEN AGE_GRP IS NULL THEN 1 ELSE 0 END) AS age_null,
    SUM(CASE WHEN CNT IS NULL THEN 1 ELSE 0 END) AS cnt_null
FROM read_parquet('../data/t27_2023_2025_all_clean.parquet')
""").fetchdf()

결측 없음 이상치 확인 후 데이터 통합

똑같이 cnt부터 이상치 확인(boxplot까지)

In [ ]:
# cnt 확인
con.execute("""
summarize
            select cnt
from read_parquet('../data/t27_2023_2025_all_clean.parquet')""").fetchdf()

동일컬럼에 행 다름
- min: 0.93
- max: 1224
- 평균: 5.06
- 표준편차: 5.79
확실히 이동데이터보단 체류데이터가 더 안정적인 패턴을 가지고 있음 표준편차가 크지 않음
27 데이터의 CNT 분포를 확인한 결과, 기존 t25 및 t26과 유사한 분포 형태를 보이며, 평균과 표준편차 수준에서도 큰 차이를 보이지 않았다.
이는 동일한 기준으로 수집된 데이터로 판단되며, 구조 및 성격이 일관된 데이터로 볼 수 있다. 따라서 컬럼 구조가 동일한 다른 데이터와의 통합이 가능한 형태로 판단.

In [ ]:
# 데이터가 너무커 100000행으로 샘플데이터로 확인해봄
df27 = con.execute("""
SELECT CNT, duration
FROM read_parquet('../data/t27_2023_2025_all_clean.parquet')
USING SAMPLE 100000
""").fetchdf()

In [ ]:
# 일반 박스플롯으로 이상치 확인(비교)
plt.boxplot(df27['CNT'])
plt.title('CNT Boxplot')
plt.show()

값이 큰게 1개 있는거 같아 로그변한 후에도 확인

In [ ]:
# 로그변환 박스플롯
plt.boxplot(np.log1p(df27['CNT']))
plt.title('CNT Boxplot (log scale)')
plt.show()

로그 변환 후 박스플롯에서도 일부 이상치가 관찰되었으나, 이는 데이터 오류가 아닌 분포 특성에 의해 발생한 상대적으로 큰 값으로 판단된다.
해당 값들은 유동인구 데이터의 자연스러운 변동 범위에 포함되며, 별도의 이상치 제거 없이 분석에 활용하는 것이 적절하다.

In [ ]:
# duration 확인
con.execute("""
summarize
            select duration
from read_parquet('../data/t27_2023_2025_all_clean.parquet')""").fetchdf()

duration도 t26이랑 동일한 걸로 확인 min30.0 max1440으로 일치

- 항목	t26	      t27
- min	30	      30
- max  1440	     1440
- 평균 ~257	     ~253
- 중앙값 ~180	 ~180
- Q3   ~360	     ~356
- std  ~211	     ~205

동일 분포로 봐도 무방함

In [ ]:
# 일반 hist
plt.hist(df27['DURATION'], bins=50)
plt.title('duration')
plt.show()

In [ ]:
# 로그 변환 후 hist
plt.hist(np.log1p(df27['DURATION']), bins=50)
plt.title('Log DURATION Distribution')
plt.show()

DURATION 변수에 대해 일반 스케일과 로그 변환 스케일에서 분포를 비교한 결과, 모두에서 데이터의 전반적인 분포 구조를 확인할 수 있었다.
로그 변환 시 분포가 보다 안정적으로 나타나기는 하나, 일반 스케일에서도 극단적인 이상치로 판단되는 값은 발견되지 않았으며, 체류시간 특성에 따른 자연스러운 분포로 해석

t13,t25,t26,27은
- 이동 → 체류 연결
- 지역 분석
- 행동 패턴

확인 결과: t26,t27은 합할 예정이었으나 도착이랑 출발이 나눠져있어 합쳤다가 데이터가 꼬일 수 있어 그대로 두고 분석하는게 좋다고 판단됨.
t26은 도착 기준의 체류 데이터를 중심으로 구성된 반면, t27은 출발 기준의 이동 데이터를 포함하고 있어 동일한 기준으로 직접 통합할 경우 데이터 해석이 왜곡될 가능성이 존재

In [ ]:
con.execute("""
SELECT PURPOSE, COUNT(*) AS cnt,
COUNT(*) * 1.0 / SUM(COUNT(*)) OVER() AS ratio
FROM read_parquet('../data/t27_2023_2025_all_clean.parquet')
GROUP BY PURPOSE
ORDER BY cnt DESC
""").fetchdf()

PURPOSE, TRANS_GB: 공식 정의 없음 → 추정 기반 분석

In [ ]:
con.execute("""
SELECT TRANS_GB, COUNT(*) AS cnt,
COUNT(*) * 1.0 / SUM(COUNT(*)) OVER() AS ratio
FROM read_parquet('../data/t27_2023_2025_all_clean.parquet')
GROUP BY TRANS_GB
ORDER BY cnt DESC
""").fetchdf()

In [ ]:
con.execute("""
SELECT SEX_CD, COUNT(*) AS cnt,
COUNT(*) * 1.0 / SUM(COUNT(*)) OVER() AS ratio
FROM read_parquet('../data/t27_2023_2025_all_clean.parquet')
GROUP BY SEX_CD
ORDER BY cnt DESC
""").fetchdf()

SEX_CD = W: 미확인 코드 → 기타로 유지

In [ ]:
con.execute("""
SELECT AGE_GRP, COUNT(*) AS cnt,
COUNT(*) * 1.0 / SUM(COUNT(*)) OVER() AS ratio
FROM read_parquet('../data/t27_2023_2025_all_clean.parquet')
GROUP BY AGE_GRP
ORDER BY cnt DESC
""").fetchdf()

In [ ]:
# agb_grp 문자열 정리 후 integer로 통합 원본은 남기고 1차 전처리 final(T26)마찬가지로 정리 후 이상치 확인
con.execute("""
COPY (
    SELECT
        CAST(TRIM(AGE_GRP) AS INTEGER) AS AGE_GRP,
        *
    EXCLUDE (AGE_GRP)
    FROM read_parquet('../data/t27_2023_2025_all_clean.parquet')
)
TO '../data/t27_2023_2025_all_final.parquet' (FORMAT PARQUET);
""")

In [ ]:
con.execute("""
SELECT
    AGE_GRP,
    COUNT(*) AS cnt,
    COUNT(*) * 1.0 / SUM(COUNT(*)) OVER() AS ratio
FROM read_parquet('../data/t27_2023_2025_all_final.parquet')
GROUP BY AGE_GRP
ORDER BY cnt DESC
""").fetchdf()

In [ ]:
# 이제 각 정리한 전처리 파일 중복값 확인
con.execute("""
SELECT COALESCE(SUM(cnt - 1), 0) AS duplicated_rows
FROM (
    SELECT
        ETL_YMD, O_ADMI_CD, O_MEGA_NM, O_CTY_NM, O_ADMI_NM,
        O_CENTER_X, O_CENTER_Y,
        D_ADMI_CD, D_MEGA_NM, D_CTY_NM, D_ADMI_NM,
        D_CENTER_X, D_CENTER_Y,
        PURPOSE, SEX_CD, AGE_GRP, CNT,
        COUNT(*) AS cnt
    FROM read_parquet('../data/t13_2023_2025_all_clean.parquet')
    GROUP BY ALL
    HAVING COUNT(*) > 1
) AS dup
""").fetchdf()

중복 없음.

- duplicate_groups: 중복 패턴 종류 수
- total_rows_in_duplicate_groups: 중복이 발생한 그룹들에 포함된 전체 행 수
- duplicated_rows: 실제 제거 대상 중복 행 수

In [ ]:
con.execute("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(DISTINCT (
        ETL_YMD, DOW, O_TIME_CD, O_CTY_CD,
        O_MEGA_NM, O_CTY_NM, O_CENTER_X, O_CENTER_Y,
        D_TIME_CD, D_CTY_CD, D_MEGA_NM, D_CTY_NM,
        D_CENTER_X, D_CENTER_Y,
        PURPOSE, TRANS_GB, SEX_CD, AGE_GRP, CNT
    )) AS duplicated_rows
FROM read_parquet('../data/t25_2023_2025_all_clean.parquet')
""").fetchdf()

In [ ]:
con.execute("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(DISTINCT (
        ETL_YMD, DOW, D_TIME_CD, D_ADMI_CD,
        D_MEGA_NM, D_CTY_NM, D_ADMI_NM,
        D_CENTER_X, D_CENTER_Y,
        PURPOSE, TRANS_GB, DURATION,
        SEX_CD, AGE_GRP, CNT
    )) AS duplicated_rows
FROM read_parquet('../data/t26_2023_2025_all_final.parquet')
""").fetchdf()

In [ ]:
con.execute("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(DISTINCT (
        ETL_YMD, DOW, O_TIME_CD, O_ADMI_CD,
        O_MEGA_NM, O_CTY_NM, O_ADMI_NM,
        O_CENTER_X, O_CENTER_Y,
        PURPOSE, TRANS_GB, DURATION,
        SEX_CD, AGE_GRP, CNT
    )) AS duplicated_rows
FROM read_parquet('../data/t27_2023_2025_all_final.parquet')
""").fetchdf()

# t13,t25,t26,t27 결론

| 원본 테이블   | 원본 컬럼    | 변수 후보명                | 의미                   | 사용 목적             |
| -------- | -------- | --------------------- | -------------------- | ----------------- |
| t13 (통신) | CNT      | 이동량 (mobility_volume) | 특정 조건에서 발생한 이동 횟수/규모 | **집계변수 (핵심 KPI)** |
| t13 (통신) | DURATION | 체류시간 (stay_duration)  | 이동 또는 체류 시간          | **설명변수**          |
| t13 (통신) | PURPOSE  | 이동목적 (trip_purpose)   | 이동의 목적 (코드 기반)       | **설명변수 (범주형)**    |
| t13 (통신) | TRANS_GB | 이동수단 (transport_type) | 이동 수단 유형             | **설명변수 (범주형)**    |
| t13 (통신) | SEX_CD   | 성별코드 (gender_code)    | 성별 또는 기타 코드          | **보조변수 (세그먼트용)**  |
| t13 (통신) | AGE_GRP  | 연령대 (age_group)       | 연령 구간                | **보조변수 (세그먼트용)**  |


- CNT = 결과값 (타겟 느낌)
→ “얼마나 이동했는가” → 분석의 중심
- DURATION / PURPOSE / TRANS_GB = 원인 변수
→ “왜 이동했는가 / 어떻게 이동했는가”
- SEX_CD / AGE_GRP = 세그먼트 변수
→ “누가 이동했는가”

| 컬럼       | 코드값    | 해석                    | 상태                 |
| -------- | ------ | --------------------- | ------------------ |
| PURPOSE  | (코드값들) | 이동 목적 추정 (예: 출근/쇼핑 등) | **추정 (확정 아님)**     |
| TRANS_GB | (코드값들) | 이동수단 (도보/차량 등 추정)     | **추정 (확정 아님)**     |
| SEX_CD   | M      | 남성 (일반적 해석)           | **추정**             |
| SEX_CD   | F      | 여성 (일반적 해석)           | **추정**             |
| SEX_CD   | W      | 미확인 코드                | **미확인 (기타로 유지)**   |
| AGE_GRP  | 숫자코드   | 연령 구간                 | **추정 (구간 범위 불명확)** |


CNT를 중심으로 이동량을 보고,
DURATION·PURPOSE·TRANS_GB로 원인을 설명하고,
SEX_CD·AGE_GRP로 사용자 특성을 분석하는 구조로 정리 완료

| 데이터셋 | 기간 | 총 행 수 | 후보 파일 |
|----------|------|----------|-----------|
| t13 | 2023 ~ 2025 | 279,996,851 | t13_2023_2025_all_final_v2.parquet |
| t25 | 2023 ~ 2025 | 261,148,533 | t25_clean.parquet |

| 구분 | t13 | t25 |
|------|-----|-----|
| 기준 컬럼 | ADMI_CD (행정동) | CTY_CD (시군구) |
| 출발지 매핑 | 완료 | 완료 |
| 도착지 매핑 | 완료 | 완료 |
| 코드 기준 보정 | 동일 코드 기반 채움 | 불필요 (구조 단순) |
| 좌표 보정 | 완료 | 완료 |
| 데이터 손실 | 없음 | 없음 |

| 유형 | 처리 방식 |
|------|----------|
| 세종특별자치시 | 시군구 없음 → UNKNOWN 처리 |
| 코드 매핑 누락 (t13) | 동일 코드 기준으로 보정 |
| 일반 결측 | 매핑으로 전부 해결 |
| 미확인 코드 (99) | 원본 유지 |

| 항목 | 상태 |
|------|------|
| 행정코드 매핑 | 완료 |
| 결측 보정 | 완료 |
| 구조적 결측 처리 | 완료 |
| 데이터 품질 | 안정 |
| 분석 가능 여부 | 즉시 가능 |

## 기존 작업 기록: 기존 1차_전처리_small.ipynb

## 노트북 정리

- t4~t11 쪽 통신 데이터를 먼저 가볍게 확인한 전처리 노트북.
- 날짜 컬럼을 분석 가능한 형태로 바꾸고, CNT 분포와 로그 변환 그래프를 보면서 이동량 특성을 확인함.
- PURPOSE, TRANS_GB, SEX_CD, AGE_GRP 같은 범주형 변수의 값 범위와 결측 여부를 확인함.
- CNT가 한쪽으로 긴 분포를 보여도 통신 이동량 데이터 특성상 자연스러운 큰 값일 수 있어서 바로 제거하지 않는 쪽으로 정리함.
- 전체 큰 데이터로 넘어가기 전에 작은 단위 테이블 구조를 먼저 파악한 파일로 보면 됨.


In [ ]:
%pip install matplotlib seaborn

In [ ]:
!pip install duckdb

In [ ]:
import pandas as pd
import duckdb
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import numpy as np
import sys

con = duckdb.connect()

In [ ]:
def quick_check(df):
    print("shape:", df.shape) # 행 확인
    print("\n[columns]") # 컬럼확인
    print(df.columns.tolist())

    print("\n[dtypes]") # 타입확인
    print(df.dtypes)

    print("\n[missing]")  # 결측값 확인
    print(df.isnull().sum())

    print("\n[duplicate rows]") # 중복값 확인
    print(df.duplicated().sum())

    print("\n[numeric summary]") # 수치형 확인
    display(df.describe())

    print("\n[categorical summary]") # 범주 확인
    display(df.describe(include=['object', 'string']))

# t4전처리

In [ ]:
# 함수적용해서 t4 결측/행 및 타입 확인
df4 = pd.read_parquet('../data/t4_all.parquet')
quick_check(df4)

In [ ]:
df4.isna().sum()

In [ ]:
df4.head()

# t4 확인결과
# T4 전처리 결과 요약

- 전체 행 수: 1,211,106
- 컬럼 수: 12개

## 1. 데이터 품질
- 결측값: 없음
- 중복값: 없음
→ 전반적으로 데이터 품질 양호

## 2. 날짜 컬럼
- ETL_YM은 YYYYMM 형태의 정수형 데이터
→ 시계열 분석을 위해 datetime 형으로 변환 필요

## 3. 범주형 데이터 분포
- DOW: 7개 (요일 데이터로 정상)
- D_MEGA_NM: 1개 (경기도로 동일)
→ 지역 범위가 제한된 데이터

- D_CTY_NM: 3개
→ 일부 지역만 포함된 데이터

- SEX_CD: 3개
→ 성별 코드 3종 존재 (추가 해석 필요)

## 4. 최빈값 (top)
- DOW: 금
- D_MEGA_NM: 경기도
- D_CTY_NM: 성남시 분당구
- SEX_CD: M

## 5. 해석 및 특이사항
- 데이터는 경기도 내 일부 지역에 한정됨
- 범주형 변수는 코드 기반으로 구성되어 있음
- SEX_CD, PURPOSE 등은 코드 정의 확인 필요

## 6. 결론
- 결측 및 중복이 없는 안정적인 데이터이며,
- 날짜 형변환 및 코드 변수 해석 후 분석에 바로 활용 가능
- cnt변수는 이동량(집계값)으로 해석되며 이상치 확인해보고 이상없으면 변수로 활용가능
- 이 데이터는 특정 월/요일/시간대에, 특정 지역에서, 특정 성별·연령대 사람들이, 특정 목적을 가지고 이동한 횟수를 cnt로 집계한 데이터
- 시간·공간·인구 특성을 모두 반영한 분석이 가능

일단 날짜 컬럼부터 date로 형 변환

In [ ]:
df4.info()

In [ ]:
df4['ETL_YM'] = pd.to_datetime(df4['ETL_YM'].astype(str), format='%Y%m')

In [ ]:
# 날짜 범위 이상없는지 확인
df4['ETL_YM'].min(), df4['ETL_YM'].max()

월 단위 집계 데이터

날짜 컬럼 형변환 완료

cnt 이상치 확인

In [ ]:
df4['CNT'].describe()

cnt의 최솟값 최댓값이 1~ 1000정도로 오른쪽으로 긴 분포를 뛸거 같음 평균값과 중앙값의 차이가 큼 max값이 75761로 엄청 큼 많은 사람이 이동한 시간대?일 가능성이 있어보임 그래서 로그변환 후 확인해보는것도 좋을거 같음

In [ ]:
#일반 박스플롯(비교)
plt.figure()
plt.boxplot(df4['CNT'])
plt.title('CNT Boxplot')
plt.show()

확실히 이상치로 나오는느낌이 강함 일반박스플롯찍었을때

In [ ]:
# 진짜 이상치인지 확인
df4.sort_values('CNT', ascending=False).head(10)

time이 8~9인걸로 보아 출근시간대에 특정연령대의 사람들이 집중적으로 이동을 많이 함 이상치로 보기는 힘들거 같음

In [ ]:
# 로그변환 후 박스플롯
plt.figure()
plt.boxplot(np.log1p(df4['CNT']))
plt.title('CNT (log scale) Boxplot')
plt.show()

차이가 너무 커 로그변환하니 박스플롯 안찍혀서 hist로 봐야 할듯

In [ ]:
plt.figure()
plt.hist(df4['CNT'], bins=50)
plt.title('CNT Distribution')
plt.xlabel('CNT')
plt.ylabel('Frequency')
plt.show()

- CNT 변수는 대부분 작은 값에 집중되어 있으나, 일부 구간에서 큰 값이 발생하는 오른쪽 꼬리(long-tail) 분포를 보임
- 이는 특정 시간대 및 지역에서 이동량이 집중되는 집계 데이터의 특성으로 판단됨

# t4범주형 데이터 확인

In [ ]:
df4['SEX_CD'].value_counts(dropna=False)

역시나 w비중이 생각보다 큼..SEX_CD는 M, F 외에 W 값이 포함되어 있으며, 해당 값은 미확인 또는 기타 범주로 판단되어 별도로 유지함

In [ ]:
df4['AGE_GRP'].value_counts().sort_index()

대대분 골고루 퍼져있는 느낌인데 11이랑 12가 극소수로 나타남
- AGE_GRP는 1~10 구간에 주로 분포하며, 11~12 구간은 소수 관측됨
- 전반적으로 정상적인 연령 분포로 판단됨

In [ ]:
df4['PURPOSE'].value_counts().sort_index()

0,1,2,3,5,6 존재
(4 없음)
- PURPOSE는 0,1,2,3,5,6으로 구성되어 있으며 일부 코드(4)는 존재하지 않음
- 코드 의미는 명확하지 않아 추후 정의 확인 필요

In [ ]:
df4['DOW'].value_counts()

범주형 변수 분석 결과

- DOW는 모든 요일이 고르게 분포되어 있어 데이터 편향이 없음
- SEX_CD는 M, F 외에 W 값이 포함되어 있으며, 미확인 또는 기타 범주로 판단됨
- AGE_GRP는 대부분 1~10 구간에 분포하며 일부 고연령 구간(11~12)은 소수 존재
- PURPOSE는 일부 코드가 누락된 상태로 나타나며, 코드 정의 확인이 필요함

## 7. 데이터 정의

- t4 데이터는 시간·지역·인구 특성에 따른 이동량을 분석하기 위한 집계 데이터임
- 주요 변수: CNT(이동량), D_TIME_CD(시간대), AGE_GRP(연령대)

이상없어 t4 date.final로 저장

In [ ]:
df4.to_parquet('../data/t4_2023_2025_all_date_final.parquet', index=False)

In [ ]:
check = pd.read_parquet('../data/t4_2023_2025_all_date_final.parquet')
check.info()

# t5 데이터

In [ ]:
# 함수적용해서 t5 결측/행 및 타입 확인
df5 = pd.read_parquet('../data/t5_all.parquet')
quick_check(df5)

t4 대비 더 세밀한 데이터 월일까지 표현되어 있고 행정동까지 나와있어 데이터도 훨씬 큼 t4는 큰틀로만들어진 데이터 t5는 더세밀한데이터 그리고 똑같이 날짜는 int로 되어있어 date로 변환 해야됨
- 전체 행 및 컬럼 수: 43306619, 12

In [ ]:
df5.head()

In [ ]:
df5.info()

In [ ]:
# etl_ymd date로 형변환
df5['ETL_YMD'] = pd.to_datetime(df5['ETL_YMD'].astype(str), format='%Y%m%d')

In [ ]:
df5.head()

In [ ]:
df5.head()

In [ ]:
df5.info()

In [ ]:
df5['ETL_YMD'].min(), df5['ETL_YMD'].max()

# 확인 결과
- T5 데이터는 T4 대비 일 단위 및 행정동 단위로 세분화된 데이터로, 보다 정밀한 분석이 가능함
- 결측값 및 중복값이 없는 안정적인 데이터로 확인됨
- CNT 및 범주형 변수 분포 또한 전반적으로 정상적인 패턴을 보임
- 따라서 추가적인 결측 처리나 이상치 제거 없이, 1차 전처리 단계에서 바로 분석에 활용 가능한 데이터로 판단됨

In [ ]:
df5['CNT'].describe()

In [ ]:
df5.groupby(['SEX_CD', 'AGE_GRP'])['CNT'].sum()

- 성별 및 연령대 조합별 이동량을 집계한 결과, 중간 연령대(AGE_GRP 4~7)에서 높은 이동량이 나타나며, 이는 경제활동 인구 중심의 이동 패턴으로 해석됨
- 극단 연령대에서는 상대적으로 낮은 이동량을 보여 자연스러운 분포를 나타냄
- 특정 조합에서 비정상적인 값은 관측되지 않아 데이터 이상은 없는 것으로 판단됨

## 결론
- T5 데이터는 T4 대비 일 단위 및 행정동 단위로 세분화된 데이터로, 보다 정밀한 분석이 가능함
- 결측값 및 중복값이 없는 안정적인 데이터로 확인됨
- CNT 변수는 평균(약 36)과 중앙값(약 14)의 차이가 존재하나, 최대값(약 6,650) 또한 특정 시간대 및 지역에 이동량이 집중된 결과로 해석 가능하여 이상치로 보기 어려움
- 범주형 변수(SEX_CD, AGE_GRP, PURPOSE, DOW)는 정상적인 범주 범위 내에서 분포하며, 특정 값에 대한 비정상적인 치우침이나 이상값은 관측되지 않음
- 따라서 본 데이터는 추가적인 결측 처리나 이상치 제거 없이, 1차 전처리 단계에서 바로 분석에 활용 가능한 데이터로 판단됨

In [ ]:
# date변환 파일 저장
df5.to_parquet('../data/t5_2023_2025_all_date_final.parquet', index=False)

# t6 데이터
- 본 데이터는 일 단위 및 행정동 단위에서 시간대, 인구 특성, 이동 목적에 따른 이동량을 집계한 데이터임
- 핵심 변수:
CNT → 이동량,
D_TIME_CD → 시간대,
D_ADMI_NM → 위치

음... 이대로는 해석까지 다 할려니 시간이 너무 오래걸려 결측치 및 이상치확인 cnt랑 범주컬럼들 정상인지만 확인 후 사용가능한 데이터로만 변환 후 추후 해석하는 걸로 그리고 이데이터가 어떤 데이터인지 핵심변수가 뭔지까지만 하는걸로 방향잡겠습니다.

In [ ]:
df6 = pd.read_parquet('../data/t6_all.parquet')
quick_check(df6)

확인 해보니 기존 t4랑 동일컬럼인데 1개의 컬럼이 추가됨(TRANS_GB (이동수단))

날짜 컬럼은 int 원본은 살리고 date로 파생변수 생성

In [ ]:
df6.head()

In [ ]:
df6['ETL_YM'] = pd.to_datetime(df6['ETL_YM'].astype(str), format='%Y%m')

In [ ]:
df6.head()

In [ ]:
df6.head()

In [ ]:
df6.info()

## 전처리 결과 요약
- 전체 행 및 컬럼 수: 1927203, 12
- 결측값: 전 컬럼에서 결측값이 존재하지 않음
- 중복값: 중복 데이터 없음
- CNT 변수: 평균(약 810)과 중앙값(약 200)의 차이가 존재하며 일부 큰 값(max 약 41,020)이 확인되나, 시간대·지역·이동수단 조건에 따른 집계 데이터 특성상 자연스러운 분포로 판단됨
- 범주형 변수: 
  - DOW는 7개 요일이 고르게 분포
  - SEX_CD는 M, F, W로 구성되어 있으며 비정상적인 코드 없음
  - TRANS_GB는 0~7 범위의 값으로 구성되어 있으며 이상값 없음
## 데이터 정의
- 본 데이터는 시간, 지역, 인구 특성 및 이동수단(TRANS_GB)에 따른 이동량을 집계한 데이터임
- 핵심 변수: CNT(이동량), D_TIME_CD(시간대), TRANS_GB(이동수단)
## 결론
- 결측 및 중복이 없고, CNT 및 범주형 변수 모두 정상 범위 내에서 분포함
- 따라서 추가적인 결측 처리나 이상치 제거 없이 1차 전처리 단계에서 바로 활용 가능한 데이터로 판단됨

- 본 단계에서는 데이터 품질 확인(결측, 중복, 이상치 여부) 및 구조 파악에 집중하였으며,
- 변수 간 관계 및 해석은 추후 EDA 단계에서 상세히 분석할 예정임(현재 남은 파일들 다 이런식의 느낌으로 해석 예정)

In [ ]:
# date변환 파일 저장
df6.to_parquet('../data/t6_2023_2025_all_date_final.parquet', index=False)

# t7 데이터

In [ ]:
# 함수적용해서 t7 결측/행 및 타입 확인
df7 = pd.read_parquet('../data/t7_all.parquet')
quick_check(df7)

- 일 단위 + 행정동 + 이동수단 기준 이동량 데이터 (T5 + 교통수단 버전)
- 언제 + 어디 + 누가 + 어떤 이동수단으로 이동했는지

In [ ]:
df7.head()

똑같이 날짜컬럼 date로 파생변수 생성

In [ ]:
df7['ETL_YMD'] = pd.to_datetime(df7['ETL_YMD'].astype(str), format='%Y%m%d')

In [ ]:
df7.head()

In [ ]:
df7.info()

In [ ]:
df7.head()

## 전처리 결과 요약
- 전체 행 및 컬럼 수: 68932750, 12
- 결측값 및 중복값이 존재하지 않아 데이터 품질이 양호함
- CNT 변수는 평균(약 22)과 중앙값(약 10)의 차이가 존재하나, 최대값(약 3,346) 또한 집계 데이터 특성상 자연스러운 범위로 판단됨
- 범주형 변수(SEX_CD, AGE_GRP, TRANS_GB)는 정상 범위 내에서 분포하며 비정상적인 값은 관측되지 않음
## 데이터 정의
- 본 데이터는 일 단위 및 행정동 단위에서 이동수단을 포함한 이동량을 집계한 데이터임
- 핵심 변수: CNT(이동량), D_TIME_CD(시간대), TRANS_GB(이동수단)
## 결론
- 데이터 품질 및 변수 분포가 전반적으로 안정적이며,
- 추가적인 결측 처리나 이상치 제거 없이 1차 전처리 단계에서 바로 활용 가능한 데이터로 판단됨
- 변수 해석 및 관계 분석은 추후 EDA 단계에서 진행 예정임

In [ ]:
# date변환 파일 저장
df7.to_parquet('../data/t7_2023_2025_all_date_final.parquet', index=False)

# t8 데이터

In [ ]:
# 함수적용해서 t8 결측/행 및 타입 확인
df8 = pd.read_parquet('../data/t8_all.parquet')
quick_check(df8)

## 전처리 결과 요약
- 전체 데이터는 1,638,511행, 12개 컬럼으로 구성됨
- 결측값 및 중복값이 존재하지 않아 데이터 품질이 양호함
- CNT 변수는 평균(약 954)과 중앙값(약 84)의 차이가 존재하는 비대칭 분포를 보이며, 최대값(약 47,264) 또한 시간대 및 지역에 따른 집계 데이터 특성상 자연스러운 범위로 판단됨
- 범주형 변수(DOW, SEX_CD)는 정상 범위 내에서 분포하며 비정상적인 값은 관측되지 않음

## 데이터 정의
- 본 데이터는 월 단위에서 출발지 기준 시간, 지역, 인구 특성 및 이동 목적에 따른 이동량을 집계한 데이터임
- 핵심 변수: CNT(이동량), O_TIME_CD(출발 시간대), PURPOSE(이동 목적)

## 결론
- 데이터 품질 및 변수 분포가 전반적으로 안정적이며,
- 추가적인 결측 처리나 이상치 제거 없이 1차 전처리 단계에서 바로 활용 가능한 데이터로 판단됨
- 변수 해석 및 관계 분석은 추후 EDA 단계에서 진행 예정임

ETL_YM은 날짜형으로 변환 필요

In [ ]:
df8.head()

In [ ]:
df8['ETL_YM'] = pd.to_datetime(df8['ETL_YM'].astype(str), format='%Y%m')

In [ ]:
df8.head()

In [ ]:
df8.info()

In [ ]:
df8.head()

In [ ]:
# date변환 파일 저장
df8.to_parquet('../data/t8_2023_2025_all_date_final.parquet', index=False)

# t9 데이터

In [ ]:
# 함수적용해서 t9 결측/행 및 타입 확인
df9 = pd.read_parquet('../data/t9_all.parquet')
quick_check(df9)

In [ ]:
df9.isna().sum()

In [ ]:
# 똑같이 날짜컬럼 int로 되어있는거 date로 변환
df9['ETL_YMD'] = pd.to_datetime(df9['ETL_YMD'].astype(str), format='%Y%m%d')

In [ ]:
df9.head()

## 전처리 결과 요약
- 전체 데이터는 **50,298,816행, 12개 컬럼**으로 구성됨  
- 결측값 및 중복값이 존재하지 않아 데이터 품질이 매우 양호함  
- `CNT` 변수는 평균(약 31.0) 대비 중앙값(약 11.6)이 낮은 **우측 치우침(비대칭 분포)**을 보임  
- 최대값(약 3,965)은 시간대 및 지역별 집계 데이터 특성을 고려할 때 **자연스러운 범위**로 판단됨  
- 범주형 변수(`O_MEGA_NM`, `O_CTY_NM`, `O_ADMI_NM`, `SEX_CD`)는 일관된 값을 가지며 비정상 값은 관측되지 않음  

---

## 데이터 정의
- 본 데이터는 **일 단위 기준으로 출발지의 시간, 지역, 인구 특성 및 이동 목적에 따른 이동량을 집계한 데이터**임  
- 특정 시간대 및 지역에서의 **이동 패턴 및 수요 특성 분석**을 목적으로 활용 가능  

### 주요 변수
- `CNT`: 이동량 (집계된 유동 인구 규모)
- `O_TIME_CD`: 출발 시간대 (0~23시)
- `O_ADMI_NM`: 출발 행정동
- `PURPOSE`: 이동 목적
- `AGE_GRP`, `SEX_CD`: 인구 특성

---

## 결론
- 데이터는 결측치 및 중복이 존재하지 않고, 전반적인 변수 분포 또한 안정적인 형태를 보임  
- 집계 데이터 특성상 `CNT`의 비대칭 분포는 자연스러운 현상으로 판단됨  
- 따라서 추가적인 결측 처리나 이상치 제거 없이 **EDA 및 분석 단계로 바로 활용 가능**한 데이터로 판단됨  
- 향후 분석에서는 시간, 지역, 인구 특성에 따른 이동 패턴을 중심으로 인사이트 도출 예정  

---

## 한 줄 요약
> 결측 및 이상치 없이 안정적인 집계 데이터로, 즉시 이동 패턴 분석이 가능한 상태(얘기 해봐야 됨)

In [ ]:
# date변환 파일 저장
df9.to_parquet('../data/t9_2023_2025_all_date_final.parquet', index=False)

# t10 데이터

In [ ]:
# 함수적용해서 t10 결측/행 및 타입 확인
df10 = pd.read_parquet('../data/t10_all.parquet')
quick_check(df10)

In [ ]:
# 날짜컬럼 date형 변환
df10['ETL_YM'] = pd.to_datetime(df10['ETL_YM'].astype(str), format='%Y%m')

In [ ]:
df10.head()

In [ ]:
df10.info()

## 전처리 결과 요약
- 전체 데이터는 **1,971,599행, 12개 컬럼**으로 구성됨  
- 결측값 및 중복값이 존재하지 않아 데이터 품질이 양호한 상태임  
- `ETL_YM` 컬럼은 날짜 타입(`datetime`)으로 변환하여 시계열 분석이 가능하도록 정제함  
- `CNT` 변수는 평균(약 792.8) 대비 중앙값(약 218.3)이 낮은 **우측 치우침 분포**를 보이며,  
  최대값(약 30,376)은 시간대 및 지역 집계 특성을 고려할 때 **자연스러운 범위**로 판단됨  
- 범주형 변수(`DOW`, `O_MEGA_NM`, `O_CTY_NM`, `SEX_CD`)는 정상 범위 내에서 분포하며 이상값은 관측되지 않음  

---

## 데이터 정의
- 본 데이터는 **월 단위 기준으로 출발지의 시간, 지역, 인구 특성 및 이동 유형에 따른 이동량을 집계한 데이터**임  
- 특정 시간대 및 지역에서의 **이동 패턴 및 교통/이동 특성 분석**을 목적으로 활용 가능  

### 주요 변수
- `CNT`: 이동량 (집계된 유동 인구 규모)
- `O_TIME_CD`: 출발 시간대 (0~23시)
- `O_CTY_NM`: 출발 시군구 (성남시 3개 구)
- `TRANS_GB`: 이동 유형 (교통/이동 구분 변수)
- `AGE_GRP`, `SEX_CD`: 인구 특성
- `DOW`: 요일 정보

---

## 결론
- 데이터는 결측치 및 중복이 존재하지 않고, 변수 분포 또한 안정적인 형태를 보임  
- 집계 데이터 특성상 `CNT`의 비대칭 분포는 자연스러운 현상으로 판단됨  
- 날짜 컬럼이 정제되어 있어 시계열 분석이 가능한 상태이며,  
  추가적인 전처리 없이 **EDA 및 분석 단계로 바로 활용 가능**한 데이터로 판단됨  
- 향후 분석에서는 시간, 요일, 이동 유형에 따른 패턴 분석을 중심으로 인사이트 도출 예정  

---

## 한 줄 요약
> 결측 및 이상치 없이 안정적인 월 단위 이동 데이터로, 시계열 기반 이동 패턴 분석이 가능한 상태

In [ ]:
# date변환 파일 저장
df10.to_parquet('../data/t10_2023_2025_all_date_final.parquet', index=False)

# t11 데이터

In [ ]:
# 함수적용해서 t11 결측/행 및 타입 확인
df11 = pd.read_parquet('../data/t11_all.parquet')
quick_check(df11)

In [ ]:
# 날짜컬럼 date형 변환
df11['ETL_YMD'] = pd.to_datetime(df11['ETL_YMD'].astype(str), format='%Y%m%d')

In [ ]:
df11.head()

In [ ]:
df11.info()

In [ ]:
# 중복값 확인할려는데 안나옴...
cols = ['ETL_YMD',
    'O_TIME_CD',
    'O_ADMI_CD',
    'SEX_CD',
    'AGE_GRP',
    'TRANS_GB']

dup = df11[df11.duplicated(subset=cols, keep=False)]
dup.head(20)

In [ ]:
# 다시 중복값 확인
cols = ['ETL_YMD',
    'O_TIME_CD',
    'O_ADMI_CD',
    'SEX_CD',
    'AGE_GRP',
    'TRANS_GB']

dup = df11[df11.duplicated(subset=cols, keep=False)]
dup.sort_values(cols).head(20)

In [ ]:
dup.groupby(cols)['CNT'].nunique().value_counts()

동일한 행이 2번씩 반복되어 있음.....

## 전처리 결과 요약

- 원천 데이터는 최대 약 **71,798,900행, 12개 컬럼**으로 구성됨  (실 원본데이터 행 수: **71,457,282행, 12개컬럼**으로 구성됨)
- 일부 데이터에서 **중복 행 약 23,988,122건 존재** → 해결 완료 데이터 통합하면서 년도별컬럼이 2번씩 들어가 생긴 문제로 원본데이터에서 새로 다시 통합 후 확인결과 중복값 없음.
- 결측값은 모든 데이터셋에서 관측되지 않아 전반적인 데이터 완전성은 양호한 상태임  
- `CNT` 변수는 평균 대비 중앙값이 낮은 **우측 치우침 분포(비대칭)**를 보이며,  
  집계 데이터 특성상 자연스러운 분포로 판단됨  

---

## 데이터 구조 및 특징

### 1. 데이터 유형 구분

본 데이터는 크게 두 가지 유형으로 구성됨:

- **일 단위 데이터 (ETL_YMD)**
  - 행 수: 약 5천만 ~ 7천만 규모
  - 특징: 출발 행정동 기준 상세 이동 데이터
  - 분석: 시간대/지역/인구 기반 정밀 분석 가능

- **월 단위 데이터 (ETL_YM)**
  - 행 수: 약 120만 ~ 190만 규모
  - 특징: 요일(DOW) 및 이동 유형(TRANS_GB) 포함
  - 분석: 패턴 및 트렌드 분석에 적합

---

### 2. 지역 범위

- 모든 데이터는 **경기도 성남시 (3개 구)**로 제한됨
  - 분당구 / 수정구 / 중원구
- 행정동 기준 약 50개 단위로 구성됨

---

### 3. 주요 변수

- `CNT`: 이동량 (집계된 유동 인구)
- `O_TIME_CD`, `D_TIME_CD`: 시간대 (0~23시)
- `O_ADMI_NM`, `D_ADMI_NM`: 행정동
- `TRANS_GB`: 이동 유형
- `PURPOSE`: 이동 목적
- `AGE_GRP`, `SEX_CD`: 인구 특성
- `DOW`: 요일

---

## 데이터 품질 평가

### ✔️ 장점
- 결측값 없음 → 데이터 완전성 높음
- 범주형 변수 이상 없음 → 정합성 양호
- 지역 및 시간 정보 명확 → 분석 적합

### ❗ 주의사항
- 일부 데이터에서 **대규모 중복 존재 (약 2,400만 건)**  
  → 분석 전 반드시 제거 필요
- `CNT`는 실측값이 아닌 **집계/가중값**
  → 해석 시 “인구 수”가 아닌 “추정 이동량”으로 접근 필요

---

## 결론

- 데이터는 전반적으로 높은 품질을 보이나,  
  **중복 데이터 제거가 필수 전처리 단계**로 판단됨  
- 중복 제거 이후에는 추가적인 결측 처리 없이  
  **EDA 및 분석 단계로 바로 활용 가능한 수준의 데이터**임  
- 일 단위 데이터는 정밀 분석, 월 단위 데이터는 패턴 분석에 적합하므로  
  목적에 따라 분리 활용하는 것이 효과적임  

---

## 한 줄 요약

> 중복 제거 이후, 성남시 생활권 이동 패턴 분석에 바로 활용 가능한 고품질 집계 데이터

In [ ]:
# date변환 파일 저장(중복값은 파일통합하면서 생긴문제라 다시 통합 후 해결 완료)
df11.to_parquet('../data/t11_2023_2025_all_date_final.parquet', index=False)

# t12 데이터

In [ ]:
# 함수적용해서 t12 결측/행 및 타입 확인
df12 = pd.read_parquet('../data/t12_2023_2025_all.parquet')
quick_check(df12)

In [ ]:
# 이번건 str로 날짜컬럼이 되있어 똑같이 형변환(date)
df12['ETL_YM'] = pd.to_datetime(df12['ETL_YM'].astype(str), format='%Y%m')

In [ ]:
df12.head()

In [ ]:
df12.info()

In [ ]:
df12.isna().sum()

In [ ]:
# 결측 확인
df12[df12['D_MEGA_NM'].isna()]

In [ ]:
# 결측 확인
df12[df12['O_CTY_NM'].isna()].head()

In [ ]:
df12[df12['D_CTY_NM'].isna()].head()

## 전처리 결과 요약
- 전체 데이터는 **6,743,756행, 16개 컬럼**으로 구성됨  
- 중복값은 존재하지 않으며 데이터 정합성이 양호한 상태임  
- 일부 도착지(`D_CTY_NM`, `D_CENTER_X/Y`)에서 결측이 존재하나, 특정 지역 또는 매핑 문제로 판단됨(세종특별자치시로 생각중)  
- `CNT`는 평균 대비 중앙값이 낮은 우측 치우침 분포를 보이며, 집계 데이터 특성상 자연스러운 범위로 판단됨  

## 데이터 정의
- 본 데이터는 **출발지–도착지(OD) 기반 이동량을 집계한 데이터**로,  
  지역 간 이동 흐름 분석에 활용 가능함  

## 결론
- 일부 도착지 결측을 제외하면 데이터 품질은 전반적으로 안정적인 상태이며,  
- 중복 제거 없이 바로 분석 단계로 활용 가능한 데이터로 판단됨  
- 향후 분석에서는 지역 간 이동 흐름(유입/유출) 및 주요 이동 경로를 중심으로 인사이트 도출 예정
- 일부 세종시는 구특성이 없어 결과값이 nan으로 처리 된거 같음.

똑같이 세종시는 시군구 unknown으로 대체 그리고 나머지 99코드는 그대로 원본 유지

In [ ]:
# 1) 코드 기준으로 매핑 가능한 값 먼저 채우기
for col in ['D_MEGA_NM', 'D_CTY_NM', 'D_CENTER_X', 'D_CENTER_Y']:
    df12[col] = df12[col].fillna(
        df12.groupby('D_CTY_CD')[col].transform('first')
    )

# 2) 세종특별자치시의 시군구 None은 UNKNOWN 처리
df12.loc[
    (df12['O_MEGA_NM'] == '세종특별자치시') & (df12['O_CTY_NM'].isna()),
    'O_CTY_NM'
] = 'UNKNOWN'

df12.loc[
    (df12['D_MEGA_NM'] == '세종특별자치시') & (df12['D_CTY_NM'].isna()),
    'D_CTY_NM'
] = 'UNKNOWN'

In [ ]:
# 일단 결측 처리한 데이터 저장
df12.to_parquet('../data/t12_2023_2025_all_final_v2.parquet', index=False)

In [ ]:
df12.isna().sum()

In [ ]:
df12.info()

In [ ]:
# 남은 결측행 보기
df12[df12.isna().any(axis=1)]

## t12 전처리 최종 결과

| 항목 | 내용 |
|------|------|
| 데이터셋 | t12 |
| 기간 | 2023 ~ 2025 |
| 처리 방식 | pandas 기반 결측 보정 |
| 후보 파일 | t12_2023_2025_all_final.parquet |

| 구분 | 처리 내용 |
|------|----------|
| 도착지 결측 보정 | D_CTY_CD 기준으로 동일 코드의 정상값을 활용해 D_MEGA_NM, D_CTY_NM, D_CENTER_X, D_CENTER_Y 보정 |
| 세종특별자치시 처리 | 시군구 컬럼이 존재하지 않는 구조적 결측으로 판단하여 UNKNOWN 처리 |
| 출발지 시군구 | O_MEGA_NM이 세종특별자치시이고 O_CTY_NM이 None인 경우 UNKNOWN 처리 |
| 도착지 시군구 | D_MEGA_NM이 세종특별자치시이고 D_CTY_NM이 None인 경우 UNKNOWN 처리 |
| 미확인 코드 | 코드 기준으로도 매핑 불가능한 값은 원본 유지 |

| 최종 상태 | 내용 |
|-----------|------|
| 코드 기준 매핑 | 완료 |
| 구조적 결측 처리 | 완료 |
| 데이터 손실 | 없음 |
| 분석 가능 여부 | 가능 |

t12 데이터는 D_CTY_CD 기준 매핑과 세종특별자치시 구조적 결측 처리를 완료했으며, 분석 후보용 파일은 `t12_2023_2025_all_final.parquet`로 저장하였다.

# t14 데이터

In [ ]:
# 함수적용해서 t14 결측/행 및 타입 확인
df14 = pd.read_parquet('../data/t14_2023_2025_all.parquet')
quick_check(df14)

In [ ]:
# 일단 날짜부터 형변환
df14['ETL_YM'] = pd.to_datetime(df14['ETL_YM'].astype(str), format='%Y%m')

In [ ]:
df14.head()

In [ ]:
df14.info()

In [ ]:
df14.isna().sum()

In [ ]:
df14[df14['O_MEGA_NM'].isna()]

In [ ]:
df14[df14['O_CTY_NM'].isna()]

In [ ]:
df14[df14['D_CTY_NM'].isna()]

세종시는 unknown으로 대체

In [ ]:
# 광역시도명 99
df14[df14['O_MEGA_NM'].isna()]

In [ ]:
# 결측 확인
df14[df14['D_MEGA_NM'].isna()]

99가 뭐지?? ?? 이동은 했지만 위치를 알수 없는 데이터가 nan처리 되있는거 같다. 세종은 표시라도 하지 하.....

## 결측 데이터 해석
- O_MEGA_NM = 세종특별자치시
- O_CTY_NM = NaN
- O_CENTER_X/Y = 존재
- 즉 현재 통신데이터 중 출발 도착이 있는 데이터는 세종이 포함된 모든 행에서 시군구가 비어 있음

- 일부 데이터에서 `D_CTY_CD = 99`인 경우,
  `D_MEGA_NM`, `D_CTY_NM`, `D_CENTER_X/Y`가 모두 결측으로 나타남  

### 판단 근거

1. **결측 패턴의 일관성**
   - `D_CTY_CD = 99`인 경우에만 도착지 관련 컬럼이 동시에 결측 발생  
   - 일반적인 결측이라면 일부 컬럼만 누락되지만, 해당 경우는 **도착지 정보 전체가 일괄적으로 누락됨**

2. **특정 코드와 결측의 결합**
   - `D_CTY_CD`는 행정구역 코드 역할을 하는 변수이며, 정상 데이터는 특정 지역 코드를 가짐  
   - 그러나 값이 `99`일 때만 도착지 정보가 존재하지 않음  
   → 이는 **결측이 아닌 “특정 상태를 의미하는 코드”일 가능성 높음**

3. **다른 컬럼은 정상 유지**
   - 동일 행에서 출발지(`O_*`), 시간, 인구 특성(`SEX_CD`, `AGE_GRP`), 이동량(`CNT`)은 정상적으로 존재  
   → 데이터 자체가 깨진 것이 아니라 **도착지 정보만 의도적으로 비어 있는 구조**

4. **데이터 구조적 특성**
   - 이동 데이터에서 일부 기록은 위치 특정이 불가능한 경우 존재 (예: 위치 추정 실패, 범위 외 이동 등)  
   - 이러한 경우 일반적으로 별도 코드(예: 99)를 사용하여 “미상/기타”를 표현

---

### 결론

- `D_CTY_CD = 99`는 단순 결측이 아닌  
  **“도착지 미상(Unknown)”을 의미하는 코드로 판단됨**
- 따라서 해당 데이터는 제거하지 않고, 하나의 범주로 유지하는 것이 타당함

## 전처리 결과 요약

- 전체 데이터는 **8,472,627행, 16개 컬럼**으로 구성됨  
- 중복값은 존재하지 않으며 데이터 정합성이 양호한 상태임  
- CNT 변수는 평균(약 259.4) 대비 중앙값(약 14.9)이 낮은 우측 치우침 분포를 보임  
- 최대값(약 105,213)은 특정 시간대·지역에서의 집중 이동을 반영한 값으로, 집계 데이터 특성상 자연스러운 범위로 판단됨  

---

## 데이터 구조 해석

### 1. 시간 구조
- ETL_YM 기준으로 약 **36개월(3년)** 데이터 구성  
- DOW(요일) 변수 포함 → 주중 vs 주말 패턴 분석 가능**

성남을 중심으로 한 전국 이동 흐름 데이터

## 결론
- 데이터는 구조적으로 매우 많으며,  
  시간,공간,인구,이동 특성이 결합된 이동 데이터임  
- 전처리 단계에서 추가적인 정제 없이  
  EDA 및 인사이트 도출이 가능한 상태로 판단됨 

일단 지금까지 결측 및 중복값은 다 그대로 두고 팀원상의 후 해결 하겠습니다.

t14도 세종시에 시군구는 결측이 있어 unknown으로 대체 그리고 나머지 99코드는 원본 유지

In [ ]:
# 일단 혹시 모르는 행정동 코드가 none값이 되있을 수 있어 그것부터 확인
for col in ['O_MEGA_NM','O_CTY_NM','O_CENTER_X','O_CENTER_Y']:
    df14[col] = df14[col].fillna(
        df14.groupby('O_CTY_CD')[col].transform('first')
    )

for col in ['D_MEGA_NM','D_CTY_NM','D_CENTER_X','D_CENTER_Y']:
    df14[col] = df14[col].fillna(
        df14.groupby('D_CTY_CD')[col].transform('first')
    )

In [ ]:
# 세종시 unknown으로 처리
df14.loc[
    (df14['O_MEGA_NM'] == '세종특별자치시') & (df14['O_CTY_NM'].isna()),
    'O_CTY_NM'
] = 'UNKNOWN'

df14.loc[
    (df14['D_MEGA_NM'] == '세종특별자치시') & (df14['D_CTY_NM'].isna()),
    'D_CTY_NM'
] = 'UNKNOWN'

In [ ]:
# 남은 결측 확인
df14.isna().sum()

In [ ]:
# 남은 결측들 99인지 확인
df14[df14.isna().any(axis=1)][['O_CTY_CD','D_CTY_CD']].value_counts()

In [ ]:
# 최종 전처리 파일 저장
df14.to_parquet('../data/t14_2023_2025_all_final_v2.parquet', index=False)

# t14 최종 전처리 결과

| 항목 | 내용 |
|------|------|
| 데이터셋 | t14 |
| 기간 | 2023 ~ 2025 |
| 총 행 수 | 178,698,530 |
| 처리 방식 | pandas 기반 결측 보정 |
| 후보 파일 | t14_2023_2025_all_final.parquet |

| 구분 | 처리 내용 |
|------|----------|
| 출발지(O) | O_CTY_CD 기준 매핑 상태 유지 |
| 도착지(D) | D_CTY_CD 기준 매핑 상태 유지 |
| 세종특별자치시 | 시군구 없음 → UNKNOWN 처리 |
| 코드 매핑 | 별도 보정 없이 기존 값 유지 |
| 좌표 데이터 | 기존 값 유지 |
| 데이터 구조 | 원본 구조 유지 (파생컬럼 없음) |

| 구분 | 결측 수 | 처리 방식 |
|------|--------|-----------|
| 출발지(O) | 60건 | 99 코드 (미확인) 유지 |
| 도착지(D) | 40건 | 99 코드 (미확인) 유지 |
| 세종 시군구 | 구조적 결측 | UNKNOWN 처리 |
| 기타 결측 | 없음 | 정상 데이터 |

### 4. 결측 원인 분석
| 유형 | 설명 |
|------|------|
| 구조적 결측 | 세종특별자치시는 시군구 컬럼이 존재하지 않음 |
| 미확인 데이터 | O_CTY_CD 또는 D_CTY_CD = 99 |
| 매핑 누락 | 없음 (데이터 자체 특성) |

### 5. 최종 상태
| 항목 | 상태 |
|------|------|
| 행정코드 매핑 | 완료 |
| 결측 보정 | 완료 |
| 데이터 손실 | 없음 |
| 데이터 품질 | 안정 |
| 분석 가능 여부 | 즉시 가능 |

In [ ]:
# 저장파일이 원본이랑 행이 똑같은지 확인
check = pd.read_parquet('../data/t14_2023_2025_all_final_v2.parquet')
check.shape

In [ ]:
check.isna().sum()

# t16 데이터

In [ ]:
df16 = pd.read_parquet('../data/t16_2023_2025_all.parquet')
quick_check(df16)

In [ ]:
# 날짜 str 되있는거 date로 형변환
df16['ETL_YM'] = pd.to_datetime(df16['ETL_YM'].astype(str), format='%Y%m')

In [ ]:
df16.head()

In [ ]:
df16.info()

## 전처리 결과 요약 (D_TIME / DURATION 데이터)

- 전체 데이터는 **1,211,106행, 13개 컬럼**으로 구성됨  
- 결측값 및 중복값이 존재하지 않아 데이터 품질이 매우 양호한 상태임  
- `ETL_YM`은 datetime 타입으로 변환되어 시계열 분석이 가능함  

---

## 데이터 구조 해석

### 1. 시간 구조
- `D_TIME` (0~23시) 기준 → 시간대별 이동 분석 가능  
- `DOW` 포함 → 요일별 패턴 분석 가능  
- `ETL_YM` 기준 약 3년 데이터 → 장기 트렌드 분석 가능  

---

### 2. 공간 구조
- `D_CTY_NM` 기준 성남시 3개 구로 구성됨  
  - 분당구 / 수정구 / 중원구  

즉,

> 성남시 내부 도착 기준 이동 데이터

---

### 3. 이동 특성

- PURPOSE: 이동 목적 (0~5 범주)
- DURATION: 체류 시간

핵심 특징:

- 평균 체류시간: 약 **308분 (약 5시간)**
- 최대 체류시간: 약 **1,439분 (약 24시간 수준)**

해석:

> 단순 이동이 아니라 **체류 기반 행동 데이터**

---

### 4. 이동량 (CNT)

- 평균: 약 1,289  
- 중앙값: 약 230  

해석
일부 시간대/지역에 이동이 집중되는 우측 치우침 구조

---

### 5. 인구 특성

- SEX_CD, AGE_GRP 포함  
연령 및 성별 기반 행동 패턴 분석 가능  

---

## 데이터 특성 요약

- 시간 × 공간 × 인구 × 체류시간이 결합된 데이터  
- 단순 이동이 아닌  
어디서 얼마나 머무는지”까지 포함된 행동 데이터

---

## 분석 가능 방향

- 시간대별 체류 패턴 (야간 vs 주간)
- 요일별 체류 변화 (주말 vs 평일)
- 지역별 체류 집중도
- 연령/성별별 체류 시간 차이
- 목적별 체류 특성 분석

---

## 결론

- 결측 및 중복이 없는 고품질 데이터이며,  
- 체류 시간(`DURATION`)을 포함하고 있어  
  기존 이동 데이터보다 한 단계 높은 분석이 가능함  
- 추가 전처리 없이 바로 EDA 및 인사이트 도출이 가능한 상태로 판단됨  

---

## 한 줄 요약

> 성남시 내 이동과 체류 행동을 동시에 분석할 수 있는 고품질 체류 기반 데이터

In [ ]:
# date변환 파일 저장
df16.to_parquet('../data/t16_2023_2025_all_date_final.parquet', index=False)

# t20 데이터

In [ ]:
df20 = pd.read_csv('../data/t20_2023_2025_all.csv')
quick_check(df20)

In [ ]:
# 날짜 int 되있는거 date로 형변환
df20['ETL_YM'] = pd.to_datetime(df20['ETL_YM'].astype(str), format='%Y%m')

In [ ]:
df20.head()

In [ ]:
df20.info()

## 전처리 결과 요약 (DISTANCE / CARBON 데이터)

- 전체 데이터는 **6,048행, 11개 컬럼**으로 구성됨  
- 결측값 및 중복값이 존재하지 않아 데이터 품질이 매우 양호한 상태임  
- `ETL_YM`은 datetime 타입으로 변환되어 시계열 분석이 가능함  

---

## 데이터 구조 해석

### 1. 데이터 성격

- 본 데이터는 이동량(`CNT`)에 더해  
**이동 거리(`DISTANCE`)와 탄소 배출량(`CARBON_EMISSIONS`)이 결합된 요약 데이터**임

### 2. 거리 (DISTANCE)

- 평균: 약 **92.3**
- 중앙값: 약 **14.2**
- 최대: 약 **455.7**

해석

> 대부분의 이동은 단거리이며,  
> 일부 장거리 이동이 평균을 끌어올리는 구조

## 결론

- 결측 및 중복이 없는 고품질 요약 데이터이며,  
- 이동 데이터에 거리 및 탄소 정보가 결합되어  
  **환경 영향 기반 분석이 가능한 데이터**로 판단됨
## 한 줄 요약

> 이동 데이터를 기반으로 거리와 탄소까지 분석할 수 있는 환경 영향 확장 데이터

In [ ]:
# date변환 파일 저장
df20.to_csv('../data/t20_2023_2025_all_date_final.csv', index=False)

# t21 데이터

In [ ]:
df21 = pd.read_csv('../data/t21_2023_2025_all.csv')
quick_check(df21)

중복값이 78000개?? ??

In [ ]:
cols = ['CTY_CD', 'CTY_NM', 'TIME_CD', 'ETL_YMD']

dup_count = (df21.groupby(cols)
      .size()
      .reset_index(name='dup_cnt')
      .query('dup_cnt > 1'))

dup_count

내가 파일 합치면서 중복값이 2개씩 들어갔음

In [ ]:
df21 = df21.drop_duplicates()

In [ ]:
df21.duplicated().sum()

In [ ]:
df21.shape

같은값이 2번 들은게 맞았음.

## 중복 발생 원인 분석

- 해당 데이터는 원래 연령대별 집계 데이터를 포함하고 있었음  
- SQL 처리 과정에서 연령 관련 컬럼을 제거하면서  
  동일한 시간/지역/날짜 조합이 반복되어 중복 발생  

### 판단 근거

- 중복 행이 동일한 CTY_CD, TIME_CD, ETL_YMD 값을 가짐
- 연령 구분 컬럼 제거 후 발생한 구조적 중복

### 결론

- 해당 중복은 데이터 오류가 아닌  
  **연령 집계 제거로 인해 발생한 구조적 중복**
- 분석 목적에 따라:
  - 기준 테이블로 사용할 경우 → 중복 제거
  - 집계 데이터로 사용할 경우 → 원본에서 재집계 필요

## 중복 데이터 처리 결과

- 동일한 CTY_CD, TIME_CD, ETL_YMD 조합이 모두 2회씩 반복됨
- 이는 데이터 생성 또는 병합 과정에서 동일 데이터가 중복 적재된 것으로 판단됨

### 판단 근거
- 모든 중복 조합의 발생 횟수가 동일하게 2로 나타남
- 특정 일부가 아닌 전체 데이터에 동일 패턴 존재

### 처리 방법
- 완전 동일 행 반복이므로 `drop_duplicates()`를 통해 제거

### 결론
- 해당 중복은 데이터 오류이며,
  제거 후 정상적인 기준 테이블로 활용 가능

In [ ]:
# date변환 파일 저장
df21.to_csv('../data/t21_2023_2025_all_date_final.csv', index=False)

# t22 데이터

In [ ]:
df22 = pd.read_parquet('../data/t22_2023_2025_all.parquet')
quick_check(df22)

In [ ]:
key_cols = ['ADMI_CD', 'CTY_NM', 'ADMI_NM', 'TIME_CD', 'FORN_GB', 'ETL_YMD']

df22.groupby(key_cols).size().value_counts().sort_index()

## t22 데이터 검증 결과

- ADMI_CD, TIME_CD, FORN_GB, ETL_YMD 기준으로 중복 여부 확인 결과  
  모든 조합이 1회씩만 존재함

### 결론
- 데이터 중복 없음
- 추가적인 중복 처리 불필요
- 전처리 완료 상태로 바로 분석 가능

## 변수 선택 기준

본 분석은 젠트리피케이션 위험도 산정을 목적으로 하며,
지역 내 유동인구의 변화와 이동 패턴을 주요 지표로 설정하였다.

연령대 및 내/외국인 구분 변수는
인구 특성 분석에는 유용하지만,
본 연구의 핵심 지표인 "공간적 변화" 및 "유입/유출 패턴"과의 직접적인 연관성이 낮다고 판단

※ 향후 확장 분석에서는 특정 연령대 유입 변화나 외국인 비율 변화 등을 추가적으로 고려할 수 있음

t22 데이터 활용 방향

- 본 데이터는 행정동, 시간, 날짜, 내/외국인 기준으로  
  성별·연령대별 인구 수치를 포함한 원본 데이터임  

- 연령 및 내/외국인 컬럼을 단순 제거할 경우  
  데이터의 핵심 값이 함께 제거되어 분석이 불가능해짐  

### 처리 방향

- 연령별 컬럼은 직접 사용하지 않더라도 쓰게 된다면 컬럼을 뽑아서 쓰는걸로 하면 좋을 거 같음.

In [ ]:
# date변환 파일 저장
df22.to_parquet('../data/t22_2023_2025_all_date_final.parquet', index=False)

# t23 데이터

In [ ]:
df23 = pd.read_parquet('../data/t23_2023_2025_all.parquet')
quick_check(df23)

In [ ]:
# etl_ymd date로 형변환
df23['ETL_YMD'] = pd.to_datetime(df23['ETL_YMD'].astype(str), format='%Y%m%d')

In [ ]:
df23.head()

In [ ]:
df23.info()

## 전처리 결과 요약 (t23: 목적 기반 이동 데이터)

- 전체 데이터는 **4,693,040행, 9개 컬럼**으로 구성됨  
- 결측값 및 중복값이 존재하지 않아 데이터 품질이 양호한 상태임  

---

## 데이터 구조 해석

### 1. 데이터 성격

- 본 데이터는 날짜(`ETL_YMD`), 지역(`CTY_NM`), 시간(`TIME_CD`) 기준으로  
  이동 목적(`PURPOSE`)에 따른 이동량(`CNT`)을 집계한 데이터

즉,

> “언제, 어디서, 어떤 목적의 이동이 얼마나 발생했는지”를 나타내는 이동 패턴 데이터

---

### 2. 주요 변수

- `CNT`: 이동량 (핵심 변수)
- `PURPOSE`: 이동 목적 (0~6 범주)
- `TIME_CD`: 시간대 (0~23)
- `AGE_GRP`, `SEX_CD`: 인구 특성

---

### 3. 데이터 특징

- 평균 이동량 대비 중앙값이 낮아 **우측 치우침 분포**를 보임  
- 일부 시간대 및 지역에서 이동이 집중되는 구조  
- 성남시 3개 구(분당구, 수정구, 중원구)로 구성된 지역 데이터  

---

## 전처리 방향

- 젠트리피케이션 위험도 분석 목적에 따라   

- 핵심 분석 변수:
  - 이동량 (`CNT`)
  - 이동 목적 (`PURPOSE`)
  - 시간 (`TIME_CD`)
  - 지역 (`CTY_NM`)

---

## 데이터 활용 방향

- 시간대별 이동 집중도 분석
- 목적별 이동 패턴 변화 분석
- 지역별 유입/활동 증가 여부 분석
- 상권 활성화 및 체류 증가 신호 탐지

---

## 결론

- 결측 및 중복이 없는 고품질 데이터이며,  
- 이동 목적 기반 패턴 분석이 가능한 핵심 테이블로 판단됨  
- 추가 전처리 없이 바로 분석에 활용 가능한 상태  

---

## 한 줄 요약

> 성남시 내 이동 목적과 시간대별 활동 패턴을 분석할 수 있는 핵심 데이터

In [ ]:
# date변환 파일 저장
df23.to_parquet('../data/t23_2023_2025_all_date_final.parquet', index=False)

# t24 데이터

In [ ]:
df24 = pd.read_parquet('../data/t24_2023_2025_all.parquet')
quick_check(df24)

| 데이터   | 역할                           |
| ----- | ---------------------------- |
| t21   | 기준 테이블                       |
| t22   | 인구 분포                        |
| t23   | 목적별 이동                       |
| t13   | OD 이동                        |
| t24   | **전체 통합 원본 (granular data)** |


In [ ]:
# etl_ymd date로 형변환
df24['ETL_YMD'] = pd.to_datetime(df24['ETL_YMD'].astype(str), format='%Y%m%d')

In [ ]:
df24.head()

In [ ]:
df24.info()

## 전처리 결과 요약 (통합 원본 데이터)

- 전체 데이터는 43,306,619행, 10개 컬럼으로 구성됨  
- 결측값 및 중복값이 존재하지 않아 데이터 품질은 양호한 상태임  

---

## 데이터 구조 해석

- 본 데이터는 행정동(ADMI_NM), 시간(TIME_CD), 인구 특성(성별, 연령), 이동 목적(PURPOSE) 기준으로  
  이동량(CNT)을 집계한 데이터

“누가 / 언제 / 어디서 / 왜 / 얼마나 이동했는지” 전부 있음

## 전처리 방향

- 데이터 규모(약 4,300만 행)가 매우 크고,
  변수 조합이 세분화되어 있어 직접 분석에는 비효율적

## 1차 전처리 진행 상황 공유

현재 데이터 파일 수가 많고 구조가 복잡하여,
전체를 동시에 깊게 전처리하기보다는 **1차적으로 전체 구조 파악 및 품질 점검 위주로 진행**했습니다.

---

## 1. 결측치 및 이상치 점검

- 전체 파일에 대해 결측치 및 이상치 여부를 우선적으로 확인
- 일부 파일에서 확인된 결측치는 데이터 특성(예: 행정구조, 코드값 등)에 따른 정상 결측 가능성이 있어
  **추후 팀원들과 기준을 정리한 뒤 재정의 예정**

- 그 외 대부분 데이터에서는:
  - 결측 없음
  - 중복 없음
  - 음수/비정상 값 없음

따라서 현재 단계에서는 **추가적인 제거 없이 유지**

---

## 2. 이상치 처리 방향

- 현재 데이터는 이동량(CNT), 체류시간 등 **집계 데이터 중심**
- 일부 큰 값은 이상치가 아니라 **특정 시간/지역 집중 현상으로 판단**

따라서:
- 1차 전처리에서는 제거하지 않고 유지
- **EDA 단계에서 분포 확인 후 필요 시 추가 처리 예정**

---

## 3. 전처리 진행 방식

파일 수가 많아 전부를 깊게 파기보다는:

- 1차: 전체 구조 및 품질 점검
- 2차: 핵심 데이터셋 정리 (t13, t22, t23 등)
- 3차: EDA 기반 추가 전처리

이런 단계로 진행 중

---

## 4. 현재 상태

- 주요 데이터 구조 파악 완료
- 핵심 데이터셋 분리 및 정리 완료
- 전반적인 데이터 품질 문제 없음 확인

---

## 5. 공유 및 요청

- 결측값 해석 기준 (예: 99 코드, 특정 지역 결측 등)
  → 팀원들과 기준 정리 필요

- 이후 EDA 단계에서:
  - 이상치 처리 기준
  - 변수 활용 범위 (연령/성별 등)
  추가 논의 필요

---

## 한 줄 요약

> 전체 데이터 구조 및 품질 점검 완료,  
> 세부 전처리는 EDA 단계에서 보완 예정

In [ ]:
# date변환 파일 저장
df24.to_parquet('../data/t24_2023_2025_all_date_final.parquet', index=False)

# 결측 정리한 파일 정리 초안 나머지는 날짜데이터를 date로 변환한거라 date.final로 통일해서 저장
├── t12_2023_2025_all_final_v2.parquet
├── t13_2023_2025_all_final_v2.parquet
├── t14_2023_2025_all_final_v2.parquet
├── t25_2023_2025_all_final_v2.parquet 결측 처리 완료

## 기존 작업 기록: 기존 eda.ipynb

## 노트북 정리

- 젠트리피케이션 분석에 연결할 수 있는 인구/유동 관련 파생변수를 만든 EDA 노트북.
- t24로 유동인구, t13으로 외부유입, AGE_GRP 03~07로 경제활동 연령층 proxy를 만들고 월별 변화율을 확인함.
- 지역별 차이, 변수 간 상관, 월별 추세를 확인해서 모델링에 바로 쓸 변수와 중복될 수 있는 변수를 구분함.
- 임대료나 지가 같은 직접 목표변수가 없어서, 외부수요 증가와 거주 구조 변화 proxy 중심으로 정리함.
- 최종적으로 젠트리피케이션 예측에 쓸 후보 변수를 고르기 위한 기준을 정리한 파일로 보면 됨.


# 인구 지표 EDA

인구 관련 세 지표를 월-행정동 단위로 집계하고 변화율을 확인한다.

- 7. 유동인구 변화율
- 8. 외부유입 변화율
- 9. 경제활동 연령층 proxy 변화율

대용량 parquet는 DuckDB로 필요한 집계 결과만 읽는다.

  | 데이터 | 단위 | 기준 | 주요 의미 |
  |---|---|---|---|
  | t4 | 시군구 | 도착지 + 목적 | 어느 구로, 어떤 목적의 사람이 유입됐는지 |
  | t5 | 행정동 | 도착지 + 목적 | 어느 행정동으로, 어떤 목적의 사람이 유입됐는지 |
  | t6 | 시군구 | 도착지 + 교통수단 | 어느 구로, 어떤 교통수단으로 도착했는지 |
  | t7 | 행정동 | 도착지 + 교통수단 | 어느 행정동으로, 어떤 교통수단으로 도착했는지 |
  | t8 | 시군구 | 출발지 + 목적 | 어느 구에서, 어떤 목적의 이동이 시작됐는지 |
  | t9 | 행정동 | 출발지 + 목적 | 어느 행정동에서, 어떤 목적의 이동이 시작됐는지 |
  | t10 | 시군구 | 출발지 + 교통수단 | 어느 구에서, 어떤 교통수단 이동이 시작됐는지 |
  | t11 | 행정동 | 출발지 + 교통수단 | 어느 행정동에서, 어떤 교통수단 이동이 시작됐는지 |
  | t12 | 시군구 OD | 출발지→도착지 + 목적 | 구 간 이동을 목적별로 본 데이터 |
  | t13 | 행정동 OD | 출발지→도착지 + 목적 | 행정동 간 이동을 목적별로 본 데이터 |
  | t14 | 시군구 OD | 출발지→도착지 + 교통수단 | 구 간 이동을 교통수단별로 본 데이터 |
  | t16 | 시군구 | 도착지 + 목적 + 체류시간 | 어느 구에 도착해 얼마나 머무는지 |
  | t22 | 행정동 | 시간대 + 성별/연령 + 내외국인 | 행정동별 시간대 생활/체류 인구 성격 |
  | t23 | 시군구 | 시간대 + 목적 | 구 단위 시간대별 목적 유동인구 |
  | t24 | 행정동 | 시간대 + 목적 | 행정동 단위 시간대별 목적 유동인구 |
  | t25 | 시군구 OD | 출발지→도착지 + 시간대 + 목적 + 교통수단 | 구 간 이동을 시간·목적·교통수단까지 세분화 |
  | t26 | 행정동 | 도착지 + 시간대 + 목적 + 교통수단 + 체류시간 | 행정동별 도착/체류 특성 |
  | t27 | 행정동 | 출발지 + 시간대 + 목적 + 교통수단 + 체류시간 | 행정동별 출발/이탈 특성 |

## 0. 라이브러리와 데이터 경로 설정

필요한 라이브러리를 불러오고, 실행 위치와 관계없이 `data` 폴더 경로를 잡는다.

In [ ]:
import pandas as pd
import duckdb
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

con = duckdb.connect()

cwd = Path.cwd()
DATA_DIR = cwd / "data" if (cwd / "data").exists() else cwd.parent / "data"

t24_path = (DATA_DIR / "t24_2023_2025_all_date_final.parquet").as_posix()
t13_path = (DATA_DIR / "t13_2023_2025_all_final_v2.parquet").as_posix()

t24_path, t13_path

## 1. 사용할 파일 확인

인구 EDA에는 `t24`, `t13` 두 파일을 사용한다.

- `t24`: 행정동별 유동/생활인구 집계
- `t13`: 출발지-도착지 기준 외부유입 집계

파일 전체를 읽지 않고 메타데이터만 확인한다.

In [ ]:
for name, path in {"t24": t24_path, "t13": t13_path}.items():
    pf = pq.ParquetFile(path)
    print(f"[{name}] rows={pf.metadata.num_rows:,}, columns={len(pf.schema_arrow.names)}")
    print(pf.schema_arrow.names)
    print()

#### 파일 확인 해석

- `t24`는 유동인구/연령대 집계에, `t13`은 외부유입 집계에 사용한다.
- 두 파일 모두 행 수가 커서 전체 로딩보다 DuckDB로 필요한 컬럼만 집계하는 방식이 적절하다.


## 이번 EDA에서 만든 파생 컬럼 정리

`CNT`를 인구/이동량 규모로 보고 월 단위 합계와 전월 대비 변화율을 만들었다.

핵심은 **규모**와 **최근 변화 방향**을 같이 보는 것이다.

| 파생 컬럼 | 의미 | 생성 기준 | 해석 |
|---|---|---|---|
| `ym` | 기준 연월 | `ETL_YMD`를 `YYYY-MM`으로 변환 | 월 단위 결합 기준 |
| `floating_pop` | 유동인구 규모 | `t24`의 월-행정동별 `CNT` 합계 | 전체 사람 규모 |
| `floating_change_rate` | 유동인구 변화율 | `floating_pop.pct_change()` | 전월 대비 증감률 |
| `external_inflow` | 외부유입 규모 | 성남 밖 → 성남 안 이동의 `CNT` 합계 | 외부 방문 수요 |
| `external_inflow_change_rate` | 외부유입 변화율 | `external_inflow.pct_change()` | 외부 방문 수요 증감률 |
| `working_pop` | 경제활동 연령층 유동인구 proxy | `AGE_GRP` 03~07 코드의 `CNT` 합계 | 정확한 생산가능인구가 아니라 연령대 코드 기반 대체 지표 |
| `working_pop_change_rate` | 경제활동 연령층 proxy 변화율 | `working_pop.pct_change()` | 해당 대체 지표의 전월 대비 증감률 |

### 해석할 때 주의할 점

변화율은 지난달 값이 작으면 과하게 튈 수 있으므로 원 규모 컬럼과 같이 본다.

첫 달 변화율 `NaN`은 비교할 전월이 없기 때문에 정상이다.

## 2. 유동인구 변화율

`t24`의 `CNT`를 월-구-행정동 단위로 합산한다.

일별 변동을 줄이기 위해 월 단위로 보고, 전월 대비 변화율을 계산한다.

`유동인구 변화율 = (이번 달 유동인구 - 지난 달 유동인구) / 지난 달 유동인구`

In [ ]:
floating_monthly = con.execute(f"""
    SELECT
        strftime(ETL_YMD, '%Y-%m') AS ym,
        CTY_NM,
        ADMI_NM,
        SUM(CNT) AS floating_pop
    FROM read_parquet('{t24_path}')
    GROUP BY 1, 2, 3
    ORDER BY 1, 2, 3
""").df()

floating_monthly["floating_change_rate"] = (
    floating_monthly
    .sort_values("ym")
    .groupby(["CTY_NM", "ADMI_NM"])["floating_pop"]
    .pct_change()
)

floating_monthly.head()

#### 유동인구 집계표 해석

- 행정동별 월간 `CNT` 합계가 `floating_pop`이며, 첫 달은 비교할 전월이 없어 변화율이 `NaN`이다.
- 이후 행은 같은 행정동의 전월 대비 유동인구 증감률을 확인하는 기준 데이터다.


### 유동인구 추세 그래프

행정동별 값은 너무 많으므로 먼저 구 단위로 합쳐서 전체 흐름을 확인한다. 이 그래프는 성남시 안에서 어느 구의 유동인구 규모가 크고, 월별 방향성이 비슷한지 보는 용도다.

In [ ]:
floating_cty = (
    floating_monthly
    .groupby(["ym", "CTY_NM"], as_index=False)["floating_pop"]
    .sum()
)

ax = floating_cty.pivot(index="ym", columns="CTY_NM", values="floating_pop").plot(
    figsize=(12, 5), marker="o"
)
ax.set_title("월별 유동인구 추세")
ax.set_xlabel("월")
ax.set_ylabel("CNT 합계")
plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.show()

### 유동인구 추세 그래프 해석

- 분당구가 전 기간 유동인구 규모가 가장 크고, 수정구와 중원구는 상대적으로 낮다.
- 2025년 7월에 세 구 모두 일시적으로 크게 증가해 특수 이벤트나 집계 영향 확인이 필요하다.
- 전반적으로 분당구와 수정구는 증가 흐름, 중원구는 정체에 가깝다.


### 최근 월 변화율 상위 행정동

최근 월 기준으로 전월 대비 유동인구가 크게 증가한 행정동을 확인한다. 급증 지역은 이후 상권, 매출, 부동산 지표와 연결해서 볼 후보가 된다.

In [ ]:
latest_ym = floating_monthly["ym"].max()
floating_monthly.query("ym == @latest_ym")    .sort_values("floating_change_rate", ascending=False)    .head(10)

#### 최근 유동인구 증가 행정동 해석

- 2025년 12월 기준 전월 대비 유동인구 증가율이 큰 행정동 목록이다.
- 백현동, 이매2동, 야탑1동 등 분당구 행정동이 상위권이라 최근 유동인구 증가는 분당구 중심으로 보인다.


## 3. 외부유입 변화율

`t13`에서 성남 밖에서 성남 안으로 들어온 이동만 집계한다.

- 도착지 `D_CTY_NM`: 성남시 포함
- 출발지 `O_CTY_NM`: 성남시 미포함

`외부유입 변화율 = (이번 달 외부유입 - 지난 달 외부유입) / 지난 달 외부유입`

In [ ]:
external_inflow_monthly = con.execute(f"""
    SELECT
        strftime(ETL_YMD, '%Y-%m') AS ym,
        D_CTY_NM AS CTY_NM,
        D_ADMI_NM AS ADMI_NM,
        SUM(CNT) AS external_inflow
    FROM read_parquet('{t13_path}')
    WHERE D_CTY_NM LIKE '성남시%'
      AND O_CTY_NM NOT LIKE '성남시%'
    GROUP BY 1, 2, 3
    ORDER BY 1, 2, 3
""").df()

external_inflow_monthly["external_inflow_change_rate"] = (
    external_inflow_monthly
    .sort_values("ym")
    .groupby(["CTY_NM", "ADMI_NM"])["external_inflow"]
    .pct_change()
)

external_inflow_monthly.head()

#### 외부유입 집계표 해석

- 성남시 밖에서 성남시 안으로 들어온 이동량을 월-행정동 단위로 합산한 표다.
- `external_inflow_change_rate`는 외부 방문 수요가 전월 대비 얼마나 변했는지 보는 지표다.


### 외부유입 추세 그래프

구 단위 외부유입 추세를 보면 성남 내부 유동이 아니라 외부에서 들어오는 수요가 어느 지역으로 집중되는지 확인할 수 있다. 유동인구와 다른 움직임을 보이면 별도 지표로 쓸 가치가 커진다.

In [ ]:
external_inflow_cty = (
    external_inflow_monthly
    .groupby(["ym", "CTY_NM"], as_index=False)["external_inflow"]
    .sum()
)

ax = external_inflow_cty.pivot(index="ym", columns="CTY_NM", values="external_inflow").plot(
    figsize=(12, 5), marker="o"
)
ax.set_title("월별 외부유입 추세")
ax.set_xlabel("월")
ax.set_ylabel("CNT 합계")
plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.show()

### 외부유입 추세 그래프 해석

- 외부유입도 분당구가 가장 크며, 외부 방문 수요가 분당구에 집중되어 있다.
- 2025년부터 분당구와 수정구의 외부유입이 증가하는 흐름이 보인다.
- 중원구는 규모와 증가폭이 모두 상대적으로 작다.


### 최근 월 외부유입 변화율 상위 행정동

외부유입 변화율이 큰 지역은 외부 방문 수요가 새로 늘어난 곳일 수 있다. 단, 작은 지역은 분모가 작아 변화율이 크게 튈 수 있으므로 실제 유입 규모도 같이 봐야 한다.

In [ ]:
latest_ym = external_inflow_monthly["ym"].max()
external_inflow_monthly.query("ym == @latest_ym")    .sort_values("external_inflow_change_rate", ascending=False)    .head(10)

#### 최근 외부유입 증가 행정동 해석

- 2025년 12월 기준 외부유입 증가율 상위 행정동이다.
- 백현동, 수내1동, 이매2동이 상위권이므로 외부 방문 수요 증가 후보지로 볼 수 있다.


## 4. 경제활동 연령층 proxy 변화율

`AGE_GRP` 코드 `03`~`07`의 `CNT` 합계로 경제활동 연령층에 가까운 대체 지표를 계산한다. 단, 연령 코드가 10세 단위라면 이 값은 정확한 생산가능인구(15~64세)가 아니라 proxy로 해석해야 한다.

먼저 `AGE_GRP`별 규모를 확인한다.

In [ ]:
age_check = con.execute(f"""
    SELECT
        LPAD(CAST(AGE_GRP AS VARCHAR), 2, '0') AS AGE_GRP,
        SUM(CNT) AS cnt
    FROM read_parquet('{t24_path}')
    GROUP BY 1
    ORDER BY 1
""").df()

age_check

#### 연령대 분포표 해석

- `AGE_GRP`별 전체 `CNT` 규모를 확인해 어떤 연령대 코드가 큰 비중을 차지하는지 보는 표다.
- 이후 `03`~`07` 코드를 경제활동 연령층 proxy로 사용하므로, 이 코드들의 규모가 충분한지 확인하는 단계다.


### 경제활동 연령층 proxy 집계

`working_age_codes`에 포함된 `AGE_GRP` 코드만 남기고 `CNT`를 합산한다. 이후 유동인구와 동일하게 행정동별 전월 대비 변화율을 계산한다.

이 지표는 전체 유동인구 중 특정 연령대 코드(03~07)의 움직임을 따로 보기 위한 것이며, 정확한 생산가능인구로 단정하지 않는다.

In [ ]:
working_age_codes = ["03", "04", "05", "06", "07"]

working_pop_monthly = con.execute(f"""
    SELECT
        strftime(ETL_YMD, '%Y-%m') AS ym,
        CTY_NM,
        ADMI_NM,
        SUM(CNT) AS working_pop
    FROM read_parquet('{t24_path}')
    WHERE LPAD(CAST(AGE_GRP AS VARCHAR), 2, '0') IN ({','.join([repr(x) for x in working_age_codes])})
    GROUP BY 1, 2, 3
    ORDER BY 1, 2, 3
""").df()

working_pop_monthly["working_pop_change_rate"] = (
    working_pop_monthly
    .sort_values("ym")
    .groupby(["CTY_NM", "ADMI_NM"])["working_pop"]
    .pct_change()
)

working_pop_monthly.head()

#### 경제활동 연령층 proxy 집계표 해석

- `AGE_GRP` 03~07만 남겨 월-행정동별 `CNT`를 합산한 결과다.
- 정확한 생산가능인구가 아니라 특정 연령대 유동인구를 대체 지표로 본 값이다.


### 경제활동 연령층 proxy 추세 그래프

구 단위 경제활동 연령층 proxy 추세를 보면 총 유동인구 증가가 특정 연령대 코드의 유동인구 증가와 같이 움직이는지 확인할 수 있다.

In [ ]:
working_pop_cty = (
    working_pop_monthly
    .groupby(["ym", "CTY_NM"], as_index=False)["working_pop"]
    .sum()
)

ax = working_pop_cty.pivot(index="ym", columns="CTY_NM", values="working_pop").plot(
    figsize=(12, 5), marker="o"
)
ax.set_title("월별 경제활동 연령층 proxy 추세")
ax.set_xlabel("월")
ax.set_ylabel("CNT 합계")
plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.show()

### 경제활동 연령층 proxy 추세 그래프 해석

- 경제활동 연령층 proxy도 분당구가 가장 크고, 전체 유동인구 그래프와 거의 같은 패턴이다.
- 2025년 7월 피크가 동일하게 나타나 전체 유동 증가가 주요 연령층에서도 함께 발생한 것으로 보인다.
- 전체 유동인구와 중복성이 높으므로 모델링 시 둘 중 하나만 쓰거나 비율 지표로 바꾸는 것이 좋다.


## 5. 세 지표 결합

세 지표를 행정동-월 단위로 합친다. 이렇게 만든 `population_eda`는 이후 매출, 부동산, 거래량 같은 다른 데이터와 `ym`, `CTY_NM`, `ADMI_NM` 기준으로 결합하기 쉽다.

외부유입이 없는 경우는 실제로 0일 수도 있고 매칭 누락일 수도 있으므로, 결측 개수와 값을 함께 확인해야 한다.

In [ ]:
population_eda = (
    floating_monthly
    .merge(external_inflow_monthly, on=["ym", "CTY_NM", "ADMI_NM"], how="left")
    .merge(working_pop_monthly, on=["ym", "CTY_NM", "ADMI_NM"], how="left")
)

population_eda["external_inflow"] = population_eda["external_inflow"].fillna(0)

population_eda.head()

#### 결합 데이터프레임 해석

- 유동인구, 외부유입, 경제활동 연령층 proxy를 `ym`, `CTY_NM`, `ADMI_NM` 기준으로 합친 최종 EDA 테이블이다.
- 이후 매출, 거래량, 공시지가 등 다른 데이터와 행정동-월 단위로 결합하기 위한 기본 형태다.


## 6. 오류 점검

EDA 전에 기본적인 데이터 품질을 확인한다.

- 행 수가 예상대로 나오는지
- 핵심 지표에 결측치가 많은지
- 무한대 값이 생겼는지
- 변화율 계산 때문에 첫 달에만 결측이 생기는지

첫 달 변화율은 비교 대상인 전월이 없어서 `NaN`이 정상이다.

In [ ]:
check_cols = [
    "floating_pop",
    "floating_change_rate",
    "external_inflow",
    "external_inflow_change_rate",
    "working_pop",
    "working_pop_change_rate",
]

quality_check = pd.DataFrame({
    "rows": [len(population_eda)] * len(check_cols),
    "missing": population_eda[check_cols].isna().sum().values,
    "missing_rate": population_eda[check_cols].isna().mean().values,
    "inf_count": [np.isinf(population_eda[c].dropna()).sum() for c in check_cols],
}, index=check_cols)

quality_check

#### 데이터 품질표 해석

- 규모 지표에는 결측과 무한대 값이 없어 기본 품질은 양호하다.
- 변화율 지표의 결측 50개는 각 행정동의 첫 달에 전월 비교값이 없어서 생긴 정상적인 결측이다.


## 7. 지표가 유의미한지 통계적으로 점검

현재 단계에서는 지표 자체의 품질만 확인한다.

1. 변동성: 값이 거의 고정되어 있으면 EDA 지표로 의미가 작다.
2. 지역 차이: 구/행정동별 차이가 통계적으로 나타나면 공간 지표로 쓸 가치가 있다.
3. 중복성: 세 지표가 서로 거의 같은 값이면 굳이 모두 쓸 필요가 없다.

지역 차이는 permutation test로 간단히 확인한다.

In [ ]:
def coefficient_of_variation(s):
    s = pd.Series(s).dropna()
    return s.std() / s.mean() if s.mean() != 0 else np.nan

level_cols = ["floating_pop", "external_inflow", "working_pop"]
change_cols = ["floating_change_rate", "external_inflow_change_rate", "working_pop_change_rate"]

variation_summary = population_eda[level_cols + change_cols].agg(["count", "mean", "std", "min", "median", "max"]).T
variation_summary["cv"] = [coefficient_of_variation(population_eda[c]) for c in variation_summary.index]
variation_summary

#### 변동성 요약표 해석

- 규모 지표는 평균 대비 표준편차가 커서 행정동별 차이가 충분히 존재한다.
- 변화율 지표는 최대·최소 폭이 커서 단독 해석보다 원래 규모와 함께 보는 것이 안전하다.


### 지역 차이 검정

구별 평균 차이가 우연인지 permutation p-value로 확인한다.

- `p_value < 0.05`: 구별 차이가 통계적으로 뚜렷하다고 볼 수 있음
- `p_value >= 0.05`: 구별 차이가 약하거나, 현재 집계 단위에서는 뚜렷하지 않음

In [ ]:
def anova_f_stat(values, groups):
    df = pd.DataFrame({"value": values, "group": groups}).dropna()
    overall_mean = df["value"].mean()
    group_stats = df.groupby("group")["value"].agg(["count", "mean"])
    ss_between = (group_stats["count"] * (group_stats["mean"] - overall_mean) ** 2).sum()
    joined = df.join(group_stats["mean"], on="group", rsuffix="_group")
    ss_within = ((joined["value"] - joined["mean"]) ** 2).sum()
    df_between = len(group_stats) - 1
    df_within = len(df) - len(group_stats)
    return (ss_between / df_between) / (ss_within / df_within)


def permutation_pvalue(values, groups, n_perm=500, seed=42):
    rng = np.random.default_rng(seed)
    clean = pd.DataFrame({"value": values, "group": groups}).dropna()
    observed = anova_f_stat(clean["value"], clean["group"])
    perm_stats = []
    group_values = clean["group"].to_numpy().copy()
    for _ in range(n_perm):
        rng.shuffle(group_values)
        perm_stats.append(anova_f_stat(clean["value"], group_values))
    perm_stats = np.array(perm_stats)
    p_value = (np.sum(perm_stats >= observed) + 1) / (n_perm + 1)
    return observed, p_value

region_test_rows = []
for col in level_cols + change_cols:
    f_stat, p_value = permutation_pvalue(population_eda[col], population_eda["CTY_NM"])
    region_test_rows.append({"indicator": col, "f_stat": f_stat, "p_value": p_value})

region_difference_test = pd.DataFrame(region_test_rows).sort_values("p_value")
region_difference_test

#### 지역 차이 검정표 해석

- `floating_pop`, `external_inflow`, `working_pop`은 p-value가 0.05보다 작아 구별 규모 차이가 뚜렷하다.
- 반면 변화율 지표들은 p-value가 커서 구별 변화율 차이는 통계적으로 뚜렷하지 않다.


### 지표 중복성 확인

상관계수가 너무 높으면 두 지표가 거의 같은 정보를 담고 있을 수 있다. 특히 `floating_pop`과 `working_pop`은 같은 `t24`에서 나온 지표라 높게 나올 가능성이 있다.

- 0.9 이상: 거의 중복일 가능성이 큼
- 0.7~0.9: 강한 관련성
- 0.3~0.7: 중간 정도 관련성
- 0.3 미만: 서로 다른 정보를 담을 가능성이 큼

In [ ]:
correlation_matrix = population_eda[level_cols + change_cols].corr()
correlation_matrix

#### 상관계수표 해석

- `floating_pop`, `external_inflow`, `working_pop`은 서로 상관이 매우 높아 규모 측면에서는 중복 정보가 많다.
- 특히 `floating_pop`과 `working_pop`은 거의 같은 패턴이므로 모델링 시 동시 사용에 주의해야 한다.


### 월별 방향성 확인

각 지표가 시간에 따라 증가/감소하는 경향이 있는지 간단한 선형 추세 기울기를 계산한다. 이 값은 엄밀한 예측 모델이 아니라 EDA용 방향성 체크다.

기울기가 양수면 시간이 지날수록 증가, 음수면 감소 경향으로 해석한다.

In [ ]:
trend_df = population_eda.copy()
trend_df["month_index"] = pd.PeriodIndex(trend_df["ym"], freq="M").astype(int)

trend_rows = []
for cty, g in trend_df.groupby("CTY_NM"):
    x = g.groupby("ym")[level_cols].sum().reset_index()
    x["month_index"] = pd.PeriodIndex(x["ym"], freq="M").astype(int)
    for col in level_cols:
        slope = np.polyfit(x["month_index"], x[col], 1)[0]
        trend_rows.append({"CTY_NM": cty, "indicator": col, "monthly_slope": slope})

trend_summary = pd.DataFrame(trend_rows)
trend_summary.pivot(index="CTY_NM", columns="indicator", values="monthly_slope")

#### 월별 추세 기울기표 해석

- 기울기가 양수이면 시간이 갈수록 증가, 음수이면 감소 경향으로 해석한다.
- 분당구와 수정구는 주요 지표가 증가 흐름이고, 중원구는 유동인구와 경제활동 연령층 proxy가 약한 감소 흐름이다.


## 8. 해석 정리

아래 셀은 앞에서 계산한 통계 결과를 바탕으로 간단한 해석 문장을 자동으로 만든다.

주의할 점은, 여기서의 검증은 지표 자체의 변동성/지역성/중복성을 보는 것이다. 최종적으로 이 지표가 정말 유의미한지는 매출, 거래량, 공시지가 등 목표변수와 결합한 뒤 회귀분석이나 상관분석으로 다시 확인해야 한다.

In [ ]:
for _, row in region_difference_test.iterrows():
    verdict = "지역 차이가 뚜렷함" if row["p_value"] < 0.05 else "지역 차이가 뚜렷하지 않음"
    print(f"{row['indicator']}: p={row['p_value']:.4f}, {verdict}")

print("\n상관계수 절댓값 0.9 이상 조합:")
cols = correlation_matrix.columns
found = False
for i, c1 in enumerate(cols):
    for c2 in cols[i+1:]:
        corr = correlation_matrix.loc[c1, c2]
        if abs(corr) >= 0.9:
            found = True
            print(f"- {c1} vs {c2}: corr={corr:.3f}")
if not found:
    print("- 없음")

| 파생 컬럼 | 의미 | 생성 기준 | 해석 |
|---|---|---|---|
| `ym` | 기준 연월 | `ETL_YMD`를 `YYYY-MM`으로 변환 | 월 단위 결합 기준 |
| `floating_pop` | 유동인구 규모 | `t24`의 월-행정동별 `CNT` 합계 | 전체 사람 규모 |
| `floating_change_rate` | 유동인구 변화율 | `floating_pop.pct_change()` | 전월 대비 증감률 |
| `external_inflow` | 외부유입 규모 | 성남 밖 → 성남 안 이동의 `CNT` 합계 | 외부 방문 수요 |
| `external_inflow_change_rate` | 외부유입 변화율 | `external_inflow.pct_change()` | 외부 방문 수요 증감률 |
| `working_pop` | 경제활동 연령층 유동인구 proxy | `AGE_GRP` 03~07 코드의 `CNT` 합계 | 정확한 생산가능인구가 아니라 연령대 코드 기반 대체 지표 |
| `working_pop_change_rate` | 경제활동 연령층 proxy 변화율 | `working_pop.pct_change()` | 해당 대체 지표의 전월 대비 증감률 |

| 분석 지표          | 젠트리피케이션 해석                 |
| -------------- | -------------------------- |
| 유동인구 증가        | 상권 활성화 또는 외부 방문 수요 증가 가능성  |
| 외부유입 증가        | 외부 소비자·방문객 유입 증가 가능성       |
| 경제활동 연령층 증가    | 소비력 있는 활동 인구 증가 가능성        |
| 인구 대비 유동 규모 증가 | 거주 인구보다 외부 활동 수요가 큰 지역 가능성 |
| 전월 대비 급증 지역    | 단기적으로 변화가 커진 위험 후보 지역      |
| 3개월 이동평균 증가    | 일시적 이벤트가 아니라 지속 상승 가능성 확인  |


## 기존 작업 기록: 기존 my_eda.ipynb

## 노트북 정리

- 전체 t 데이터 파일을 한 번에 훑어보고, 어떤 테이블을 분석에 쓸지 정리한 EDA 노트북.
- 테이블별 컬럼, 기간, CNT 규모, 월별 추세, 지역별 차이를 확인해서 데이터 구조를 먼저 파악함.
- population_total.csv를 붙여서 행정동별 인구, 세대수, 세대당 인구와 유동인구를 같이 볼 수 있게 만듦.
- t24를 중심으로 인구 1,000명당 유동 규모, 상위 행정동, 변화율, 시간대/목적 코드 구성을 시각화함.
- 맨 아래에는 젠트리피케이션 예측에 쓸 변수별 해석과 예측 변수 후보 선택 기준을 정리함.


In [ ]:
import pandas as pd
import duckdb
import pyarrow.parquet as pq
import matplotlib

try:
    shell = get_ipython()
except NameError:
    shell = None

if shell is not None:
    matplotlib.use("module://matplotlib_inline.backend_inline", force=True)
    shell.run_line_magic("matplotlib", "inline")
else:
    matplotlib.use("Agg", force=True)

import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
from pathlib import Path

def find_data_dir():
    candidates = [
        Path("../data"),
        Path("data"),
        Path.cwd() / "data",
        Path.cwd().parent / "data",
    ]
    for candidate in candidates:
        if (candidate / "population_total.csv").exists():
            return candidate
    raise FileNotFoundError("data 폴더를 찾지 못했습니다. 노트북을 final_project 또는 Jiryun 폴더에서 실행해 주세요.")

DATA_DIR = find_data_dir()

available_fonts = {font.name for font in font_manager.fontManager.ttflist}
for font_name in ["Malgun Gothic", "AppleGothic", "NanumGothic", "DejaVu Sans"]:
    if font_name in available_fonts:
        plt.rcParams["font.family"] = font_name
        break
plt.rcParams["axes.unicode_minus"] = False

# 전체 t 데이터를 모두 스캔하는 EDA는 오래 걸릴 수 있으므로 기본값은 False입니다.
# 전체 t 통합 표가 필요할 때만 True로 바꾼 뒤 앞쪽 전체 t EDA 셀들을 실행하세요.
RUN_FULL_T_EDA = True

# 전체 t 데이터 EDA 실행 옵션
# RUN_FULL_T_EDA=True: 상대적으로 가벼운 전체 t 요약/월별/지역/목적/교통/성연령/시간/요일/OD 집계 실행
# RUN_FULL_T_HEAVY_EDA=True: 체류시간, 품질점검처럼 수억 행 파일을 다시 스캔하는 고비용 집계 실행
# RUN_MONTHLY_ADMIN_ALL_T=True: 전체 t 테이블을 월별 행정동 단위로 통합 집계
RUN_FULL_T_EDA = False
RUN_FULL_T_HEAVY_EDA = False
RUN_MONTHLY_ADMIN_ALL_T = False


  | 데이터 | 단위 | 기준 | 주요 의미 |
  |---|---|---|---|
  | t4 | 시군구 | 도착지 + 목적 | 어느 구로, 어떤 목적의 사람이 유입됐는지 |
  | t5 | 행정동 | 도착지 + 목적 | 어느 행정동으로, 어떤 목적의 사람이 유입됐는지 |
  | t6 | 시군구 | 도착지 + 교통수단 | 어느 구로, 어떤 교통수단으로 도착했는지 |
  | t7 | 행정동 | 도착지 + 교통수단 | 어느 행정동으로, 어떤 교통수단으로 도착했는지 |
  | t8 | 시군구 | 출발지 + 목적 | 어느 구에서, 어떤 목적의 이동이 시작됐는지 |
  | t9 | 행정동 | 출발지 + 목적 | 어느 행정동에서, 어떤 목적의 이동이 시작됐는지 |
  | t10 | 시군구 | 출발지 + 교통수단 | 어느 구에서, 어떤 교통수단 이동이 시작됐는지 |
  | t11 | 행정동 | 출발지 + 교통수단 | 어느 행정동에서, 어떤 교통수단 이동이 시작됐는지 |
  | t12 | 시군구 OD | 출발지→도착지 + 목적 | 구 간 이동을 목적별로 본 데이터 |
  | t13 | 행정동 OD | 출발지→도착지 + 목적 | 행정동 간 이동을 목적별로 본 데이터 |
  | t14 | 시군구 OD | 출발지→도착지 + 교통수단 | 구 간 이동을 교통수단별로 본 데이터 |
  | t16 | 시군구 | 도착지 + 목적 + 체류시간 | 어느 구에 도착해 얼마나 머무는지 |
  | t22 | 행정동 | 시간대 + 성별/연령 + 내외국인 | 행정동별 시간대 생활/체류 인구 성격 |
  | t23 | 시군구 | 시간대 + 목적 | 구 단위 시간대별 목적 유동인구 |
  | t24 | 행정동 | 시간대 + 목적 | 행정동 단위 시간대별 목적 유동인구 |
  | t25 | 시군구 OD | 출발지→도착지 + 시간대 + 목적 + 교통수단 | 구 간 이동을 시간·목적·교통수단까지 세분화 |
  | t26 | 행정동 | 도착지 + 시간대 + 목적 + 교통수단 + 체류시간 | 행정동별 도착/체류 특성 |
  | t27 | 행정동 | 출발지 + 시간대 + 목적 + 교통수단 + 체류시간 | 행정동별 출발/이탈 특성 |

# 전체 t 데이터 EDA

아래 섹션은 `data` 폴더에 있는 t 데이터 전체를 대상으로 합니다. 같은 테이블의 중복 버전(`_all`, `_final`, `_final_v2`, `_date_final`)을 모두 더하면 동일 데이터가 중복 집계되므로, 각 `t번호`별 최종본 1개씩만 사용합니다.

사용 대상은 `t4, t5, t6, t7, t8, t9, t10, t11, t12, t13, t14, t16, t22, t23, t24, t25, t26, t27`입니다. 원본 행을 일부 샘플링하지 않고 전체 parquet를 DuckDB로 스캔한 뒤, 그래프에 필요한 집계 결과만 pandas 데이터프레임으로 가져옵니다.

## 실행 옵션

`RUN_FULL_T_EDA=True`는 전체 t 데이터를 요약하는 비교적 가벼운 집계만 실행합니다. `RUN_FULL_T_HEAVY_EDA=True`나 `RUN_MONTHLY_ADMIN_ALL_T=True`는 수억 행 파일을 추가로 스캔하므로 시간이 오래 걸립니다. 팀 기준인 월별 행정동 분석은 아래의 빠른 `t24_monthly_admin_panel`만으로 바로 진행할 수 있습니다.


In [ ]:
from pathlib import Path
import pyarrow.parquet as parquet

T_DATA_FILES = {
    "t4": "t4_2023_2025_all_date_final.parquet",
    "t5": "t5_2023_2025_all_date_final.parquet",
    "t6": "t6_2023_2025_all_date_final.parquet",
    "t7": "t7_2023_2025_all_date_final.parquet",
    "t8": "t8_2023_2025_all_date_final.parquet",
    "t9": "t9_2023_2025_all_date_final.parquet",
    "t10": "t10_2023_2025_all_date_final.parquet",
    "t11": "t11_2023_2025_all_date_final.parquet",
    "t12": "t12_2023_2025_all_final_v2.parquet",
    "t13": "t13_2023_2025_all_final_v2.parquet",
    "t14": "t14_2023_2025_all_final_v2.parquet",
    "t16": "t16_2023_2025_all_date_final.parquet",
    "t22": "t22_2023_2025_all_date_final.parquet",
    "t23": "t23_2023_2025_all_date_final.parquet",
    "t24": "t24_2023_2025_all_date_final.parquet",
    "t25": "t25_2023_2025_all_final_v2.parquet",
    "t26": "t26_2023_2025_all_final.parquet",
    "t27": "t27_2023_2025_all_final.parquet",
}

T_DATA_DESC = {
    "t4": "도착 시군구-목적-성별-연령-월/요일",
    "t5": "도착 행정동-목적-성별-연령-일자",
    "t6": "도착 시군구-교통수단-성별-연령-월/요일",
    "t7": "도착 행정동-교통수단-성별-연령-일자",
    "t8": "출발 시군구-목적-성별-연령-월/요일",
    "t9": "출발 행정동-목적-성별-연령-일자",
    "t10": "출발 시군구-교통수단-성별-연령-월/요일",
    "t11": "출발 행정동-교통수단-성별-연령-일자",
    "t12": "시군구 OD-목적-성별-연령-월/요일",
    "t13": "행정동 OD-목적-성별-연령-일자",
    "t14": "시군구 OD-교통수단-성별-연령-월/요일",
    "t16": "도착 시군구-체류시간-목적-성별-연령-월/요일",
    "t22": "행정동 시간대별 성별·연령 생활/체류 인구",
    "t23": "시군구 시간대별 목적 유동인구",
    "t24": "행정동 시간대별 목적 유동인구",
    "t25": "시군구 OD-시간대-목적-교통수단-성별-연령",
    "t26": "도착 행정동-시간대-목적-교통수단-체류시간",
    "t27": "출발 행정동-시간대-목적-교통수단-체류시간",
}

def parquet_path(table_id):
    return (DATA_DIR / T_DATA_FILES[table_id]).as_posix()

metadata_rows = []
schema_rows = []

for table_id, file_name in T_DATA_FILES.items():
    file_path = DATA_DIR / file_name
    pf = parquet.ParquetFile(file_path)
    columns = pf.schema_arrow.names
    metadata_rows.append({
        "TABLE_ID": table_id,
        "DESCRIPTION": T_DATA_DESC[table_id],
        "FILE_NAME": file_name,
        "ROWS": pf.metadata.num_rows,
        "COLS": len(columns),
        "ROW_GROUPS": pf.metadata.num_row_groups,
        "SIZE_MB": round(file_path.stat().st_size / 1024 / 1024, 1),
    })
    for order, column in enumerate(columns, start=1):
        schema_rows.append({
            "TABLE_ID": table_id,
            "COLUMN_ORDER": order,
            "COLUMN": column,
        })

t_file_inventory = pd.DataFrame(metadata_rows)
t_schema_inventory = pd.DataFrame(schema_rows)
t_schema_map = t_schema_inventory.groupby("TABLE_ID")["COLUMN"].apply(list).to_dict()

t_file_inventory


**데이터프레임 설명 - `t_file_inventory`**

전체 EDA에 사용하는 최종 t 테이블 목록입니다. `ROWS`는 실제 parquet 메타데이터 기준 전체 행 수이고, `SIZE_MB`는 파일 크기입니다. 여기서 보이는 모든 테이블이 이후 EDA에 사용됩니다.

핵심 설명: 이 표는 이후 그래프를 해석하기 위한 기준 데이터입니다. 행 수, 기간, 지역 단위, 비중·변화율 컬럼을 먼저 확인하고 그래프를 읽으면 됩니다.


In [ ]:
t_schema_inventory


**데이터프레임 설명 - `t_schema_inventory`**

각 t 테이블이 가진 컬럼 목록입니다. `O_`로 시작하면 출발지(origin), `D_`로 시작하면 도착지(destination), 접두사가 없는 `CTY_NM`, `ADMI_NM`은 단일 지역 기준 컬럼으로 해석합니다.

핵심 설명: 이 표는 이후 그래프를 해석하기 위한 기준 데이터입니다. 행 수, 기간, 지역 단위, 비중·변화율 컬럼을 먼저 확인하고 그래프를 읽으면 됩니다.


In [ ]:
con = duckdb.connect()
con.execute("PRAGMA threads=4")

def scan_sql(table_id):
    return f"read_parquet('{parquet_path(table_id)}')"

def q(sql):
    return con.execute(sql).df()

def qname(column):
    return '"' + column.replace('"', '""') + '"'

def table_columns(table_id):
    return t_schema_map[table_id]

def cnt_expr(table_id):
    cols = table_columns(table_id)
    if "CNT" in cols:
        return "CNT"
    cnt_cols = [col for col in cols if col.endswith("_CNT")]
    return " + ".join([f"COALESCE({qname(col)}, 0)" for col in cnt_cols])

def parse_date_sql(column):
    col = qname(column)
    value = f"CAST({col} AS VARCHAR)"
    return (
        f"COALESCE("
        f"TRY_CAST({value} AS DATE), "
        f"CAST(try_strptime({value}, '%Y%m%d') AS DATE), "
        f"CAST(try_strptime({value}, '%Y%m') AS DATE), "
        f"CAST(try_strptime({value}, '%Y-%m-%d') AS DATE)"
        f")"
    )

def date_expr(table_id):
    cols = table_columns(table_id)
    if "ETL_YMD" in cols:
        return parse_date_sql("ETL_YMD")
    if "ETL_YM" in cols:
        return parse_date_sql("ETL_YM")
    return None

def union_query(parts):
    return "\nUNION ALL\n".join(parts)


**설명 - 전체 parquet 스캔 함수**

`scan_sql()`은 parquet 파일을 DuckDB가 직접 읽도록 하고, `cnt_expr()`은 대부분의 테이블의 `CNT`와 `t22`처럼 성별·연령별 count 컬럼이 넓게 펼쳐진 구조를 같은 총량 기준으로 맞춥니다. 이 방식은 원본 전체 행을 사용하지만, 집계 결과만 메모리에 올립니다.


In [ ]:
if RUN_FULL_T_EDA:
    overview_parts = []
    for table_id in T_DATA_FILES:
        date_col = date_expr(table_id)
        start_date_sql = f"MIN({date_col})" if date_col else "NULL"
        end_date_sql = f"MAX({date_col})" if date_col else "NULL"
        overview_parts.append(f'''
            SELECT
                '{table_id}' AS TABLE_ID,
                '{T_DATA_DESC[table_id]}' AS DESCRIPTION,
                COUNT(*) AS ROWS_SCANNED,
                SUM({cnt_expr(table_id)}) AS CNT_SUM,
                AVG({cnt_expr(table_id)}) AS CNT_MEAN,
                {start_date_sql} AS START_DATE,
                {end_date_sql} AS END_DATE
            FROM {scan_sql(table_id)}
        ''')

    all_t_overview = q(union_query(overview_parts)).merge(t_file_inventory, on=["TABLE_ID", "DESCRIPTION"], how="left")
    all_t_overview = all_t_overview[[
        "TABLE_ID", "DESCRIPTION", "FILE_NAME", "ROWS", "ROWS_SCANNED", "COLS",
        "SIZE_MB", "CNT_SUM", "CNT_MEAN", "START_DATE", "END_DATE"
    ]]
    all_t_overview
else:
    all_t_overview = pd.DataFrame()
    pd.DataFrame({"안내": ["전체 t 테이블 요약는 대용량 전체 t 스캔이라 기본 실행을 건너뜁니다.", "필요하면 첫 셀에서 RUN_FULL_T_EDA = True로 바꾸고 다시 실행하세요."]})


**데이터프레임 설명 - `all_t_overview`**

모든 t 최종본을 실제로 스캔해서 만든 전체 요약입니다. `ROWS`와 `ROWS_SCANNED`가 같으면 파일 메타데이터와 DuckDB 스캔 결과가 일치한다는 뜻입니다. `CNT_SUM`은 테이블별 총 유동/생활 인구 규모를 비교하는 기준입니다.

핵심 설명: 이 표는 이후 그래프를 해석하기 위한 기준 데이터입니다. 행 수, 기간, 지역 단위, 비중·변화율 컬럼을 먼저 확인하고 그래프를 읽으면 됩니다.


In [ ]:
if "all_t_overview" not in globals() or (hasattr(all_t_overview, "empty") and all_t_overview.empty):
    pd.DataFrame({"안내": ["전체 t 테이블별 CNT 총합 그래프는 대용량 셀을 건너뛰어서 실행하지 않습니다.", "월별 행정동 분석은 아래 빠른 t24 패널과 시각화 섹션을 사용하세요."]})
else:
    ax = all_t_overview.sort_values("CNT_SUM").plot(
        kind="barh",
        x="TABLE_ID",
        y="CNT_SUM",
        figsize=(10, 8),
        legend=False,
        title="전체 t 테이블별 CNT 총합",
    )
    ax.set_xlabel("CNT 총합")
    ax.set_ylabel("테이블")
    plt.tight_layout()
    plt.show()


**그래프 설명 - 전체 t 테이블별 CNT 총합**

각 t 테이블을 전체 행 기준으로 집계한 총량 비교입니다. 단, 테이블마다 분석 단위가 다르므로 절대 총량이 큰 테이블이 더 중요하다는 의미는 아닙니다. 어느 데이터가 전체 분석에서 규모가 큰지, 이후 세부 EDA 우선순위를 정하는 용도로 봅니다.


In [ ]:
if RUN_FULL_T_EDA:
    monthly_parts = []
    for table_id in T_DATA_FILES:
        d = date_expr(table_id)
        if d is None:
            continue
        monthly_parts.append(f'''
            SELECT
                '{table_id}' AS TABLE_ID,
                CAST(date_trunc('month', {d}) AS DATE) AS MONTH_START,
                SUM({cnt_expr(table_id)}) AS CNT_SUM
            FROM {scan_sql(table_id)}
            GROUP BY 1, 2
        ''')

    all_t_monthly = q(union_query(monthly_parts)).sort_values(["TABLE_ID", "MONTH_START"])
    all_t_monthly["YEAR_MONTH"] = pd.to_datetime(all_t_monthly["MONTH_START"]).dt.strftime("%Y-%m")
    all_t_monthly
else:
    all_t_monthly = pd.DataFrame()
    pd.DataFrame({"안내": ["전체 t 월별 추세는 대용량 전체 t 스캔이라 기본 실행을 건너뜁니다.", "필요하면 첫 셀에서 RUN_FULL_T_EDA = True로 바꾸고 다시 실행하세요."]})


**데이터프레임 설명 - `all_t_monthly`**

모든 t 테이블을 월 단위로 맞춘 추세 데이터입니다. 일자 데이터(`ETL_YMD`)와 월 데이터(`ETL_YM`)가 섞여 있으므로, 공통 비교를 위해 모두 월 시작일(`MONTH_START`)로 변환했습니다.

핵심 설명: 이 표는 이후 그래프를 해석하기 위한 기준 데이터입니다. 행 수, 기간, 지역 단위, 비중·변화율 컬럼을 먼저 확인하고 그래프를 읽으면 됩니다.


In [ ]:
if "all_t_monthly" not in globals() or (hasattr(all_t_monthly, "empty") and all_t_monthly.empty):
    pd.DataFrame({"안내": ["전체 t 월별 CNT 추세 그래프는 대용량 셀을 건너뛰어서 실행하지 않습니다.", "월별 행정동 분석은 아래 빠른 t24 패널과 시각화 섹션을 사용하세요."]})
else:
    monthly_pivot = all_t_monthly.pivot(index="YEAR_MONTH", columns="TABLE_ID", values="CNT_SUM")
    ax = monthly_pivot.plot(figsize=(16, 6), marker="o", title="전체 t 데이터 월별 CNT 추세")
    ax.set_xlabel("연-월")
    ax.set_ylabel("CNT 총합")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


**그래프 설명 - 전체 t 데이터 월별 CNT 추세**

각 t 테이블의 월별 총량 흐름입니다. 테이블별 스케일 차이가 커서 큰 테이블 위주로 보일 수 있습니다. 특정 월에 여러 테이블이 동시에 변하면 실제 유동 변화 가능성이 크고, 한 테이블만 튀면 해당 테이블의 정의나 수집 특성을 따로 확인해야 합니다.


In [ ]:
if "monthly_pivot" not in globals() or (hasattr(monthly_pivot, "empty") and monthly_pivot.empty):
    pd.DataFrame({"안내": ["월별 지수 추세 그래프는 대용량 셀을 건너뛰어서 실행하지 않습니다.", "월별 행정동 분석은 아래 빠른 t24 패널과 시각화 섹션을 사용하세요."]})
else:
    monthly_index = monthly_pivot.divide(monthly_pivot.iloc[0]).multiply(100)
    ax = monthly_index.plot(figsize=(16, 6), marker="o", title="전체 t 데이터 월별 지수 추세 - 첫 월=100")
    ax.set_xlabel("연-월")
    ax.set_ylabel("지수")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


**그래프 설명 - 월별 지수 추세**

각 테이블의 첫 월을 100으로 맞춘 변화율 관점 그래프입니다. 절대 규모가 다른 테이블을 같은 축에서 비교하기 위해 사용합니다. 선이 100보다 높아지면 첫 월 대비 증가, 낮아지면 감소입니다.


In [ ]:
if RUN_FULL_T_EDA:
    region_columns = ["CTY_NM", "ADMI_NM", "D_CTY_NM", "D_ADMI_NM", "O_CTY_NM", "O_ADMI_NM"]
    region_parts = []

    for table_id in T_DATA_FILES:
        cols = table_columns(table_id)
        for col in region_columns:
            if col in cols:
                if col.startswith("O_"):
                    role = "출발"
                elif col.startswith("D_"):
                    role = "도착"
                else:
                    role = "단일지역"
                level = "행정동" if "ADMI" in col else "시군구"
                region_parts.append(f'''
                    SELECT
                        '{table_id}' AS TABLE_ID,
                        '{role}' AS REGION_ROLE,
                        '{level}' AS REGION_LEVEL,
                        {qname(col)} AS REGION_NAME,
                        SUM({cnt_expr(table_id)}) AS CNT_SUM
                    FROM {scan_sql(table_id)}
                    WHERE {qname(col)} IS NOT NULL
                    GROUP BY 1, 2, 3, 4
                ''')

    all_t_region = q(union_query(region_parts)).sort_values("CNT_SUM", ascending=False)
    all_t_region.head(30)
else:
    all_t_region = pd.DataFrame()
    pd.DataFrame({"안내": ["전체 t 지역별 집계는 대용량 전체 t 스캔이라 기본 실행을 건너뜁니다.", "필요하면 첫 셀에서 RUN_FULL_T_EDA = True로 바꾸고 다시 실행하세요."]})


**데이터프레임 설명 - `all_t_region`**

모든 t 테이블에서 지역명 컬럼을 찾아 출발/도착/단일지역 기준으로 길게 합친 지역별 집계입니다. `REGION_ROLE`은 출발지인지 도착지인지, `REGION_LEVEL`은 시군구인지 행정동인지를 나타냅니다.

핵심 설명: 이 표는 이후 그래프를 해석하기 위한 기준 데이터입니다. 행 수, 기간, 지역 단위, 비중·변화율 컬럼을 먼저 확인하고 그래프를 읽으면 됩니다.


In [ ]:
if "all_t_region" not in globals() or (hasattr(all_t_region, "empty") and all_t_region.empty):
    pd.DataFrame({"안내": ["전체 지역 TOP 30 그래프는 대용량 셀을 건너뛰어서 실행하지 않습니다.", "월별 행정동 분석은 아래 빠른 t24 패널과 시각화 섹션을 사용하세요."]})
else:
    region_top30 = all_t_region.head(30).copy()
    region_top30["LABEL"] = region_top30["TABLE_ID"] + " | " + region_top30["REGION_ROLE"] + " | " + region_top30["REGION_NAME"]

    ax = region_top30.sort_values("CNT_SUM").plot(
        kind="barh",
        x="LABEL",
        y="CNT_SUM",
        figsize=(12, 10),
        legend=False,
        title="전체 t 데이터 지역별 CNT TOP 30",
    )
    ax.set_xlabel("CNT 총합")
    ax.set_ylabel("테이블 | 지역역할 | 지역명")
    plt.tight_layout()
    plt.show()


**그래프 설명 - 전체 지역 TOP 30**

전체 t 데이터에서 CNT 총합이 큰 지역 조합을 보여줍니다. 행정동 단위 테이블은 행 수와 세분화 정도가 다르므로, 같은 지역이라도 `TABLE_ID`와 `REGION_ROLE`을 함께 봐야 합니다.


In [ ]:
if RUN_FULL_T_EDA:
    purpose_parts = []
    for table_id in T_DATA_FILES:
        if "PURPOSE" not in table_columns(table_id):
            continue
        purpose_parts.append(f'''
            SELECT
                '{table_id}' AS TABLE_ID,
                PURPOSE,
                SUM({cnt_expr(table_id)}) AS CNT_SUM
            FROM {scan_sql(table_id)}
            GROUP BY 1, 2
        ''')

    all_t_purpose = q(union_query(purpose_parts)).sort_values(["TABLE_ID", "CNT_SUM"], ascending=[True, False])
    all_t_purpose["CNT_SHARE_IN_TABLE"] = all_t_purpose["CNT_SUM"] / all_t_purpose.groupby("TABLE_ID")["CNT_SUM"].transform("sum")
    all_t_purpose
else:
    all_t_purpose = pd.DataFrame()
    pd.DataFrame({"안내": ["전체 t 목적별 집계는 대용량 전체 t 스캔이라 기본 실행을 건너뜁니다.", "필요하면 첫 셀에서 RUN_FULL_T_EDA = True로 바꾸고 다시 실행하세요."]})


**데이터프레임 설명 - `all_t_purpose`**

목적 코드(`PURPOSE`)가 있는 모든 t 테이블의 목적별 총량과 테이블 내부 비중입니다. `CNT_SHARE_IN_TABLE`을 보면 테이블 규모 차이를 제거하고 목적 구성이 어떻게 다른지 비교할 수 있습니다.

핵심 설명: 이 표는 이후 그래프를 해석하기 위한 기준 데이터입니다. 행 수, 기간, 지역 단위, 비중·변화율 컬럼을 먼저 확인하고 그래프를 읽으면 됩니다.


In [ ]:
if "all_t_purpose" not in globals() or (hasattr(all_t_purpose, "empty") and all_t_purpose.empty):
    pd.DataFrame({"안내": ["목적 코드별 비중 그래프는 대용량 셀을 건너뛰어서 실행하지 않습니다.", "월별 행정동 분석은 아래 빠른 t24 패널과 시각화 섹션을 사용하세요."]})
else:
    purpose_pivot = all_t_purpose.pivot(index="PURPOSE", columns="TABLE_ID", values="CNT_SHARE_IN_TABLE").fillna(0)
    ax = purpose_pivot.plot(kind="bar", figsize=(14, 5), title="전체 t 데이터 목적 코드별 테이블 내부 비중")
    ax.set_xlabel("목적 코드")
    ax.set_ylabel("테이블 내부 비중")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()


**그래프 설명 - 목적 코드별 비중**

목적 코드가 있는 테이블들을 대상으로, 각 테이블 안에서 목적 코드가 차지하는 비중을 비교합니다. 실제 목적명 매핑표가 있다면 `PURPOSE` 코드를 목적명으로 바꿔 해석하면 됩니다.


In [ ]:
if RUN_FULL_T_EDA:
    trans_parts = []
    for table_id in T_DATA_FILES:
        if "TRANS_GB" not in table_columns(table_id):
            continue
        trans_parts.append(f'''
            SELECT
                '{table_id}' AS TABLE_ID,
                TRANS_GB,
                SUM({cnt_expr(table_id)}) AS CNT_SUM
            FROM {scan_sql(table_id)}
            GROUP BY 1, 2
        ''')

    all_t_transport = q(union_query(trans_parts)).sort_values(["TABLE_ID", "CNT_SUM"], ascending=[True, False])
    all_t_transport["CNT_SHARE_IN_TABLE"] = all_t_transport["CNT_SUM"] / all_t_transport.groupby("TABLE_ID")["CNT_SUM"].transform("sum")
    all_t_transport
else:
    all_t_transport = pd.DataFrame()
    pd.DataFrame({"안내": ["전체 t 교통수단 집계는 대용량 전체 t 스캔이라 기본 실행을 건너뜁니다.", "필요하면 첫 셀에서 RUN_FULL_T_EDA = True로 바꾸고 다시 실행하세요."]})


**데이터프레임 설명 - `all_t_transport`**

교통수단 코드(`TRANS_GB`)가 있는 모든 t 테이블의 교통수단별 집계입니다. 목적 EDA와 마찬가지로 테이블 내부 비중을 같이 보면서 교통수단 구성이 다른 테이블을 찾습니다.

핵심 설명: 이 표는 이후 그래프를 해석하기 위한 기준 데이터입니다. 행 수, 기간, 지역 단위, 비중·변화율 컬럼을 먼저 확인하고 그래프를 읽으면 됩니다.


In [ ]:
if "all_t_transport" not in globals() or (hasattr(all_t_transport, "empty") and all_t_transport.empty):
    pd.DataFrame({"안내": ["교통수단 코드별 비중 그래프는 대용량 셀을 건너뛰어서 실행하지 않습니다.", "월별 행정동 분석은 아래 빠른 t24 패널과 시각화 섹션을 사용하세요."]})
else:
    transport_pivot = all_t_transport.pivot(index="TRANS_GB", columns="TABLE_ID", values="CNT_SHARE_IN_TABLE").fillna(0)
    ax = transport_pivot.plot(kind="bar", figsize=(14, 5), title="전체 t 데이터 교통수단 코드별 테이블 내부 비중")
    ax.set_xlabel("교통수단 코드")
    ax.set_ylabel("테이블 내부 비중")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()


**그래프 설명 - 교통수단 코드별 비중**

교통수단 코드별 비중 차이를 테이블별로 비교합니다. 특정 교통수단이 특정 테이블에서만 높다면 해당 테이블이 출발/도착/OD 중 어떤 관점인지 함께 해석해야 합니다.


In [ ]:
if RUN_FULL_T_EDA:
    sex_age_parts = []
    for table_id in T_DATA_FILES:
        cols = table_columns(table_id)
        if {"SEX_CD", "AGE_GRP"}.issubset(cols):
            sex_age_parts.append(f'''
                SELECT
                    '{table_id}' AS TABLE_ID,
                    SEX_CD,
                    AGE_GRP,
                    SUM({cnt_expr(table_id)}) AS CNT_SUM
                FROM {scan_sql(table_id)}
                GROUP BY 1, 2, 3
            ''')
        elif table_id == "t22":
            for col in cols:
                if col.startswith(("M_", "F_")) and col.endswith("_CNT"):
                    sex_cd = col.split("_")[0]
                    age_grp = col.split("_")[1]
                    sex_age_parts.append(f'''
                        SELECT
                            't22' AS TABLE_ID,
                            '{sex_cd}' AS SEX_CD,
                            '{age_grp}' AS AGE_GRP,
                            SUM({qname(col)}) AS CNT_SUM
                        FROM {scan_sql(table_id)}
                    ''')

    all_t_sex_age = q(union_query(sex_age_parts)).sort_values(["TABLE_ID", "AGE_GRP", "SEX_CD"])
    all_t_sex_age["CNT_SHARE_IN_TABLE"] = all_t_sex_age["CNT_SUM"] / all_t_sex_age.groupby("TABLE_ID")["CNT_SUM"].transform("sum")
    all_t_sex_age
else:
    all_t_sex_age = pd.DataFrame()
    pd.DataFrame({"안내": ["전체 t 성별·연령 집계는 대용량 전체 t 스캔이라 기본 실행을 건너뜁니다.", "필요하면 첫 셀에서 RUN_FULL_T_EDA = True로 바꾸고 다시 실행하세요."]})


**데이터프레임 설명 - `all_t_sex_age`**

성별·연령대 정보를 가진 모든 t 테이블을 같은 구조로 맞춘 집계입니다. `t22`는 `M_10_CNT`, `F_10_CNT`처럼 컬럼이 넓게 펼쳐져 있어서 성별·연령 컬럼으로 변환해 합쳤습니다.

핵심 설명: 이 표는 이후 그래프를 해석하기 위한 기준 데이터입니다. 행 수, 기간, 지역 단위, 비중·변화율 컬럼을 먼저 확인하고 그래프를 읽으면 됩니다.


In [ ]:
if "all_t_sex_age" not in globals() or (hasattr(all_t_sex_age, "empty") and all_t_sex_age.empty):
    pd.DataFrame({"안내": ["전체 성별·연령대 분포 그래프는 대용량 셀을 건너뛰어서 실행하지 않습니다.", "월별 행정동 분석은 아래 빠른 t24 패널과 시각화 섹션을 사용하세요."]})
else:
    sex_age_total = (
        all_t_sex_age.groupby(["SEX_CD", "AGE_GRP"], as_index=False)["CNT_SUM"]
        .sum()
        .sort_values(["AGE_GRP", "SEX_CD"])
    )
    sex_age_total_pivot = sex_age_total.pivot(index="AGE_GRP", columns="SEX_CD", values="CNT_SUM")

    ax = sex_age_total_pivot.plot(kind="bar", figsize=(12, 5), title="전체 t 데이터 성별·연령대별 CNT 총합")
    ax.set_xlabel("연령대 코드")
    ax.set_ylabel("CNT 총합")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()


**그래프 설명 - 전체 성별·연령대 분포**

모든 t 테이블의 성별·연령대 집계를 합산한 그래프입니다. 테이블 간 중복 관점이 포함되어 있어 실제 순인구로 해석하면 안 되고, 전체 t 데이터에서 어떤 성별·연령대 조합의 관측 규모가 큰지 보는 용도입니다.


In [ ]:
if RUN_FULL_T_EDA:
    time_columns = ["TIME_CD", "D_TIME_CD", "O_TIME_CD", "D_TIME"]
    time_parts = []

    for table_id in T_DATA_FILES:
        cols = table_columns(table_id)
        for col in time_columns:
            if col not in cols:
                continue
            if col.startswith("O_"):
                role = "출발시간"
            elif col.startswith("D_"):
                role = "도착시간"
            else:
                role = "시간"
            time_parts.append(f'''
                SELECT
                    '{table_id}' AS TABLE_ID,
                    '{role}' AS TIME_ROLE,
                    CAST({qname(col)} AS VARCHAR) AS TIME_CD,
                    SUM({cnt_expr(table_id)}) AS CNT_SUM
                FROM {scan_sql(table_id)}
                WHERE {qname(col)} IS NOT NULL
                GROUP BY 1, 2, 3
            ''')

    all_t_time = q(union_query(time_parts)).sort_values(["TABLE_ID", "TIME_ROLE", "TIME_CD"])
    all_t_time
else:
    all_t_time = pd.DataFrame()
    pd.DataFrame({"안내": ["전체 t 시간대 집계는 대용량 전체 t 스캔이라 기본 실행을 건너뜁니다.", "필요하면 첫 셀에서 RUN_FULL_T_EDA = True로 바꾸고 다시 실행하세요."]})


**데이터프레임 설명 - `all_t_time`**

시간대 컬럼을 가진 모든 t 테이블을 출발시간·도착시간·단일 시간 기준으로 합친 집계입니다. `TIME_CD`, `D_TIME_CD`, `O_TIME_CD`, `D_TIME`처럼 이름이 다른 컬럼을 `TIME_ROLE`로 구분했습니다.

핵심 설명: 이 표는 이후 그래프를 해석하기 위한 기준 데이터입니다. 행 수, 기간, 지역 단위, 비중·변화율 컬럼을 먼저 확인하고 그래프를 읽으면 됩니다.


In [ ]:
if "all_t_time" not in globals() or (hasattr(all_t_time, "empty") and all_t_time.empty):
    pd.DataFrame({"안내": ["전체 시간대 패턴 그래프는 대용량 셀을 건너뛰어서 실행하지 않습니다.", "월별 행정동 분석은 아래 빠른 t24 패널과 시각화 섹션을 사용하세요."]})
else:
    time_total = (
        all_t_time.groupby(["TIME_ROLE", "TIME_CD"], as_index=False)["CNT_SUM"]
        .sum()
        .sort_values(["TIME_ROLE", "TIME_CD"])
    )

    for role, role_df in time_total.groupby("TIME_ROLE"):
        ax = role_df.plot(kind="bar", x="TIME_CD", y="CNT_SUM", figsize=(12, 4), legend=False, title=f"전체 t 데이터 {role}별 CNT")
        ax.set_xlabel(role)
        ax.set_ylabel("CNT 총합")
        plt.tight_layout()
        plt.show()


**그래프 설명 - 전체 시간대 패턴**

전체 t 데이터를 시간 역할별로 나눠 본 그래프입니다. 출발시간과 도착시간은 의미가 다르기 때문에 한 그래프에 섞지 않고 각각 따로 봅니다. 피크 시간대가 출발 기준과 도착 기준에서 다르게 나타나는지 확인합니다.


In [ ]:
if RUN_FULL_T_EDA:
    dow_parts = []
    for table_id in T_DATA_FILES:
        cols = table_columns(table_id)
        d = date_expr(table_id)
        if "DOW" in cols:
            dow_sql = "CAST(DOW AS VARCHAR)"
        elif d is not None:
            dow_sql = f"strftime({d}, '%w')"
        else:
            continue
        dow_parts.append(f'''
            SELECT
                '{table_id}' AS TABLE_ID,
                {dow_sql} AS DOW,
                SUM({cnt_expr(table_id)}) AS CNT_SUM
            FROM {scan_sql(table_id)}
            GROUP BY 1, 2
        ''')

    all_t_dow = q(union_query(dow_parts)).sort_values(["TABLE_ID", "DOW"])
    all_t_dow["CNT_SHARE_IN_TABLE"] = all_t_dow["CNT_SUM"] / all_t_dow.groupby("TABLE_ID")["CNT_SUM"].transform("sum")
    all_t_dow
else:
    all_t_dow = pd.DataFrame()
    pd.DataFrame({"안내": ["전체 t 요일 집계는 대용량 전체 t 스캔이라 기본 실행을 건너뜁니다.", "필요하면 첫 셀에서 RUN_FULL_T_EDA = True로 바꾸고 다시 실행하세요."]})


**데이터프레임 설명 - `all_t_dow`**

요일 정보가 있는 테이블은 `DOW`를 사용하고, 없는 테이블은 `ETL_YMD`에서 요일을 계산해 만든 전체 요일 집계입니다. `CNT_SHARE_IN_TABLE`은 각 테이블 내부에서 요일별 비중을 뜻합니다.

핵심 설명: 이 표는 이후 그래프를 해석하기 위한 기준 데이터입니다. 행 수, 기간, 지역 단위, 비중·변화율 컬럼을 먼저 확인하고 그래프를 읽으면 됩니다.


In [ ]:
if "all_t_dow" not in globals() or (hasattr(all_t_dow, "empty") and all_t_dow.empty):
    pd.DataFrame({"안내": ["요일별 비중 그래프는 대용량 셀을 건너뛰어서 실행하지 않습니다.", "월별 행정동 분석은 아래 빠른 t24 패널과 시각화 섹션을 사용하세요."]})
else:
    dow_pivot = all_t_dow.pivot(index="DOW", columns="TABLE_ID", values="CNT_SHARE_IN_TABLE").fillna(0)
    ax = dow_pivot.plot(kind="bar", figsize=(14, 5), title="전체 t 데이터 요일별 테이블 내부 비중")
    ax.set_xlabel("요일 코드")
    ax.set_ylabel("테이블 내부 비중")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()


**그래프 설명 - 요일별 비중**

테이블별 요일 구성이 평일 중심인지 주말 중심인지 비교합니다. 요일 코드의 기준은 데이터 정의서와 맞춰 확인해야 하며, `ETL_YMD`에서 계산한 경우 DuckDB 기준 `0=일요일`입니다.


In [ ]:
if RUN_FULL_T_EDA:
    od_parts = []
    for table_id in T_DATA_FILES:
        cols = table_columns(table_id)
        if {"O_CTY_NM", "D_CTY_NM"}.issubset(cols):
            od_parts.append(f'''
                SELECT
                    '{table_id}' AS TABLE_ID,
                    O_CTY_NM,
                    D_CTY_NM,
                    SUM({cnt_expr(table_id)}) AS CNT_SUM
                FROM {scan_sql(table_id)}
                WHERE O_CTY_NM IS NOT NULL AND D_CTY_NM IS NOT NULL
                GROUP BY 1, 2, 3
            ''')

    all_t_od_city = q(union_query(od_parts)).sort_values("CNT_SUM", ascending=False)
    all_t_od_city.head(30)
else:
    all_t_od_city = pd.DataFrame()
    pd.DataFrame({"안내": ["전체 t OD 집계는 대용량 전체 t 스캔이라 기본 실행을 건너뜁니다.", "필요하면 첫 셀에서 RUN_FULL_T_EDA = True로 바꾸고 다시 실행하세요."]})


**데이터프레임 설명 - `all_t_od_city`**

출발 시군구와 도착 시군구가 모두 있는 OD 테이블의 이동쌍 집계입니다. `O_CTY_NM`에서 `D_CTY_NM`으로 이동한 규모를 나타내며, 상위 이동축을 찾는 데 사용합니다.

핵심 설명: 이 표는 이후 그래프를 해석하기 위한 기준 데이터입니다. 행 수, 기간, 지역 단위, 비중·변화율 컬럼을 먼저 확인하고 그래프를 읽으면 됩니다.


In [ ]:
if "all_t_od_city" not in globals() or (hasattr(all_t_od_city, "empty") and all_t_od_city.empty):
    pd.DataFrame({"안내": ["시군구 OD TOP 30 그래프는 대용량 셀을 건너뛰어서 실행하지 않습니다.", "월별 행정동 분석은 아래 빠른 t24 패널과 시각화 섹션을 사용하세요."]})
else:
    od_top30 = all_t_od_city.head(30).copy()
    od_top30["OD_LABEL"] = od_top30["TABLE_ID"] + " | " + od_top30["O_CTY_NM"] + " -> " + od_top30["D_CTY_NM"]

    ax = od_top30.sort_values("CNT_SUM").plot(
        kind="barh",
        x="OD_LABEL",
        y="CNT_SUM",
        figsize=(12, 10),
        legend=False,
        title="전체 t 데이터 시군구 OD TOP 30",
    )
    ax.set_xlabel("CNT 총합")
    ax.set_ylabel("테이블 | 출발 -> 도착")
    plt.tight_layout()
    plt.show()


**그래프 설명 - 시군구 OD TOP 30**

전체 OD 테이블에서 가장 큰 출발-도착 시군구 조합입니다. 같은 구 내부 이동과 구 간 이동이 함께 나타날 수 있으므로, `O_CTY_NM == D_CTY_NM` 여부를 추가로 나누면 내부 이동과 외부 유입·유출을 분리할 수 있습니다.


In [ ]:
if RUN_FULL_T_HEAVY_EDA:
    duration_parts = []
    for table_id in T_DATA_FILES:
        if "DURATION" not in table_columns(table_id):
            continue
        duration_parts.append(f'''
            SELECT
                '{table_id}' AS TABLE_ID,
                DURATION,
                SUM({cnt_expr(table_id)}) AS CNT_SUM
            FROM {scan_sql(table_id)}
            GROUP BY 1, 2
        ''')

    all_t_duration = q(union_query(duration_parts)).sort_values(["TABLE_ID", "DURATION"])
    all_t_duration["CNT_SHARE_IN_TABLE"] = all_t_duration["CNT_SUM"] / all_t_duration.groupby("TABLE_ID")["CNT_SUM"].transform("sum")
    all_t_duration
else:
    all_t_duration = pd.DataFrame()
    pd.DataFrame({"안내": ["전체 t 체류시간 집계는 고비용 전체 t 스캔이라 기본 실행을 건너뜁니다.", "필요하면 첫 셀에서 RUN_FULL_T_HEAVY_EDA = True로 바꾸고 다시 실행하세요."]})


**데이터프레임 설명 - `all_t_duration`**

체류시간(`DURATION`)이 있는 `t16`, `t26`, `t27`의 체류시간별 집계입니다. 도착 기반, 출발 기반, 시군구/행정동 기준이 섞여 있으므로 테이블별 비중으로 비교하는 것이 안전합니다.

핵심 설명: 이 표는 이후 그래프를 해석하기 위한 기준 데이터입니다. 행 수, 기간, 지역 단위, 비중·변화율 컬럼을 먼저 확인하고 그래프를 읽으면 됩니다.


In [ ]:
if "all_t_duration" not in globals() or (hasattr(all_t_duration, "empty") and all_t_duration.empty):
    pd.DataFrame({"안내": ["체류시간별 비중 그래프는 대용량 셀을 건너뛰어서 실행하지 않습니다.", "월별 행정동 분석은 아래 빠른 t24 패널과 시각화 섹션을 사용하세요."]})
else:
    duration_pivot = all_t_duration.pivot(index="DURATION", columns="TABLE_ID", values="CNT_SHARE_IN_TABLE").fillna(0)
    ax = duration_pivot.plot(kind="bar", figsize=(14, 5), title="전체 t 데이터 체류시간별 테이블 내부 비중")
    ax.set_xlabel("체류시간 코드")
    ax.set_ylabel("테이블 내부 비중")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


**그래프 설명 - 체류시간별 비중**

체류시간 코드별 비중을 테이블별로 비교합니다. 짧은 체류와 긴 체류 중 어느 구간이 큰지 확인하고, 목적·교통수단과 교차하면 방문 성격을 더 잘 해석할 수 있습니다.


In [ ]:
if RUN_FULL_T_EDA:
    t22_forn = q(f'''
        SELECT
            FORN_GB,
            SUM({cnt_expr("t22")}) AS CNT_SUM
        FROM {scan_sql("t22")}
        GROUP BY 1
        ORDER BY CNT_SUM DESC
    ''')
    t22_forn["CNT_SHARE"] = t22_forn["CNT_SUM"] / t22_forn["CNT_SUM"].sum()
    t22_forn
else:
    t22_forn = pd.DataFrame()
    pd.DataFrame({"안내": ["t22 내외국인 집계는 대용량 전체 t 스캔이라 기본 실행을 건너뜁니다.", "필요하면 첫 셀에서 RUN_FULL_T_EDA = True로 바꾸고 다시 실행하세요."]})


**데이터프레임 설명 - `t22_forn`**

`t22`에만 있는 내외국인 구분(`FORN_GB`)별 집계입니다. `CNT_SHARE`는 전체 `t22` 생활/체류 인구 중 각 구분이 차지하는 비중입니다.

핵심 설명: 이 표는 이후 그래프를 해석하기 위한 기준 데이터입니다. 행 수, 기간, 지역 단위, 비중·변화율 컬럼을 먼저 확인하고 그래프를 읽으면 됩니다.


In [ ]:
if "t22_forn" not in globals() or (hasattr(t22_forn, "empty") and t22_forn.empty):
    pd.DataFrame({"안내": ["t22 내외국인 구분별 비중 그래프는 대용량 셀을 건너뛰어서 실행하지 않습니다.", "월별 행정동 분석은 아래 빠른 t24 패널과 시각화 섹션을 사용하세요."]})
else:
    ax = t22_forn.plot(kind="bar", x="FORN_GB", y="CNT_SHARE", figsize=(8, 4), legend=False, title="t22 내외국인 구분별 비중")
    ax.set_xlabel("내외국인 구분 코드")
    ax.set_ylabel("비중")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()


**그래프 설명 - t22 내외국인 구분별 비중**

생활/체류 인구 성격의 `t22`에서 내국인과 외국인 구분이 전체에서 차지하는 비중을 봅니다. 외국인 비중이 높은 시간대나 행정동을 추가로 나누면 지역 특성을 더 구체적으로 볼 수 있습니다.


In [ ]:
if RUN_FULL_T_HEAVY_EDA:
    quality_parts = []
    key_candidates = [
        "ETL_YMD", "ETL_YM", "CTY_NM", "ADMI_NM", "D_CTY_NM", "D_ADMI_NM",
        "O_CTY_NM", "O_ADMI_NM", "SEX_CD", "AGE_GRP", "PURPOSE", "TRANS_GB", "DURATION"
    ]

    for table_id in T_DATA_FILES:
        cols = table_columns(table_id)
        checks = ["COUNT(*) AS ROWS"]
        for col in key_candidates:
            alias = f"{col}_NULLS"
            if col in cols:
                checks.append(f"SUM(CASE WHEN {qname(col)} IS NULL THEN 1 ELSE 0 END) AS {alias}")
            else:
                checks.append(f"NULL AS {alias}")
        checks.append(f"SUM(CASE WHEN {cnt_expr(table_id)} < 0 THEN 1 ELSE 0 END) AS NEGATIVE_CNT_ROWS")
        checks.append(f"SUM(CASE WHEN {cnt_expr(table_id)} = 0 THEN 1 ELSE 0 END) AS ZERO_CNT_ROWS")
        quality_parts.append(f'''
            SELECT
                '{table_id}' AS TABLE_ID,
                {", ".join(checks)}
            FROM {scan_sql(table_id)}
        ''')

    all_t_quality = q(union_query(quality_parts)).fillna(0)
    all_t_quality
else:
    all_t_quality = pd.DataFrame()
    pd.DataFrame({"안내": ["전체 t 품질 점검는 고비용 전체 t 스캔이라 기본 실행을 건너뜁니다.", "필요하면 첫 셀에서 RUN_FULL_T_HEAVY_EDA = True로 바꾸고 다시 실행하세요."]})


**데이터프레임 설명 - `all_t_quality`**

전체 t 테이블의 핵심 컬럼 결측치, 음수 CNT, 0 CNT 행을 점검하는 품질 확인표입니다. 결측이 많은 컬럼은 해당 테이블의 분석 축에서 제외하거나, 결측 자체를 별도 범주로 볼지 결정해야 합니다.

핵심 설명: 이 표는 이후 그래프를 해석하기 위한 기준 데이터입니다. 행 수, 기간, 지역 단위, 비중·변화율 컬럼을 먼저 확인하고 그래프를 읽으면 됩니다.


## 전체 t 데이터 EDA에서 만든 파생변수 의미

| 파생변수/집계 컬럼 | 생성 위치 | 의미 | 해석 포인트 |
|---|---|---|---|
| `TABLE_ID` | 모든 통합 집계 | 원본 t 테이블 구분 | 서로 다른 정의의 데이터를 섞어 해석하지 않기 위한 기준 |
| `DESCRIPTION` | 파일 인벤토리 | 테이블 분석 단위 설명 | 출발/도착/OD, 목적/교통/체류시간 중심인지 확인 |
| `CNT_SUM` | 모든 집계 | 전체 행을 기준으로 합산한 규모 | 원본 전체 사용 결과, 단 순인구가 아니라 테이블 정의별 관측 총량 |
| `CNT_MEAN` | 전체 요약 | 행당 평균 CNT | 행 단위가 세분화될수록 작아질 수 있음 |
| `MONTH_START` | 월별 추세 | `ETL_YMD` 또는 `ETL_YM`을 월 단위로 통일 | 일자/월자 테이블을 같은 축에서 비교 |
| `YEAR_MONTH` | 월별 추세 | 그래프 표시용 연-월 문자열 | 월별 선 그래프 x축 |
| `CNT_SHARE_IN_TABLE` | 목적·교통·성별연령·요일·체류시간 | 테이블 내부 비중 | 테이블 규모 차이를 제거한 구성 비교 |
| `REGION_ROLE` | 지역 통합 EDA | 출발/도착/단일지역 구분 | `O_`, `D_` 접두사의 의미 보존 |
| `REGION_LEVEL` | 지역 통합 EDA | 시군구/행정동 구분 | 공간 해상도가 다른 테이블을 분리해서 해석 |
| `TIME_ROLE` | 시간대 통합 EDA | 출발시간/도착시간/단일 시간 구분 | 시간 컬럼 의미가 다른 테이블을 구분 |
| `OD_LABEL` | OD TOP 그래프 | 출발지 -> 도착지 표시 라벨 | 주요 이동축 시각화용 |

이 섹션의 핵심은 “전체 원본 행을 사용하되, 테이블 정의가 다르기 때문에 집계 단위를 명확히 나누어 해석한다”는 점입니다.


# 인구 데이터 EDA

이 섹션은 `../data/population_total.csv`를 사용합니다. t 데이터가 유동·방문·OD 관측 규모라면, 인구 데이터는 행정동별 등록 인구와 세대 규모를 설명하는 기준선입니다. 따라서 단순 총량뿐 아니라 인구 대비 관측 규모를 같이 보면 지역 규모 차이를 줄이고 비교할 수 있습니다.


In [ ]:
population_raw = pd.read_csv(DATA_DIR / "population_total.csv", encoding="cp949")
population_raw.head()


**데이터프레임 설명 - `population_raw`**

원본 인구 데이터입니다. `총인구수`, `세대수`는 쉼표가 들어간 문자열이므로 수치 분석 전에 정수형으로 변환해야 합니다. `행정동 코드`는 10자리 행정기관 코드라서 t 데이터의 8자리 행정동 코드와 맞추는 파생키가 필요합니다.

핵심 설명: 이 표는 이후 그래프를 해석하기 위한 기준 데이터입니다. 행 수, 기간, 지역 단위, 비중·변화율 컬럼을 먼저 확인하고 그래프를 읽으면 됩니다.


In [ ]:
population = population_raw.copy()
population["TOTAL_POP"] = population["총인구수"].astype(str).str.replace(",", "", regex=False).astype(int)
population["HOUSEHOLDS"] = population["세대수"].astype(str).str.replace(",", "", regex=False).astype(int)
population["HOUSEHOLD_SIZE"] = population["세대당_인구"].astype(float)
population["BASE_MONTH"] = pd.to_datetime(population["날짜"] + "-01")
population["YEAR_MONTH"] = population["BASE_MONTH"].dt.strftime("%Y-%m")
population["ADMI_CD"] = (population["행정동 코드"].astype("int64") // 100).astype("int64")

area_parts = population["행정구역"].str.split()
population["MEGA_NM"] = area_parts.str[0]
population["CTY_NM"] = area_parts.str[1] + " " + area_parts.str[2]
population["ADMI_NM"] = area_parts.str[-1]

population_clean = population[
    [
        "BASE_MONTH", "YEAR_MONTH", "MEGA_NM", "CTY_NM", "ADMI_NM",
        "행정동 코드", "ADMI_CD", "TOTAL_POP", "HOUSEHOLDS", "HOUSEHOLD_SIZE"
    ]
].sort_values(["BASE_MONTH", "CTY_NM", "ADMI_NM"])

population_clean.head()


**데이터프레임 설명 - `population_clean`**

분석용으로 정리한 인구 데이터입니다. `TOTAL_POP`, `HOUSEHOLDS`, `HOUSEHOLD_SIZE`는 수치형이고, `ADMI_CD`는 t 데이터의 행정동 코드와 결합하기 위한 8자리 코드입니다. `CTY_NM`, `ADMI_NM`은 구·행정동 단위 집계와 그래프에 사용합니다.

핵심 설명: 인구 데이터의 기준 테이블입니다. `ADMI_CD`와 `BASE_MONTH`가 t 데이터와 연결되는 핵심 키입니다.


In [ ]:
population_overview = pd.DataFrame([{
    "ROWS": len(population_clean),
    "START_MONTH": population_clean["BASE_MONTH"].min(),
    "END_MONTH": population_clean["BASE_MONTH"].max(),
    "MONTH_COUNT": population_clean["BASE_MONTH"].nunique(),
    "CTY_COUNT": population_clean["CTY_NM"].nunique(),
    "ADMI_COUNT": population_clean["ADMI_CD"].nunique(),
    "TOTAL_POP_MIN": population_clean["TOTAL_POP"].min(),
    "TOTAL_POP_MAX": population_clean["TOTAL_POP"].max(),
    "MISSING_VALUES": int(population_clean.isna().sum().sum()),
}])
population_overview


**데이터프레임 설명 - `population_overview`**

인구 데이터의 기간, 행정동 수, 결측 여부를 요약한 표입니다. 현재 데이터는 월별 행정동 인구 패널이므로, 월별 변화와 행정동별 차이를 모두 볼 수 있습니다.

핵심 설명: 이 표는 이후 그래프를 해석하기 위한 기준 데이터입니다. 행 수, 기간, 지역 단위, 비중·변화율 컬럼을 먼저 확인하고 그래프를 읽으면 됩니다.


In [ ]:
population_monthly = (
    population_clean.groupby("BASE_MONTH", as_index=False)
    .agg(
        TOTAL_POP=("TOTAL_POP", "sum"),
        HOUSEHOLDS=("HOUSEHOLDS", "sum"),
        AVG_HOUSEHOLD_SIZE=("HOUSEHOLD_SIZE", "mean"),
    )
)
population_monthly["YEAR_MONTH"] = population_monthly["BASE_MONTH"].dt.strftime("%Y-%m")
population_monthly["POP_MOM_PCT"] = population_monthly["TOTAL_POP"].pct_change()
population_monthly["HOUSEHOLDS_MOM_PCT"] = population_monthly["HOUSEHOLDS"].pct_change()
population_monthly


**데이터프레임 설명 - `population_monthly`**

성남시 전체 기준 월별 총인구, 세대수, 평균 세대당 인구입니다. `POP_MOM_PCT`와 `HOUSEHOLDS_MOM_PCT`는 전월 대비 변화율로, 인구와 세대가 같은 방향으로 움직이는지 확인할 때 사용합니다.

핵심 설명: 전체 인구·세대수의 큰 흐름을 먼저 확인하는 표입니다. 행정동 세부 분석 전에 전체 추세를 보는 기준입니다.


In [ ]:
ax = population_monthly.plot(
    x="BASE_MONTH",
    y=["TOTAL_POP", "HOUSEHOLDS"],
    figsize=(14, 4),
    marker="o",
    title="월별 총인구수와 세대수 추세",
)
ax.set_xlabel("월")
ax.set_ylabel("명 / 세대")
plt.tight_layout()
plt.show()


**그래프 설명 - 월별 총인구수와 세대수 추세**

전체 인구와 세대수가 시간에 따라 어떻게 변하는지 확인합니다. 인구는 줄지만 세대수가 늘면 1인 가구 증가나 세대 분화 가능성을 의심할 수 있고, 두 지표가 함께 증가하면 주거 수요 확대 신호로 볼 수 있습니다.


In [ ]:
latest_month = population_clean["BASE_MONTH"].max()
latest_population = population_clean[population_clean["BASE_MONTH"] == latest_month].copy()

latest_by_cty = (
    latest_population.groupby("CTY_NM", as_index=False)
    .agg(
        TOTAL_POP=("TOTAL_POP", "sum"),
        HOUSEHOLDS=("HOUSEHOLDS", "sum"),
        AVG_HOUSEHOLD_SIZE=("HOUSEHOLD_SIZE", "mean"),
        ADMI_COUNT=("ADMI_CD", "nunique"),
    )
    .sort_values("TOTAL_POP", ascending=False)
)
latest_by_cty


**데이터프레임 설명 - `latest_by_cty`**

가장 최근 월 기준 구별 인구·세대 요약입니다. 구별 인구 규모를 비교할 때 사용하며, `AVG_HOUSEHOLD_SIZE`가 낮은 구는 상대적으로 소형 세대 또는 1인 가구 비중이 높을 가능성이 있습니다.

핵심 설명: 이 표는 이후 그래프를 해석하기 위한 기준 데이터입니다. 행 수, 기간, 지역 단위, 비중·변화율 컬럼을 먼저 확인하고 그래프를 읽으면 됩니다.


In [ ]:
ax = latest_by_cty.sort_values("TOTAL_POP").plot(
    kind="barh",
    x="CTY_NM",
    y="TOTAL_POP",
    figsize=(8, 4),
    legend=False,
    title=f"{latest_month:%Y-%m} 구별 총인구수",
)
ax.set_xlabel("총인구수")
ax.set_ylabel("구")
plt.tight_layout()
plt.show()


**그래프 설명 - 최근 월 구별 총인구수**

최근 월 기준으로 어느 구의 등록 인구 규모가 큰지 비교합니다. 이후 t 데이터의 유동·방문 규모를 볼 때, 인구가 많은 구라서 총량이 큰 것인지 인구 대비 활동성이 높은 것인지 구분하는 기준이 됩니다.


In [ ]:
admi_pop_rank = latest_population.sort_values("TOTAL_POP", ascending=False)
admi_pop_top_bottom = pd.concat([
    admi_pop_rank.head(10).assign(RANK_TYPE="인구 상위 10"),
    admi_pop_rank.tail(10).assign(RANK_TYPE="인구 하위 10"),
])
admi_pop_top_bottom[["RANK_TYPE", "CTY_NM", "ADMI_NM", "ADMI_CD", "TOTAL_POP", "HOUSEHOLDS", "HOUSEHOLD_SIZE"]]


**데이터프레임 설명 - `admi_pop_top_bottom`**

최근 월 기준 행정동별 인구 상위·하위 10개입니다. 같은 구 안에서도 행정동별 인구 기반이 크게 다를 수 있으므로, 유동인구 TOP 지역과 비교할 때 유용합니다.

핵심 설명: 이 표는 이후 그래프를 해석하기 위한 기준 데이터입니다. 행 수, 기간, 지역 단위, 비중·변화율 컬럼을 먼저 확인하고 그래프를 읽으면 됩니다.


In [ ]:
start_month = population_clean["BASE_MONTH"].min()
end_month = population_clean["BASE_MONTH"].max()

pop_start = (
    population_clean[population_clean["BASE_MONTH"] == start_month]
    [["ADMI_CD", "CTY_NM", "ADMI_NM", "TOTAL_POP", "HOUSEHOLDS", "HOUSEHOLD_SIZE"]]
    .rename(columns={
        "TOTAL_POP": "START_POP",
        "HOUSEHOLDS": "START_HOUSEHOLDS",
        "HOUSEHOLD_SIZE": "START_HOUSEHOLD_SIZE",
    })
)
pop_end = (
    population_clean[population_clean["BASE_MONTH"] == end_month]
    [["ADMI_CD", "TOTAL_POP", "HOUSEHOLDS", "HOUSEHOLD_SIZE"]]
    .rename(columns={
        "TOTAL_POP": "END_POP",
        "HOUSEHOLDS": "END_HOUSEHOLDS",
        "HOUSEHOLD_SIZE": "END_HOUSEHOLD_SIZE",
    })
)

population_change = pop_start.merge(pop_end, on="ADMI_CD", how="inner")
population_change["POP_CHANGE"] = population_change["END_POP"] - population_change["START_POP"]
population_change["POP_CHANGE_PCT"] = population_change["POP_CHANGE"] / population_change["START_POP"]
population_change["HOUSEHOLDS_CHANGE"] = population_change["END_HOUSEHOLDS"] - population_change["START_HOUSEHOLDS"]
population_change["HOUSEHOLD_SIZE_CHANGE"] = population_change["END_HOUSEHOLD_SIZE"] - population_change["START_HOUSEHOLD_SIZE"]
population_change.sort_values("POP_CHANGE", ascending=False)


**데이터프레임 설명 - `population_change`**

첫 월과 마지막 월을 비교한 행정동별 인구 변화표입니다. `POP_CHANGE`는 순증감 인구수, `POP_CHANGE_PCT`는 시작 월 대비 변화율입니다. 인구는 줄었는데 세대수가 늘어난 지역은 세대 규모 축소 가능성이 있습니다.

핵심 설명: 이 표는 이후 그래프를 해석하기 위한 기준 데이터입니다. 행 수, 기간, 지역 단위, 비중·변화율 컬럼을 먼저 확인하고 그래프를 읽으면 됩니다.


In [ ]:
pop_change_plot = pd.concat([
    population_change.nlargest(10, "POP_CHANGE"),
    population_change.nsmallest(10, "POP_CHANGE"),
]).copy()
pop_change_plot["LABEL"] = pop_change_plot["CTY_NM"] + " " + pop_change_plot["ADMI_NM"]

ax = pop_change_plot.sort_values("POP_CHANGE").plot(
    kind="barh",
    x="LABEL",
    y="POP_CHANGE",
    figsize=(10, 8),
    legend=False,
    title=f"행정동별 인구 증감 TOP/BOTTOM ({start_month:%Y-%m}~{end_month:%Y-%m})",
)
ax.set_xlabel("인구 증감")
ax.set_ylabel("행정동")
plt.tight_layout()
plt.show()


**그래프 설명 - 행정동별 인구 증감 TOP/BOTTOM**

분석 기간 동안 인구가 가장 많이 늘어난 행정동과 가장 많이 줄어든 행정동을 함께 보여줍니다. 유동인구 증가 지역이 실제 거주 인구 증가 지역인지, 아니면 방문·통행 증가 지역인지 구분하는 데 활용합니다.


In [ ]:
household_size_by_cty = (
    population_clean.groupby(["BASE_MONTH", "CTY_NM"], as_index=False)
    .agg(AVG_HOUSEHOLD_SIZE=("HOUSEHOLD_SIZE", "mean"))
)
household_size_pivot = household_size_by_cty.pivot(index="BASE_MONTH", columns="CTY_NM", values="AVG_HOUSEHOLD_SIZE")

ax = household_size_pivot.plot(figsize=(12, 4), marker="o", title="구별 평균 세대당 인구 추세")
ax.set_xlabel("월")
ax.set_ylabel("평균 세대당 인구")
plt.tight_layout()
plt.show()


**그래프 설명 - 구별 평균 세대당 인구 추세**

구별 세대 규모가 시간에 따라 줄어드는지 확인합니다. 세대당 인구가 지속적으로 낮아지면 1~2인 가구 증가, 주거 구성 변화, 생활 서비스 수요 변화와 연결해서 볼 수 있습니다.


In [ ]:
if RUN_MONTHLY_ADMIN_ALL_T and all(name in globals() for name in ["T_DATA_FILES", "table_columns", "q", "union_query", "qname", "cnt_expr", "scan_sql"]):
    admin_code_columns = ["ADMI_CD", "D_ADMI_CD", "O_ADMI_CD"]
    admin_parts = []

    for table_id in T_DATA_FILES:
        cols = table_columns(table_id)
        for col in admin_code_columns:
            if col not in cols:
                continue
            if col.startswith("O_"):
                role = "출발행정동"
            elif col.startswith("D_"):
                role = "도착행정동"
            else:
                role = "행정동"
            admin_parts.append(f'''
                SELECT
                    '{table_id}' AS TABLE_ID,
                    '{role}' AS ADMIN_ROLE,
                    TRY_CAST({qname(col)} AS BIGINT) AS ADMI_CD,
                    SUM({cnt_expr(table_id)}) AS CNT_SUM
                FROM {scan_sql(table_id)}
                WHERE TRY_CAST({qname(col)} AS BIGINT) IS NOT NULL
                GROUP BY 1, 2, 3
            ''')

    all_t_admin = q(union_query(admin_parts)).sort_values("CNT_SUM", ascending=False)

    latest_population_key = latest_population[["ADMI_CD", "CTY_NM", "ADMI_NM", "TOTAL_POP", "HOUSEHOLDS", "HOUSEHOLD_SIZE"]]
    all_t_admin_population = all_t_admin.merge(latest_population_key, on="ADMI_CD", how="inner")
    all_t_admin_population["CNT_PER_PERSON"] = all_t_admin_population["CNT_SUM"] / all_t_admin_population["TOTAL_POP"]
    all_t_admin_population["CNT_PER_HOUSEHOLD"] = all_t_admin_population["CNT_SUM"] / all_t_admin_population["HOUSEHOLDS"]
    all_t_admin_population.sort_values("CNT_PER_PERSON", ascending=False).head(30)
else:
    all_t_admin_population = pd.DataFrame(columns=[
        "TABLE_ID", "ADMIN_ROLE", "ADMI_CD", "CNT_SUM", "CTY_NM", "ADMI_NM",
        "TOTAL_POP", "HOUSEHOLDS", "HOUSEHOLD_SIZE", "CNT_PER_PERSON", "CNT_PER_HOUSEHOLD"
    ])
    pd.DataFrame({
        "안내": [
            "전체 t 데이터 결합 셀입니다. 첫 셀에서 RUN_MONTHLY_ADMIN_ALL_T=True로 바꾸고 앞쪽 준비 셀을 실행하면 결과가 생성됩니다.",
            "월별 행정동 시각화만 보려면 아래의 빠른 t24 월별 행정동 패널부터 실행해도 됩니다."
        ]
    })


**데이터프레임 설명 - `all_t_admin_population`**

행정동 코드가 있는 전체 t 테이블 집계와 최근 월 인구 데이터를 결합한 표입니다. `CNT_PER_PERSON`은 등록인구 1명당 t 데이터 관측 규모로, 실제 개인별 방문 횟수가 아니라 지역 규모를 보정한 비교 지표입니다. 외부 유입이 많은 지역은 이 값이 높게 나올 수 있습니다.

핵심 설명: 이 표는 이후 그래프를 해석하기 위한 기준 데이터입니다. 행 수, 기간, 지역 단위, 비중·변화율 컬럼을 먼저 확인하고 그래프를 읽으면 됩니다.


In [ ]:
if all_t_admin_population.empty:
    pd.DataFrame({"안내": ["all_t_admin_population이 비어 있습니다. 전체 t 데이터 EDA 준비 셀을 실행한 뒤 다시 실행하세요."]})
else:
    admin_pop_top30 = all_t_admin_population.sort_values("CNT_PER_PERSON", ascending=False).head(30).copy()
    admin_pop_top30["LABEL"] = (
        admin_pop_top30["TABLE_ID"] + " | "
        + admin_pop_top30["ADMIN_ROLE"] + " | "
        + admin_pop_top30["CTY_NM"] + " "
        + admin_pop_top30["ADMI_NM"]
    )

    ax = admin_pop_top30.sort_values("CNT_PER_PERSON").plot(
        kind="barh",
        x="LABEL",
        y="CNT_PER_PERSON",
        figsize=(12, 10),
        legend=False,
        title="인구 대비 t 데이터 관측 규모 TOP 30",
    )
    ax.set_xlabel("CNT / 등록인구")
    ax.set_ylabel("테이블 | 역할 | 행정동")
    plt.tight_layout()
    plt.show()


**그래프 설명 - 인구 대비 t 데이터 관측 규모 TOP 30**

인구가 많은 지역이 총량에서 유리한 문제를 줄이기 위해 `CNT_SUM`을 최근 월 등록인구로 나눈 그래프입니다. 값이 높은 지역은 거주 인구 대비 유동·방문·이동 관측이 많은 지역으로 볼 수 있으며, 상권·교통 중심지 또는 외부 유입 지역 후보입니다.


In [ ]:
population_feature_table = pd.DataFrame([
    ["TOTAL_POP", "총인구수에서 쉼표 제거 후 정수 변환", "행정동별 등록 인구 규모", "구/동별 인구 기반 비교"],
    ["HOUSEHOLDS", "세대수에서 쉼표 제거 후 정수 변환", "행정동별 세대 규모", "세대 증가와 인구 증가 비교"],
    ["HOUSEHOLD_SIZE", "세대당_인구 실수형 변환", "세대당 평균 인구", "1~2인 가구 증가 가능성 확인"],
    ["BASE_MONTH", "날짜 + '-01'을 datetime 변환", "월 단위 기준일", "월별 추세와 t 데이터 월 집계 결합"],
    ["YEAR_MONTH", "BASE_MONTH를 YYYY-MM 문자열로 변환", "월 표시용 라벨", "그래프 x축과 월별 피벗"],
    ["ADMI_CD", "10자리 행정동 코드 // 100", "t 데이터와 결합 가능한 8자리 행정동 코드", "인구 대비 t 관측 규모 계산"],
    ["CTY_NM", "행정구역 문자열에서 시+구 추출", "구 단위 지역명", "구별 인구/세대 비교"],
    ["ADMI_NM", "행정구역 문자열 마지막 토큰", "행정동명", "행정동 순위와 변화 분석"],
    ["POP_MOM_PCT", "TOTAL_POP.pct_change()", "전월 대비 인구 변화율", "월별 인구 급변 확인"],
    ["POP_CHANGE", "마지막 월 인구 - 첫 월 인구", "기간 전체 인구 순증감", "증가/감소 행정동 탐색"],
    ["POP_CHANGE_PCT", "POP_CHANGE / 첫 월 인구", "기간 전체 인구 변화율", "규모가 다른 행정동 비교"],
    ["CNT_PER_PERSON", "t CNT_SUM / 최근 월 TOTAL_POP", "인구 1명당 t 관측 규모", "거주 인구 대비 유동·방문 강도 비교"],
], columns=["파생변수", "생성 기준", "의미", "EDA 활용"])

population_feature_table


**데이터프레임 설명 - `population_feature_table`**

인구 데이터 EDA에서 새로 만든 파생변수의 의미표입니다. 핵심은 `ADMI_CD`로 t 데이터와 인구 데이터를 연결하고, `CNT_PER_PERSON`으로 지역 인구 규모를 보정해 비교하는 것입니다.

핵심 설명: 이 표는 이후 그래프를 해석하기 위한 기준 데이터입니다. 행 수, 기간, 지역 단위, 비중·변화율 컬럼을 먼저 확인하고 그래프를 읽으면 됩니다.


## 빠른 시각화용 t24 월별 행정동 패널

전체 t 테이블을 모두 집계하는 `monthly_admin_panel`은 가장 완전하지만 실행 시간이 오래 걸릴 수 있습니다. 아래 셀은 팀 기준인 월별 행정동 단위를 유지하면서, 기본 예시인 `t24`만 먼저 집계해 시각화를 빠르게 확인하기 위한 패널입니다.

이 셀을 실행하면 뒤의 시각화 모음은 `t24` 기준으로 바로 그릴 수 있습니다.


In [ ]:
if "con" not in globals():
    con = duckdb.connect()

if "scan_sql" not in globals():
    def scan_sql(table_id):
        file_map = {
            "t24": "t24_2023_2025_all_date_final.parquet",
        }
        return f"read_parquet('{(DATA_DIR / file_map[table_id]).as_posix()}')"

if "q" not in globals():
    def q(sql):
        return con.execute(sql).df()

t24_monthly_admin_panel = q(f'''
    SELECT
        't24' AS TABLE_ID,
        '행정동' AS ADMIN_ROLE,
        CAST(date_trunc('month', CAST(ETL_YMD AS DATE)) AS DATE) AS BASE_MONTH,
        CAST(ADMI_CD AS BIGINT) AS ADMI_CD,
        SUM(CNT) AS CNT_SUM
    FROM {scan_sql("t24")}
    GROUP BY 1, 2, 3, 4
''')

t24_monthly_admin_panel["BASE_MONTH"] = pd.to_datetime(t24_monthly_admin_panel["BASE_MONTH"])
t24_monthly_admin_panel["YEAR_MONTH"] = t24_monthly_admin_panel["BASE_MONTH"].dt.strftime("%Y-%m")

t24_monthly_admin_panel = t24_monthly_admin_panel.merge(
    population_clean[
        ["BASE_MONTH", "YEAR_MONTH", "ADMI_CD", "CTY_NM", "ADMI_NM", "TOTAL_POP", "HOUSEHOLDS", "HOUSEHOLD_SIZE"]
    ],
    on=["BASE_MONTH", "YEAR_MONTH", "ADMI_CD"],
    how="inner",
)

t24_monthly_admin_panel["CNT_PER_PERSON"] = t24_monthly_admin_panel["CNT_SUM"] / t24_monthly_admin_panel["TOTAL_POP"]
t24_monthly_admin_panel["CNT_PER_1K_POP"] = t24_monthly_admin_panel["CNT_PER_PERSON"] * 1000
t24_monthly_admin_panel["CNT_PER_HOUSEHOLD"] = t24_monthly_admin_panel["CNT_SUM"] / t24_monthly_admin_panel["HOUSEHOLDS"]

t24_monthly_admin_panel = t24_monthly_admin_panel.sort_values(["ADMI_CD", "BASE_MONTH"])
t24_monthly_admin_panel["CNT_MOM_CHANGE"] = t24_monthly_admin_panel.groupby("ADMI_CD")["CNT_SUM"].diff()
t24_monthly_admin_panel["CNT_MOM_PCT"] = t24_monthly_admin_panel.groupby("ADMI_CD")["CNT_SUM"].pct_change()
t24_monthly_admin_panel["CNT_PER_PERSON_MOM_PCT"] = t24_monthly_admin_panel.groupby("ADMI_CD")["CNT_PER_PERSON"].pct_change()
t24_monthly_admin_panel["CNT_PER_PERSON_MA3"] = (
    t24_monthly_admin_panel.groupby("ADMI_CD")["CNT_PER_PERSON"]
    .transform(lambda s: s.rolling(3, min_periods=1).mean())
)

focus_table_id = "t24"
focus_admin_role = "행정동"
focus_monthly_admin = t24_monthly_admin_panel.copy()

monthly_admin_outliers = (
    focus_monthly_admin.dropna(subset=["CNT_PER_PERSON_MOM_PCT"])
    .assign(ABS_MOM_PCT=lambda df: df["CNT_PER_PERSON_MOM_PCT"].abs())
    .sort_values("ABS_MOM_PCT", ascending=False)
)

t24_monthly_admin_panel.head()


**데이터프레임 설명 - `t24_monthly_admin_panel`**

`t24`만 월별 행정동 단위로 집계하고 인구 데이터를 붙인 빠른 시각화용 패널입니다. 전체 t 데이터 통합 분석 전에도 이 표로 행정동별 월별 유동 규모, 인구 보정 지표, 전월 대비 변화율을 바로 확인할 수 있습니다.

핵심 설명: 팀 분석의 기준 테이블입니다. 월별·행정동별로 인구 보정 유동 규모와 변화율을 비교할 때 이 표를 중심으로 해석하면 됩니다.


# 월별 행정동 단위 통합 EDA

팀 분석 기준을 **월별 행정동 단위**로 맞춥니다. 이 섹션에서는 전체 t 데이터 중 행정동 코드가 있는 테이블을 월(`BASE_MONTH`)과 행정동(`ADMI_CD`) 기준으로 집계한 뒤, `population_total.csv`의 월별 행정동 인구와 결합합니다.

핵심 해석 기준은 아래와 같습니다.

| 기준 | 의미 | 왜 필요한가 |
|---|---|---|
| 월별 | 일별 변동을 월 단위로 완화 | 휴일·요일·단기 이벤트 영향을 줄이고 추세를 보기 좋음 |
| 행정동 | 가장 세밀한 공통 공간 단위 | 구 단위보다 실제 지역 차이를 잘 보여줌 |
| 테이블/역할 분리 | `t24`, `t26`, 출발/도착 등 정의를 보존 | 서로 다른 의미의 데이터를 섞어 잘못 해석하는 것을 방지 |
| 인구 대비 지표 | `CNT / TOTAL_POP` | 인구가 많은 동이 무조건 커 보이는 문제를 보정 |
| 전월 대비 변화율 | 이번 달이 지난달보다 얼마나 변했는지 | 급증·급감 행정동을 찾기 쉬움 |

주의할 점은 `CNT_PER_PERSON`이 실제 1인당 방문 횟수를 뜻하는 것은 아니라는 점입니다. t 데이터의 `CNT`는 테이블 정의별 관측 규모이므로, 이 지표는 **등록인구 대비 상대적 유동/이동 강도**로 해석합니다.


In [ ]:
if RUN_MONTHLY_ADMIN_ALL_T:
    monthly_admin_parts = []
    admin_code_columns = ["ADMI_CD", "D_ADMI_CD", "O_ADMI_CD"]

    for table_id in T_DATA_FILES:
        cols = table_columns(table_id)
        d = date_expr(table_id)
        if d is None:
            continue

        for col in admin_code_columns:
            if col not in cols:
                continue

            if col.startswith("O_"):
                role = "출발행정동"
            elif col.startswith("D_"):
                role = "도착행정동"
            else:
                role = "행정동"

            monthly_admin_parts.append(f'''
                SELECT
                    '{table_id}' AS TABLE_ID,
                    '{role}' AS ADMIN_ROLE,
                    CAST(date_trunc('month', {d}) AS DATE) AS BASE_MONTH,
                    TRY_CAST({qname(col)} AS BIGINT) AS ADMI_CD,
                    SUM({cnt_expr(table_id)}) AS CNT_SUM
                FROM {scan_sql(table_id)}
                WHERE TRY_CAST({qname(col)} AS BIGINT) IS NOT NULL
                GROUP BY 1, 2, 3, 4
            ''')

    monthly_admin_t = q(union_query(monthly_admin_parts))
    monthly_admin_t["BASE_MONTH"] = pd.to_datetime(monthly_admin_t["BASE_MONTH"])
    monthly_admin_t["YEAR_MONTH"] = monthly_admin_t["BASE_MONTH"].dt.strftime("%Y-%m")
    monthly_admin_t.head()
else:
    monthly_admin_t = pd.DataFrame()
    pd.DataFrame({"안내": ["전체 t 월별 행정동 집계는 전체 t 월별 행정동 스캔이라 기본 실행을 건너뜁니다.", "필요하면 첫 셀에서 RUN_MONTHLY_ADMIN_ALL_T = True로 바꾸고 다시 실행하세요."]})


**데이터프레임 설명 - `monthly_admin_t`**

전체 t 데이터에서 행정동 코드가 있는 테이블만 월별·행정동별로 집계한 결과입니다. 한 행은 `TABLE_ID`, `ADMIN_ROLE`, `BASE_MONTH`, `ADMI_CD` 조합의 `CNT_SUM`입니다. `ADMIN_ROLE`은 출발 행정동인지, 도착 행정동인지, 단일 행정동 기준인지 구분합니다.

핵심 설명: t 데이터만 월별 행정동 단위로 맞춘 중간 집계입니다. 여기에 인구를 붙이면 분석 후보 패널이 됩니다.


In [ ]:
if "monthly_admin_t" not in globals() or (hasattr(monthly_admin_t, "empty") and monthly_admin_t.empty):
    pd.DataFrame({"안내": ["전체 t 월별 행정동 인구 결합는 대용량 셀을 건너뛰어서 실행하지 않습니다.", "월별 행정동 분석은 아래 빠른 t24 패널과 시각화 섹션을 사용하세요."]})
else:
    monthly_admin_panel = monthly_admin_t.merge(
        population_clean[
            ["BASE_MONTH", "YEAR_MONTH", "ADMI_CD", "CTY_NM", "ADMI_NM", "TOTAL_POP", "HOUSEHOLDS", "HOUSEHOLD_SIZE"]
        ],
        on=["BASE_MONTH", "YEAR_MONTH", "ADMI_CD"],
        how="inner",
    )

    monthly_admin_panel["CNT_PER_PERSON"] = monthly_admin_panel["CNT_SUM"] / monthly_admin_panel["TOTAL_POP"]
    monthly_admin_panel["CNT_PER_1K_POP"] = monthly_admin_panel["CNT_PER_PERSON"] * 1000
    monthly_admin_panel["CNT_PER_HOUSEHOLD"] = monthly_admin_panel["CNT_SUM"] / monthly_admin_panel["HOUSEHOLDS"]

    monthly_admin_panel = monthly_admin_panel.sort_values(["TABLE_ID", "ADMIN_ROLE", "ADMI_CD", "BASE_MONTH"])
    monthly_admin_panel["CNT_MOM_CHANGE"] = monthly_admin_panel.groupby(["TABLE_ID", "ADMIN_ROLE", "ADMI_CD"])["CNT_SUM"].diff()
    monthly_admin_panel["CNT_MOM_PCT"] = monthly_admin_panel.groupby(["TABLE_ID", "ADMIN_ROLE", "ADMI_CD"])["CNT_SUM"].pct_change()
    monthly_admin_panel["CNT_PER_PERSON_MOM_PCT"] = monthly_admin_panel.groupby(["TABLE_ID", "ADMIN_ROLE", "ADMI_CD"])["CNT_PER_PERSON"].pct_change()
    monthly_admin_panel["CNT_PER_PERSON_MA3"] = (
        monthly_admin_panel.groupby(["TABLE_ID", "ADMIN_ROLE", "ADMI_CD"])["CNT_PER_PERSON"]
        .transform(lambda s: s.rolling(3, min_periods=1).mean())
    )

    monthly_admin_panel.head()


**데이터프레임 설명 - `monthly_admin_panel`**

월별 행정동 t 집계에 같은 월의 행정동 인구를 붙인 핵심 분석 테이블입니다. 이 표를 기준으로 팀 분석을 진행하면 됩니다.

주요 컬럼 해석:

| 컬럼 | 의미 |
|---|---|
| `CNT_SUM` | 해당 월·행정동·테이블·역할의 전체 관측 규모 |
| `TOTAL_POP` | 같은 월·행정동의 등록 인구 |
| `CNT_PER_PERSON` | 인구 1명당 t 관측 규모 |
| `CNT_PER_1K_POP` | 인구 1,000명당 t 관측 규모 |
| `CNT_MOM_PCT` | 전월 대비 `CNT_SUM` 변화율 |
| `CNT_PER_PERSON_MOM_PCT` | 전월 대비 인구 보정 관측 규모 변화율 |
| `CNT_PER_PERSON_MA3` | 인구 보정 관측 규모의 3개월 이동평균 |

핵심 설명: 팀 분석의 기준 테이블입니다. 월별·행정동별로 인구 보정 유동 규모와 변화율을 비교할 때 이 표를 중심으로 해석하면 됩니다.


In [ ]:
if "monthly_admin_panel" not in globals() or (hasattr(monthly_admin_panel, "empty") and monthly_admin_panel.empty):
    pd.DataFrame({"안내": ["전체 t 월별 행정동 커버리지는 대용량 셀을 건너뛰어서 실행하지 않습니다.", "월별 행정동 분석은 아래 빠른 t24 패널과 시각화 섹션을 사용하세요."]})
else:
    monthly_admin_coverage = (
        monthly_admin_panel.groupby(["TABLE_ID", "ADMIN_ROLE"], as_index=False)
        .agg(
            ROWS=("CNT_SUM", "size"),
            MONTH_COUNT=("BASE_MONTH", "nunique"),
            ADMI_COUNT=("ADMI_CD", "nunique"),
            CNT_SUM=("CNT_SUM", "sum"),
            AVG_CNT_PER_PERSON=("CNT_PER_PERSON", "mean"),
            MAX_CNT_PER_PERSON=("CNT_PER_PERSON", "max"),
        )
        .sort_values(["TABLE_ID", "ADMIN_ROLE"])
    )
    monthly_admin_coverage


**데이터프레임 설명 - `monthly_admin_coverage`**

월별 행정동 단위로 인구와 매칭된 t 테이블의 커버리지입니다. `MONTH_COUNT`가 36이면 2023-01~2025-12 전체 월이 잡힌 것이고, `ADMI_COUNT`가 50이면 인구 데이터의 50개 행정동과 모두 연결된 것입니다.

핵심 설명: 이 표는 이후 그래프를 해석하기 위한 기준 데이터입니다. 행 수, 기간, 지역 단위, 비중·변화율 컬럼을 먼저 확인하고 그래프를 읽으면 됩니다.


In [ ]:
focus_table_id = "t24"
focus_admin_role = "행정동"

if "monthly_admin_panel" in globals():
    focus_monthly_admin = monthly_admin_panel[
        (monthly_admin_panel["TABLE_ID"] == focus_table_id)
        & (monthly_admin_panel["ADMIN_ROLE"] == focus_admin_role)
    ].copy()
elif "t24_monthly_admin_panel" in globals():
    focus_monthly_admin = t24_monthly_admin_panel.copy()
else:
    raise NameError("먼저 '빠른 시각화용 t24 월별 행정동 패널' 셀을 실행해 주세요.")

focus_monthly_admin.head()


**설명 - 기본 확인 테이블 선택**

월별 행정동 EDA의 기본 예시는 `t24`를 사용합니다. `t24`는 행정동 단위 시간대별 목적 유동인구라서, 인구 데이터와 공간 단위가 잘 맞고 팀원들이 해석하기 쉽습니다. 다른 테이블을 보고 싶으면 `focus_table_id`, `focus_admin_role`만 바꾸면 같은 그래프를 재사용할 수 있습니다.


In [ ]:
latest_focus_month = focus_monthly_admin["BASE_MONTH"].max()
latest_focus_admin = focus_monthly_admin[focus_monthly_admin["BASE_MONTH"] == latest_focus_month].copy()

latest_focus_top15 = latest_focus_admin.sort_values("CNT_PER_PERSON", ascending=False).head(15)
latest_focus_top15[
    ["YEAR_MONTH", "CTY_NM", "ADMI_NM", "TOTAL_POP", "CNT_SUM", "CNT_PER_PERSON", "CNT_PER_1K_POP"]
]


**데이터프레임 설명 - `latest_focus_top15`**

최근 월 기준 인구 대비 유동인구 규모가 큰 행정동 TOP 15입니다. `CNT_SUM`이 큰 지역이 아니라 `CNT_PER_PERSON`이 큰 지역을 보는 이유는, 등록 인구 규모를 보정해 외부 유입·상권·교통 중심지 성격을 더 잘 보기 위해서입니다.

핵심 설명: 이 표는 이후 그래프를 해석하기 위한 기준 데이터입니다. 행 수, 기간, 지역 단위, 비중·변화율 컬럼을 먼저 확인하고 그래프를 읽으면 됩니다.


In [ ]:
plot_df = latest_focus_top15.copy()
plot_df["LABEL"] = plot_df["CTY_NM"] + " " + plot_df["ADMI_NM"]

ax = plot_df.sort_values("CNT_PER_1K_POP").plot(
    kind="barh",
    x="LABEL",
    y="CNT_PER_1K_POP",
    figsize=(10, 7),
    legend=False,
    title=f"{focus_table_id} {latest_focus_month:%Y-%m} 행정동별 인구 1,000명당 유동 규모 TOP 15",
)
ax.set_xlabel("CNT / 인구 1,000명")
ax.set_ylabel("행정동")
plt.tight_layout()
plt.show()


**그래프 설명 - 최근 월 인구 1,000명당 유동 규모 TOP 15**

최근 월에 등록 인구 대비 유동 규모가 큰 행정동을 보여줍니다. 값이 높을수록 거주 인구보다 외부 방문·통행·활동 관측이 많은 지역일 가능성이 큽니다. 단, 테이블 정의가 `t24`이므로 “행정동 시간대별 목적 유동인구” 기준의 상대 강도로 해석합니다.


In [ ]:
trend_admin_codes = latest_focus_top15["ADMI_CD"].head(5).tolist()
trend_df = focus_monthly_admin[focus_monthly_admin["ADMI_CD"].isin(trend_admin_codes)].copy()
trend_df["LABEL"] = trend_df["CTY_NM"] + " " + trend_df["ADMI_NM"]

trend_pivot = trend_df.pivot(index="BASE_MONTH", columns="LABEL", values="CNT_PER_1K_POP")
ax = trend_pivot.plot(figsize=(14, 5), marker="o", title=f"{focus_table_id} 주요 행정동 월별 인구 1,000명당 유동 규모")
ax.set_xlabel("월")
ax.set_ylabel("CNT / 인구 1,000명")
plt.tight_layout()
plt.show()


**그래프 설명 - 주요 행정동 월별 추세**

최근 월 상위 행정동 5개의 월별 추세입니다. 한 달만 높은 지역인지, 지속적으로 높은 지역인지 구분할 수 있습니다. 지속적으로 높으면 구조적인 중심지일 가능성이 있고, 특정 월만 튀면 이벤트나 계절 요인을 확인해야 합니다.


In [ ]:
top10_admin_codes = latest_focus_top15["ADMI_CD"].head(10).tolist()
heatmap_df = focus_monthly_admin[focus_monthly_admin["ADMI_CD"].isin(top10_admin_codes)].copy()
heatmap_df["LABEL"] = heatmap_df["CTY_NM"] + " " + heatmap_df["ADMI_NM"]
heatmap_pivot = heatmap_df.pivot(index="LABEL", columns="YEAR_MONTH", values="CNT_PER_1K_POP")

fig, ax = plt.subplots(figsize=(16, 6))
im = ax.imshow(heatmap_pivot.values, aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(len(heatmap_pivot.columns)))
ax.set_xticklabels(heatmap_pivot.columns, rotation=45, ha="right")
ax.set_yticks(range(len(heatmap_pivot.index)))
ax.set_yticklabels(heatmap_pivot.index)
ax.set_title(f"{focus_table_id} 행정동별 월별 인구 1,000명당 유동 규모 히트맵")
fig.colorbar(im, ax=ax, label="CNT / 인구 1,000명")
plt.tight_layout()
plt.show()


**그래프 설명 - 월별 행정동 히트맵**

행은 행정동, 열은 월입니다. 색이 진할수록 인구 대비 유동 규모가 큽니다. 히트맵은 특정 행정동이 어느 달에 강해지는지, 여러 행정동이 동시에 강해지는 계절 패턴이 있는지 빠르게 찾는 데 좋습니다.


In [ ]:
cty_monthly_panel = (
    focus_monthly_admin.groupby(["BASE_MONTH", "YEAR_MONTH", "CTY_NM"], as_index=False)
    .agg(
        CNT_SUM=("CNT_SUM", "sum"),
        TOTAL_POP=("TOTAL_POP", "sum"),
        HOUSEHOLDS=("HOUSEHOLDS", "sum"),
    )
)
cty_monthly_panel["CNT_PER_1K_POP"] = cty_monthly_panel["CNT_SUM"] / cty_monthly_panel["TOTAL_POP"] * 1000

cty_pivot = cty_monthly_panel.pivot(index="BASE_MONTH", columns="CTY_NM", values="CNT_PER_1K_POP")
ax = cty_pivot.plot(figsize=(12, 4), marker="o", title=f"{focus_table_id} 구별 월별 인구 1,000명당 유동 규모")
ax.set_xlabel("월")
ax.set_ylabel("CNT / 인구 1,000명")
plt.tight_layout()
plt.show()


**그래프 설명 - 구별 월별 인구 보정 추세**

행정동 데이터를 구 단위로 합쳐 월별 흐름을 비교한 그래프입니다. 세부 분석은 행정동 단위로 하되, 팀 공유나 발표에서는 구 단위 선 그래프로 큰 방향을 먼저 설명하면 이해가 쉽습니다.


In [ ]:
monthly_admin_outliers = (
    focus_monthly_admin.dropna(subset=["CNT_PER_PERSON_MOM_PCT"])
    .assign(ABS_MOM_PCT=lambda df: df["CNT_PER_PERSON_MOM_PCT"].abs())
    .sort_values("ABS_MOM_PCT", ascending=False)
    .head(30)
)
monthly_admin_outliers[
    [
        "YEAR_MONTH", "CTY_NM", "ADMI_NM", "TOTAL_POP", "CNT_SUM",
        "CNT_PER_1K_POP", "CNT_PER_PERSON_MOM_PCT"
    ]
]


**데이터프레임 설명 - `monthly_admin_outliers`**

전월 대비 인구 보정 유동 규모가 크게 변한 행정동·월 TOP 30입니다. 급증·급감 행정동을 찾는 표이며, 실제 해석 단계에서는 행사, 개학/방학, 명절, 데이터 누락 가능성 등을 함께 확인해야 합니다.

핵심 설명: 이 표는 이후 그래프를 해석하기 위한 기준 데이터입니다. 행 수, 기간, 지역 단위, 비중·변화율 컬럼을 먼저 확인하고 그래프를 읽으면 됩니다.


In [ ]:
monthly_admin_feature_table = pd.DataFrame([
    ["BASE_MONTH", "ETL_YMD 또는 ETL_YM을 월 시작일로 변환", "월별 분석 기준일", "모든 t 데이터와 인구 데이터를 같은 시간 단위로 맞춤"],
    ["ADMI_CD", "t 데이터 행정동 코드 또는 인구 데이터 10자리 코드 // 100", "행정동 결합 키", "t 데이터와 population_total.csv 연결"],
    ["ADMIN_ROLE", "ADMI_CD/D_ADMI_CD/O_ADMI_CD 컬럼명에서 생성", "행정동의 의미 구분", "출발·도착·단일 행정동 기준 혼동 방지"],
    ["CNT_SUM", "월·행정동·테이블·역할별 CNT 합계", "전체 관측 규모", "행정동별 총량 비교"],
    ["TOTAL_POP", "같은 월·행정동 등록 인구", "인구 기준선", "지역 규모 보정"],
    ["CNT_PER_PERSON", "CNT_SUM / TOTAL_POP", "인구 1명당 관측 규모", "거주 인구 대비 유동/이동 강도 비교"],
    ["CNT_PER_1K_POP", "CNT_PER_PERSON * 1000", "인구 1,000명당 관측 규모", "그래프 축을 읽기 쉽게 변환"],
    ["CNT_PER_HOUSEHOLD", "CNT_SUM / HOUSEHOLDS", "세대당 관측 규모", "주거 세대 기준 활동성 비교"],
    ["CNT_MOM_PCT", "행정동별 CNT_SUM 전월 대비 변화율", "총량 기준 급증·급감", "월별 변동 탐지"],
    ["CNT_PER_PERSON_MOM_PCT", "행정동별 CNT_PER_PERSON 전월 대비 변화율", "인구 보정 급증·급감", "인구 변화 영향을 줄인 이상 변화 탐지"],
    ["CNT_PER_PERSON_MA3", "CNT_PER_PERSON 3개월 이동평균", "단기 변동 완화 추세", "지속 상승/하락 지역 확인"],
], columns=["파생변수", "생성 기준", "의미", "EDA 활용"])

monthly_admin_feature_table


**핵심 설명 - 월별 행정동 단위 EDA 정리**

이 분석의 최종 기준 테이블은 `monthly_admin_panel`입니다. 팀원들과 결과를 맞출 때는 먼저 `TABLE_ID`와 `ADMIN_ROLE`을 고정하고, 그 안에서 월별 행정동 비교를 해야 합니다. 그래야 `t24`의 행정동 유동인구, `t26`의 도착 행정동 체류/교통 데이터, `t27`의 출발 행정동 데이터가 섞이지 않습니다.

발표 흐름은 `coverage 확인 -> 최근 월 TOP 행정동 -> 월별 추세 -> 히트맵 -> 급증/급감 행정동` 순서가 가장 자연스럽습니다.


# 월별 행정동 시각화 모음

아래 그래프들은 `monthly_admin_panel`을 기준으로 월별 행정동 EDA를 발표/공유하기 쉽게 만든 시각화입니다. 기본값은 `t24`의 `행정동` 기준이며, 다른 t 테이블을 보고 싶으면 앞에서 정의한 `focus_table_id`, `focus_admin_role` 값을 바꾸면 됩니다.

시각화 해석 순서:

1. 최근 월 기준으로 인구 대비 유동 규모가 큰 행정동을 찾습니다.
2. 그 행정동들이 일시적으로 튄 것인지, 지속적으로 높은지 월별 추세를 봅니다.
3. 전체 행정동을 히트맵으로 보며 계절성·동시 상승 구간을 확인합니다.
4. 인구 규모와 유동 규모의 관계를 산점도로 확인합니다.
5. 급증·급감 행정동을 따로 뽑아 원인 확인 후보로 정리합니다.


In [ ]:
viz_df = focus_monthly_admin.copy()
viz_latest_month = viz_df["BASE_MONTH"].max()
viz_latest = viz_df[viz_df["BASE_MONTH"] == viz_latest_month].copy()
viz_latest["LABEL"] = viz_latest["CTY_NM"] + " " + viz_latest["ADMI_NM"]

viz_top20 = viz_latest.sort_values("CNT_PER_1K_POP", ascending=False).head(20)
viz_top20[["YEAR_MONTH", "CTY_NM", "ADMI_NM", "TOTAL_POP", "CNT_SUM", "CNT_PER_1K_POP"]]


**시각화 준비 설명 - `viz_top20`**

최근 월 기준 인구 1,000명당 유동 규모가 큰 행정동 20개입니다. 이후 그래프들은 이 표를 기준으로 상위 행정동의 순위와 추세를 보여줍니다.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))
colors = {"성남시 수정구": "#4C78A8", "성남시 중원구": "#F58518", "성남시 분당구": "#54A24B"}
bar_colors = viz_top20["CTY_NM"].map(colors).fillna("#999999")

ax.barh(viz_top20["LABEL"][::-1], viz_top20["CNT_PER_1K_POP"][::-1], color=bar_colors[::-1])
ax.set_title(f"{focus_table_id} {viz_latest_month:%Y-%m} 행정동별 인구 1,000명당 유동 규모 TOP 20")
ax.set_xlabel("CNT / 인구 1,000명")
ax.set_ylabel("행정동")

for i, value in enumerate(viz_top20["CNT_PER_1K_POP"][::-1]):
    ax.text(value, i, f" {value:,.0f}", va="center", fontsize=9)

plt.tight_layout()
plt.show()


**그래프 설명 - 최근 월 TOP 20 막대그래프**

최근 월에 인구 대비 유동 규모가 큰 행정동을 순위로 보여줍니다. 색은 구를 구분합니다. 같은 구의 행정동이 상위권에 몰리면 구 단위 특성이 강한 것이고, 여러 구에 흩어져 있으면 특정 행정동의 개별 특성이 강한 것으로 볼 수 있습니다.


In [ ]:
top12_codes = viz_top20["ADMI_CD"].head(12).tolist()
top12_trend = viz_df[viz_df["ADMI_CD"].isin(top12_codes)].copy()
top12_trend["LABEL"] = top12_trend["CTY_NM"] + " " + top12_trend["ADMI_NM"]

fig, axes = plt.subplots(4, 3, figsize=(16, 11), sharex=True)
axes = axes.ravel()

for ax, (label, sub) in zip(axes, top12_trend.groupby("LABEL")):
    sub = sub.sort_values("BASE_MONTH")
    ax.plot(sub["BASE_MONTH"], sub["CNT_PER_1K_POP"], marker="o", linewidth=1.5)
    ax.set_title(label, fontsize=10)
    ax.grid(alpha=0.25)
    ax.tick_params(axis="x", rotation=45)

for ax in axes[len(top12_trend["LABEL"].unique()):]:
    ax.axis("off")

fig.suptitle(f"{focus_table_id} 상위 행정동별 월별 인구 1,000명당 유동 규모", y=1.02, fontsize=14)
fig.text(0.5, 0.01, "월", ha="center")
fig.text(0.01, 0.5, "CNT / 인구 1,000명", va="center", rotation="vertical")
plt.tight_layout()
plt.show()


**그래프 설명 - 상위 행정동 소형 추세 그래프**

상위 행정동 12개를 각각 작은 선 그래프로 나눠 봅니다. 모든 행정동을 한 그래프에 겹치면 복잡해지므로, 행정동별로 상승·하락·반복 패턴을 따로 확인하기 위한 그래프입니다.


In [ ]:
heatmap_all = viz_df.copy()
heatmap_all["LABEL"] = heatmap_all["CTY_NM"] + " " + heatmap_all["ADMI_NM"]

latest_order = (
    heatmap_all[heatmap_all["BASE_MONTH"] == viz_latest_month]
    .sort_values("CNT_PER_1K_POP", ascending=False)["LABEL"]
    .tolist()
)

heatmap_all_pivot = heatmap_all.pivot(index="LABEL", columns="YEAR_MONTH", values="CNT_PER_1K_POP")
heatmap_all_pivot = heatmap_all_pivot.loc[[x for x in latest_order if x in heatmap_all_pivot.index]]

fig_height = max(10, len(heatmap_all_pivot) * 0.28)
fig, ax = plt.subplots(figsize=(16, fig_height))
im = ax.imshow(heatmap_all_pivot.values, aspect="auto", cmap="YlGnBu")
ax.set_xticks(range(len(heatmap_all_pivot.columns)))
ax.set_xticklabels(heatmap_all_pivot.columns, rotation=45, ha="right")
ax.set_yticks(range(len(heatmap_all_pivot.index)))
ax.set_yticklabels(heatmap_all_pivot.index)
ax.set_title(f"{focus_table_id} 전체 행정동 월별 인구 1,000명당 유동 규모")
fig.colorbar(im, ax=ax, label="CNT / 인구 1,000명")
plt.tight_layout()
plt.show()


**그래프 설명 - 전체 행정동 월별 히트맵**

전체 행정동을 최근 월 인구 보정 유동 규모 순서로 정렬한 히트맵입니다. 가로로 진한 구간이 이어지면 해당 행정동의 지속적 강세, 세로로 여러 행정동이 동시에 진해지면 특정 월의 공통 상승 요인을 의심할 수 있습니다.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for cty_name, sub in viz_latest.groupby("CTY_NM"):
    ax.scatter(
        sub["TOTAL_POP"],
        sub["CNT_PER_1K_POP"],
        s=np.sqrt(sub["CNT_SUM"]) / 40,
        alpha=0.7,
        label=cty_name,
        color=colors.get(cty_name, None),
    )

label_df = viz_latest.sort_values("CNT_PER_1K_POP", ascending=False).head(8)
for _, row in label_df.iterrows():
    ax.annotate(row["ADMI_NM"], (row["TOTAL_POP"], row["CNT_PER_1K_POP"]), fontsize=9, xytext=(4, 4), textcoords="offset points")

ax.set_title(f"{focus_table_id} {viz_latest_month:%Y-%m} 인구 규모와 인구 보정 유동 규모")
ax.set_xlabel("등록 인구")
ax.set_ylabel("CNT / 인구 1,000명")
ax.legend(title="구")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


**그래프 설명 - 인구 규모 vs 인구 보정 유동 규모 산점도**

x축은 등록 인구, y축은 인구 1,000명당 유동 규모입니다. 오른쪽 위는 인구도 많고 유동도 큰 지역, 왼쪽 위는 거주 인구는 작지만 외부 유입이나 활동성이 큰 지역 후보입니다. 점 크기는 `CNT_SUM` 규모를 반영합니다.


In [ ]:
rank_first_month = viz_df["BASE_MONTH"].min()
rank_latest_month = viz_df["BASE_MONTH"].max()

rank_first = (
    viz_df[viz_df["BASE_MONTH"] == rank_first_month]
    [["ADMI_CD", "CTY_NM", "ADMI_NM", "CNT_PER_1K_POP"]]
    .rename(columns={"CNT_PER_1K_POP": "FIRST_CNT_PER_1K_POP"})
)
rank_latest = (
    viz_df[viz_df["BASE_MONTH"] == rank_latest_month]
    [["ADMI_CD", "CNT_PER_1K_POP"]]
    .rename(columns={"CNT_PER_1K_POP": "LATEST_CNT_PER_1K_POP"})
)

rank_change = rank_first.merge(rank_latest, on="ADMI_CD", how="inner")
rank_change["CHANGE"] = rank_change["LATEST_CNT_PER_1K_POP"] - rank_change["FIRST_CNT_PER_1K_POP"]
rank_change["LABEL"] = rank_change["CTY_NM"] + " " + rank_change["ADMI_NM"]

rank_change_plot = pd.concat([
    rank_change.nlargest(10, "CHANGE").assign(TYPE="증가 TOP 10"),
    rank_change.nsmallest(10, "CHANGE").assign(TYPE="감소 TOP 10"),
]).sort_values("CHANGE")

fig, ax = plt.subplots(figsize=(10, 8))
bar_colors = np.where(rank_change_plot["CHANGE"] >= 0, "#4C78A8", "#E45756")
ax.barh(rank_change_plot["LABEL"], rank_change_plot["CHANGE"], color=bar_colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title(f"{focus_table_id} 인구 1,000명당 유동 규모 변화 ({rank_first_month:%Y-%m} -> {rank_latest_month:%Y-%m})")
ax.set_xlabel("CNT / 인구 1,000명 변화량")
ax.set_ylabel("행정동")
plt.tight_layout()
plt.show()


**그래프 설명 - 기간 전체 증가/감소 행정동**

첫 월과 최근 월의 인구 보정 유동 규모 차이를 보여줍니다. 파란색은 증가, 빨간색은 감소입니다. 단기 급변이 아니라 분석 기간 전체에서 구조적으로 커졌거나 작아진 행정동을 찾는 데 사용합니다.


In [ ]:
outlier_plot = monthly_admin_outliers.head(20).copy()
outlier_plot["LABEL"] = outlier_plot["YEAR_MONTH"] + " | " + outlier_plot["CTY_NM"] + " " + outlier_plot["ADMI_NM"]

fig, ax = plt.subplots(figsize=(11, 8))
bar_colors = np.where(outlier_plot["CNT_PER_PERSON_MOM_PCT"] >= 0, "#4C78A8", "#E45756")
ax.barh(outlier_plot["LABEL"][::-1], (outlier_plot["CNT_PER_PERSON_MOM_PCT"] * 100)[::-1], color=bar_colors[::-1])
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title(f"{focus_table_id} 전월 대비 인구 보정 유동 규모 급변 TOP 20")
ax.set_xlabel("전월 대비 변화율(%)")
ax.set_ylabel("월 | 행정동")
plt.tight_layout()
plt.show()


**그래프 설명 - 전월 대비 급증/급감 TOP 20**

전월 대비 변화율이 큰 행정동·월을 보여줍니다. 양수는 급증, 음수는 급감입니다. 이 그래프는 원인 분석 후보를 좁히는 용도이며, 실제 해석은 해당 월의 행사, 명절, 개학/방학, 데이터 결측 가능성을 함께 확인해야 합니다.


## t24 목적·시간대 세부 시각화

아래 두 그래프는 기본 예시인 `t24`를 더 자세히 보기 위한 세부 시각화입니다. 월별 행정동 기준은 유지하되, 목적 코드와 시간대까지 나눠서 어떤 성격의 유동이 상위 행정동을 만들었는지 확인합니다. 이 셀들은 `t24` 원본 parquet를 다시 집계하므로 실행 시간이 걸릴 수 있습니다.


In [ ]:
t24_latest_month = viz_latest_month
t24_latest_month_str = t24_latest_month.strftime("%Y-%m")
t24_top10_codes = viz_top20["ADMI_CD"].head(10).tolist()
t24_top10_codes_sql = ", ".join(map(str, t24_top10_codes))
latest_population_for_viz = population_clean[population_clean["BASE_MONTH"] == t24_latest_month][["ADMI_CD", "CTY_NM", "ADMI_NM"]]

t24_latest_purpose = q(f'''
    SELECT
        ADMI_CD,
        PURPOSE,
        SUM(CNT) AS CNT_SUM
    FROM {scan_sql("t24")}
    WHERE date_trunc('month', CAST(ETL_YMD AS DATE)) = DATE '{t24_latest_month:%Y-%m-01}'
      AND ADMI_CD IN ({t24_top10_codes_sql})
    GROUP BY 1, 2
''')

t24_latest_purpose = t24_latest_purpose.merge(
    latest_population_for_viz,
    on="ADMI_CD",
    how="left",
)
t24_latest_purpose["LABEL"] = t24_latest_purpose["CTY_NM"] + " " + t24_latest_purpose["ADMI_NM"]
t24_latest_purpose["PURPOSE"] = t24_latest_purpose["PURPOSE"].astype(str)
t24_latest_purpose["PURPOSE_SHARE"] = t24_latest_purpose["CNT_SUM"] / t24_latest_purpose.groupby("ADMI_CD")["CNT_SUM"].transform("sum")

purpose_stack = t24_latest_purpose.pivot(index="LABEL", columns="PURPOSE", values="PURPOSE_SHARE").fillna(0)
purpose_stack = purpose_stack.loc[viz_top20.head(10)["LABEL"]]

ax = purpose_stack.plot(kind="barh", stacked=True, figsize=(11, 7), title=f"t24 {t24_latest_month_str} 상위 행정동 목적 코드 구성비")
ax.set_xlabel("목적 코드 구성비")
ax.set_ylabel("행정동")
ax.legend(title="PURPOSE", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


**그래프 설명 - 상위 행정동 목적 코드 구성비**

최근 월 인구 보정 유동 규모 상위 10개 행정동의 목적 코드 비중입니다. 상위 행정동이라도 목적 구성이 다르면 지역 성격이 다를 수 있습니다. 예를 들어 특정 목적 코드가 한 행정동에서 압도적으로 높다면 해당 지역의 방문 목적이 뚜렷하다고 볼 수 있습니다.


In [ ]:
t24_latest_hour = q(f'''
    SELECT
        ADMI_CD,
        CAST(TIME_CD AS INTEGER) AS HOUR,
        SUM(CNT) AS CNT_SUM
    FROM {scan_sql("t24")}
    WHERE date_trunc('month', CAST(ETL_YMD AS DATE)) = DATE '{t24_latest_month:%Y-%m-01}'
      AND ADMI_CD IN ({t24_top10_codes_sql})
    GROUP BY 1, 2
''')

t24_latest_hour = t24_latest_hour.merge(
    latest_population_for_viz,
    on="ADMI_CD",
    how="left",
)
t24_latest_hour["LABEL"] = t24_latest_hour["CTY_NM"] + " " + t24_latest_hour["ADMI_NM"]
t24_latest_hour["HOUR_SHARE"] = t24_latest_hour["CNT_SUM"] / t24_latest_hour.groupby("ADMI_CD")["CNT_SUM"].transform("sum")

hour_pivot = t24_latest_hour.pivot(index="LABEL", columns="HOUR", values="HOUR_SHARE").fillna(0)
hour_pivot = hour_pivot.loc[viz_top20.head(10)["LABEL"]]

fig, ax = plt.subplots(figsize=(14, 6))
im = ax.imshow(hour_pivot.values, aspect="auto", cmap="PuBuGn")
ax.set_xticks(range(len(hour_pivot.columns)))
ax.set_xticklabels(hour_pivot.columns)
ax.set_yticks(range(len(hour_pivot.index)))
ax.set_yticklabels(hour_pivot.index)
ax.set_title(f"t24 {t24_latest_month_str} 상위 행정동 시간대별 유동 구성비")
ax.set_xlabel("시간대")
ax.set_ylabel("행정동")
fig.colorbar(im, ax=ax, label="시간대 구성비")
plt.tight_layout()
plt.show()


**그래프 설명 - 상위 행정동 시간대별 유동 구성비**

최근 월 상위 행정동 10개의 시간대별 유동 구성비 히트맵입니다. 특정 행정동이 오전·점심·저녁 중 어느 시간대에 강한지 확인할 수 있습니다. 목적 코드 구성비와 함께 보면 업무지, 상업지, 주거지 성격을 더 잘 구분할 수 있습니다.


## 9. 젠트리피케이션 예측용 변수별 해석 메모

아래 내용은 각 변수를 예측 분석에 쓸지 판단하기 위한 정리이다. 현재 데이터에는 임대료, 지가, 공실률, 폐업률처럼 젠트리피케이션을 직접 측정하는 변수가 없으므로, 아래 변수들은 **젠트리피케이션 자체가 아니라 외부 수요 증가, 상권 활성화, 거주 구조 변화의 proxy**로 해석한다.

### `external_inflow` (성남시 외부에서 해당 행정동으로 들어온 이동량)

- 왜 이렇게 나올 수 있는가: 업무지구, 상업시설, 교통 접근성이 좋은 행정동은 성남시 외부에서 들어오는 이동량이 커질 수 있다. 분당구나 판교·서현·야탑처럼 외부 방문 목적이 뚜렷한 지역에서 높게 나타날 가능성이 있다.
- 젠트리피케이션과의 연결: 외부 방문객 증가는 외부 소비 수요가 지역 안으로 들어온다는 뜻이다. 이는 상권 활성화, 임대료 상승 압력, 업종 변화와 연결될 수 있어 젠트리피케이션 압력을 설명하는 핵심 변수로 볼 수 있다.
- 사용 판단: **사용 권장**. 다만 행정동 규모 차이가 있으므로 `external_inflow_per_1k_pop`처럼 인구 보정 지표도 같이 만드는 것이 좋다.

### `external_inflow_change_rate` (외부유입의 전월 대비 변화율)

- 왜 이렇게 나올 수 있는가: 신규 시설, 행사, 계절성, 교통 변화, 전월 값이 작은 지역의 기저효과 때문에 크게 튈 수 있다.
- 젠트리피케이션과의 연결: 외부유입의 빠른 증가는 외부 수요가 특정 지역으로 몰리고 있다는 신호다. 젠트리피케이션은 한 시점의 규모보다 지속적인 변화 방향이 중요하므로 변화율은 의미가 있다.
- 사용 판단: **조건부 사용**. 월별 변화율은 노이즈가 크므로 3개월 이동평균이나 전년동월 대비 변화율로 바꾸는 것이 안전하다.

### `floating_pop` (행정동별 전체 유동인구 규모)

- 왜 이렇게 나올 수 있는가: 인구가 많거나 직장·상업·교통 기능이 집중된 지역은 자연스럽게 유동인구가 크게 나온다.
- 젠트리피케이션과의 연결: 유동인구 증가는 지역 활동량과 소비 가능성이 커졌다는 신호다. 상권이 활성화되는 지역은 임대료 상승 압력과 기존 상권 변화 가능성이 커질 수 있다.
- 사용 판단: **보조 변수로 사용**. 원값은 지역 규모 영향을 많이 받으므로 단독 사용보다 인구 대비 지표로 변환하는 것이 좋다.

### `floating_change_rate` (전체 유동인구의 전월 대비 변화율)

- 왜 이렇게 나올 수 있는가: 날씨, 계절, 명절, 방학, 행사, 월별 주말 수 차이 같은 단기 요인에 영향을 받을 수 있다.
- 젠트리피케이션과의 연결: 유동인구가 지속적으로 증가하면 지역이 거주 중심보다 방문·소비 중심으로 바뀌는 흐름을 의심할 수 있다.
- 사용 판단: **조건부 사용**. 한 달 변화율보다 3개월 이동평균 증가율로 쓰는 편이 좋다.

### `floating_pop_per_1k_pop` (등록 인구 1,000명당 유동인구)

- 왜 이렇게 나올 수 있는가: 거주 인구는 적지만 방문자가 많은 상업·업무 지역에서 높게 나타난다. 순수 주거지는 총인구가 많아도 이 값이 낮을 수 있다.
- 젠트리피케이션과의 연결: 거주 인구 대비 유동 규모가 커지는 것은 지역이 주민 생활권보다 외부 방문·소비 중심 공간으로 바뀌고 있다는 신호다.
- 사용 판단: **강력 추천**. `floating_pop` 원값보다 예측 변수로 더 적합하다.

### `working_pop` (경제활동 연령층 유동인구 proxy)

- 왜 이렇게 나올 수 있는가: 업무지구, 상업지, 교통 중심지는 경제활동 연령층의 이동이 많다. 다만 전체 유동인구가 큰 지역에서는 자연스럽게 같이 커질 수 있다.
- 젠트리피케이션과의 연결: 경제활동 연령층은 소비·근로·여가 활동과 연결되므로 신규 소비층 증가를 설명하는 proxy가 될 수 있다.
- 사용 판단: **원값은 주의**. `floating_pop`과 중복성이 높으면 모델에서 제외하고 `working_pop_share`를 쓰는 편이 낫다.

### `working_pop_change_rate` (경제활동 연령층 유동인구의 전월 대비 변화율)

- 왜 이렇게 나올 수 있는가: 전체 유동인구 변화와 같은 방향으로 움직일 가능성이 크다.
- 젠트리피케이션과의 연결: 소비 가능 연령층의 유입 증가를 볼 수 있지만, 전체 유동인구 증가와 구분되지 않으면 해석력이 약하다.
- 사용 판단: **보조 변수**. `floating_change_rate`와 상관이 높으면 버리거나 `working_pop_share_change`로 바꾼다.

### `working_pop_share` (전체 유동인구 중 경제활동 연령층 비중)

- 왜 이렇게 나올 수 있는가: 상업·업무 중심지는 전체 유동인구 중 경제활동 연령층 비중이 높게 나타날 수 있다.
- 젠트리피케이션과의 연결: 단순히 사람이 많은지가 아니라 어떤 연령층이 늘어나는지를 보여준다. 소비력 있는 활동 인구 비중 증가는 상권 고급화나 업종 변화와 연결해 해석할 수 있다.
- 사용 판단: **사용 권장**. `working_pop` 원값보다 모델에 넣기 좋다.

### `TOTAL_POP` (행정동별 등록 총인구)

- 왜 이렇게 나올 수 있는가: 신축 입주, 재개발, 전출입, 고령화 등에 따라 달라진다.
- 젠트리피케이션과의 연결: 유동인구와 외부유입은 증가하는데 등록 인구가 정체되거나 감소하면, 지역이 거주 공간보다 소비·방문 공간으로 바뀌고 있을 가능성을 설명할 수 있다.
- 사용 판단: **보정 변수로 사용**. 단독 핵심 변수보다는 `per_1k_pop` 계산에 필요하다.

### `population_growth_rate` (등록 총인구 증가율)

- 왜 이렇게 나올 수 있는가: 주택 공급, 이주, 재개발, 지역 선호 변화가 반영될 수 있다.
- 젠트리피케이션과의 연결: 외부유입은 증가하지만 인구 증가율이 낮거나 음수이면 기존 거주 안정성이 약해지는 지역으로 해석할 여지가 있다.
- 사용 판단: **보조 변수로 사용**. 월별보다 6개월 또는 1년 변화율이 더 안정적이다.

### `HOUSEHOLDS` (행정동별 세대수)

- 왜 이렇게 나올 수 있는가: 1인 가구 증가, 소형 주택·오피스텔 증가, 가족 단위 가구 분화로 세대수가 증가할 수 있다.
- 젠트리피케이션과의 연결: 총인구가 크게 늘지 않는데 세대수만 늘면 가족 단위 거주보다 소형 가구 중심으로 주거 구조가 바뀌는 신호일 수 있다.
- 사용 판단: **사용 가능**. 반드시 `TOTAL_POP`, `HOUSEHOLD_SIZE`와 같이 봐야 한다.

### `household_growth_rate` (세대수 증가율)

- 왜 이렇게 나올 수 있는가: 소형 가구 증가나 신규 소형 주택 공급의 영향이 반영될 수 있다.
- 젠트리피케이션과의 연결: 세대수 증가와 세대당 인구 감소가 동시에 나타나면 신규 소형 가구 유입 또는 기존 가족형 주거 구조 약화를 의심할 수 있다.
- 사용 판단: **조건부 사용**. 총인구 증가율과 함께 비교해야 의미가 생긴다.

### `HOUSEHOLD_SIZE` (세대당 평균 인구)

- 왜 이렇게 나올 수 있는가: 1인 가구, 청년층, 고령층 단독가구 증가 또는 가족 단위 가구 감소에 따라 낮아질 수 있다.
- 젠트리피케이션과의 연결: 세대당 인구 감소는 지역의 거주 구조 변화 가능성을 보여준다. 외부유입 증가와 함께 나타나면 주거지 성격 변화 해석에 도움이 된다.
- 사용 판단: **사용 권장**. 단, 고령화 영향과 구분이 어려우므로 유동인구 변수와 같이 해석한다.

### `household_size_change_rate` (세대당 인구 변화율)

- 왜 이렇게 나올 수 있는가: 단기간에는 변화가 작지만 특정 지역에서 지속적으로 감소하면 주거 구조 변화가 진행 중일 수 있다.
- 젠트리피케이션과의 연결: 외부유입과 유동인구가 증가하는 동시에 세대당 인구가 감소하면, 지역이 가족형 주거지에서 소형 가구·방문 중심 구조로 바뀌는 흐름을 설명할 수 있다.
- 사용 판단: **보조 핵심 변수**. 변화폭이 작으면 장기 변화율로 쓰는 것이 낫다.

### `DURATION` (지역 내 체류시간)

- 왜 이렇게 나올 수 있는가: 업무, 쇼핑, 여가, 병원 방문 등 실제 목적 활동이 있는 지역에서 길게 나타날 수 있다. 단순 통과 지역은 짧을 가능성이 높다.
- 젠트리피케이션과의 연결: 체류시간 증가는 사람들이 그 지역에서 소비·업무·여가 활동을 하고 있다는 신호다. 외부유입 증가와 함께 나타나면 상권 활성화 가능성이 더 강해진다.
- 사용 판단: **사용 권장**. 평균보다 `long_stay_share`로 바꾸면 해석력이 더 좋아진다.

### `long_stay_share` (일정 시간 이상 장기 체류 비중)

- 왜 이렇게 나올 수 있는가: 상업시설, 업무시설, 여가 공간이 많은 지역은 긴 체류 비중이 높아질 수 있다.
- 젠트리피케이션과의 연결: 단순 방문이 아니라 지역 안에서 머무는 시간이 늘어나는 것이므로 상권 활성화 proxy로 적합하다.
- 사용 판단: **강력 추천**. 120분 이상 또는 180분 이상처럼 기준을 고정해서 만든다.

### `PURPOSE` (이동 목적 코드)

- 왜 이렇게 나올 수 있는가: 이동 목적 코드이지만 현재 코드 정의가 명확하지 않다.
- 젠트리피케이션과의 연결: 목적 코드 정의가 확보되면 쇼핑·여가·업무 목적 이동을 상권 변화와 연결할 수 있다. 하지만 정의가 없으면 해석이 불안정하다.
- 사용 판단: **현재는 메인 변수 제외 권장**. 코드 정의 확인 후 보조 변수로 다시 검토한다.

### `TRANS_GB` (이동수단 또는 이동 유형 코드)

- 왜 이렇게 나올 수 있는가: 이동수단 또는 이동 유형 코드로 보이지만 현재 정의가 명확하지 않다.
- 젠트리피케이션과의 연결: 교통 접근성과 외부 방문 가능성을 설명할 수 있지만, 코드 정의 없이는 어떤 이동수단 증가인지 말하기 어렵다.
- 사용 판단: **현재는 제외 또는 보조 분석만 권장**. 코드 정의가 확인되면 교통 접근성 proxy로 활용 가능하다.

## 예측 변수 후보 선택 기준

| 우선순위 | 변수 | 변수 설명 | 판단 |
|---|---|---|---|
| 1 | `floating_pop_per_1k_pop` | 등록 인구 1,000명당 유동인구 | 지역 규모 보정이 가능하므로 사용 권장 |
| 2 | `external_inflow_per_1k_pop` | 등록 인구 1,000명당 외부유입 이동량 | 외부 수요를 직접 보여주므로 사용 권장 |
| 3 | `external_inflow_change_rate_3m` | 외부유입 변화율의 3개월 이동평균 | 이동평균 처리 후 사용 |
| 4 | `working_pop_share` | 전체 유동인구 중 경제활동 연령층 비중 | 소비 가능 연령층 비율로 사용 권장 |
| 5 | `HOUSEHOLD_SIZE`, `household_size_change_rate` | 세대당 평균 인구와 그 변화율 | 주거 구조 변화 보조 지표로 사용 |
| 6 | `long_stay_share` | 일정 시간 이상 장기 체류한 비중 | 체류 기반 상권 활성화 지표로 사용 |
| 7 | `floating_pop`, `working_pop` 원값 | 전체 유동인구와 경제활동 연령층 유동인구 원규모 | 중복 가능성이 높아 보조로만 사용 |
| 8 | `PURPOSE`, `TRANS_GB` | 이동 목적 코드와 이동수단/이동유형 코드 | 코드 정의 확인 전까지 메인 변수 제외 |

최종 모델링 후보는 `external_inflow_per_1k_pop`, `external_inflow_change_rate_3m`, `floating_pop_per_1k_pop`, `floating_change_rate_3m`, `working_pop_share`, `household_growth_rate`, `population_growth_rate`, `household_size_change_rate`, `long_stay_share`로 정리하는 것이 적절하다.
